### 1. Εισαγωγή και Φόρτωση Δεδομένων
Σε αυτό το στάδιο προετοιμάζεται το υπολογιστικό περιβάλλον για τη διαχείριση χημικών δεδομένων. Η χρήση της Τεχνητής Νοημοσύνης στον σχεδιασμό φαρμάκων (de novo drug design) απαιτεί εξειδικευμένα εργαλεία που γεφυρώνουν τη Χημεία με την Πληροφορική.

Βασικά Εργαλεία:
RDKit : Η κορυφαία βιβλιοθήκη για τον χειρισμό χημικών δομών, τον υπολογισμό φυσικοχημικών ιδιοτήτων και τη μετατροπή μοριακών γραφημάτων σε ψηφιακή μορφή.

ZINC Dataset: Χρησιμοποιούμε το υποσύνολο ZINC-250k, το οποίο περιλαμβάνει περίπου 250.000 εμπορικά διαθέσιμα μόρια. Τα μόρια αυτά έχουν επιλεγεί γιατί "μοιάζουν με φάρμακα" (drug-like molecules), τηρώντας κανόνες όπως το Lipinski's Rule of Five.

Η Σημασία των Δεδομένων:
Κάθε εγγραφή στο dataset περιλαμβάνει τη δομή του μορίου σε μορφή SMILES (Simplified Molecular Input Line Entry System). Η ποιότητα και η καθαρότητα αυτών των δεδομένων είναι καθοριστική για την ικανότητα του μοντέλου να παράγει ρεαλιστικές χημικές δομές.

In [ ]:
# ==========================================
# Εισαγωγή βιβλιοθηκών
# ==========================================

import os
import time
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Descriptors, AllChem, Lipinski, QED, Draw
from rdkit.Chem import MolFromSmiles

import selfies as sf

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.notebook import tqdm

# ==========================================
# Ρυθμίσεις RDKit
# ==========================================

# Απενεργοποιούμε τα προειδοποιητικά μηνύματα της RDKit,
# ώστε η έξοδος του notebook να παραμένει καθαρή.
RDLogger.DisableLog("rdApp.*")

# ==========================================
# Ρύθμιση συσκευής εκτέλεσης
# ==========================================

# Χρησιμοποιούμε GPU αν είναι διαθέσιμη, διαφορετικά CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Συσκευή εκτέλεσης: {device}")
print("Οι βιβλιοθήκες φορτώθηκαν επιτυχώς.")

### 2. Συλλογή και Ενοποίηση Πολυδιάστατων Χημικών Δεδομένων

Για την υλοποίηση της πτυχιακής εργασίας, επιλέχθηκαν δεδομένα που εξισορροπούν τη χημική πολυπλοκότητα με τη βιολογική συνάφεια:

**ZINC-250k:** Περιλαμβάνει 250.000 μόρια που είναι εμπορικά διαθέσιμα και παρουσιάζουν **"φαρμακοειδείς"** ιδιότητες (drug-like properties). Τα μόρια αυτά έχουν φιλτραριστεί ώστε να τηρούν τα κριτήρια καθαρότητας και σταθερότητας που απαιτούνται στην προκλινική έρευνα.

**Guacamol Benchmark Dataset:** Αποτελεί τον κεντρικό πυρήνα της εργασίας, περιλαμβάνοντας άνω του 1.5 εκατομμυρίου έγκυρων χημικών δομών. Aποτελεί μια τυποποιημένη έκδοση δεδομένων που προέρχεται κατά κύριο λόγο από τη βάση ChEMBL και τη ZINC. Επιλέχθηκε επειδή παρέχει έναν αυστηρά φιλτραρισμένο υποσύνολο μορίων που τηρούν τα κριτήρια 'drug-likeness', εξασφαλίζοντας ότι το μοντέλο εκπαιδεύεται σε χημικά έγκυρες και βιολογικά σχετικές δομές."

In [ ]:
# ==========================================
# Φόρτωση και ενοποίηση αρχικών datasets
# ==========================================

# Φόρτωση του ZINC dataset.
# Το αρχείο περιέχει SMILES και προϋπολογισμένες ιδιότητες όπως logP και QED.
df_zinc = pd.read_csv("data/250k_rndm_zinc_drugs_clean_3.csv")

# Κρατάμε μόνο τις στήλες που χρειάζονται για την παρούσα εργασία.
df_zinc = df_zinc[["smiles", "logP", "qed"]]


# Φόρτωση του GuacaMol dataset.
# Το αρχείο είναι σε μορφή .smiles και δεν περιέχει επικεφαλίδες στηλών,
# οπότε ορίζουμε εμείς τη στήλη "smiles".
df_guaca = pd.read_csv(
    "data/guacamol_v1_train.smiles",
    names=["smiles"]
)


# ==========================================
# Υπολογισμός logP για το GuacaMol
# ==========================================

# Το GuacaMol dataset περιέχει μόνο τις χημικές δομές σε μορφή SMILES.
# Για να μπορεί να χρησιμοποιηθεί στο conditional generation,
# υπολογίζουμε το logP κάθε μορίου με την RDKit.
print("Υπολογισμός logP για το GuacaMol dataset...")

tqdm.pandas()

def calculate_logp(smiles):
    """
    Υπολογίζει την τιμή logP για ένα μόριο από τη SMILES αναπαράστασή του.
    Αν το SMILES δεν μπορεί να διαβαστεί από την RDKit, επιστρέφει None.
    """
    mol = Chem.MolFromSmiles(str(smiles))
    
    if mol is None:
        return None
    
    return Descriptors.MolLogP(mol)


df_guaca["logP"] = df_guaca["smiles"].progress_apply(calculate_logp)

# Το GuacaMol δεν παρέχει έτοιμη τιμή QED σε αυτό το αρχείο.
# Η στήλη κρατιέται για να υπάρχει κοινή δομή με το ZINC dataset.
df_guaca["qed"] = None


# ==========================================
# Ενοποίηση datasets
# ==========================================

# Ενώνουμε τα δύο datasets σε ένα κοινό dataframe.
df_all = pd.concat(
    [df_zinc, df_guaca],
    axis=0
).reset_index(drop=True)

print(f"Συνολικά μόρια μετά την ενοποίηση: {len(df_all)}")

df_all.head()

### 3. Καθαρισμός και Κανονικοποίηση (Preprocessing)
Η φάση της προεπεξεργασίας είναι καθοριστική, καθώς η ποιότητα των παραγόμενων μορίων εξαρτάται άμεσα από τη δομή των δεδομένων εισόδου. Χρησιμοποιούμε τη βιβλιοθήκη RDKit για να μετατρέψουμε τις ακατέργαστες συμβολοσειρές σε μια ομοιογενή και μαθηματικά συνεπή μορφή.

Βασικά Στάδια Επεξεργασίας:
Επαλήθευση Χημικής Εγκυρότητας (Sanitization): Κάθε SMILES ελέγχεται αν τηρεί τους κανόνες του σθένους (valence). Μόρια που παραβιάζουν τους νόμους της χημείας (π.χ. πεντασθενής άνθρακας) απορρίπτονται άμεσα, διασφαλίζοντας ότι το μοντέλο εκπαιδεύεται μόνο σε ρεαλιστικές χημικές οντότητες.

Κανονικοποίηση (Canonicalization): Μια χημική δομή μπορεί να περιγραφεί με πολλά διαφορετικά SMILES ανάλογα με το ποιο άτομο θεωρείται ως αρχή.

Παράδειγμα: Το CN και το NC περιγράφουν την ίδια ουσία.

Μετατρέπουμε όλα τα μόρια σε Unique Canonical SMILES, ώστε κάθε δομή να έχει μία και μοναδική ψηφιακή ταυτότητα, αποτρέποντας το μοντέλο από το να μπερδεύεται με διπλότυπες εγγραφές.



In [ ]:
# ==========================================
# Καθαρισμός SMILES και μετατροπή σε SELFIES
# ==========================================

def canonicalize_smiles(smiles):
    """
    Μετατρέπει ένα SMILES σε canonical SMILES με χρήση της RDKit.
    
    Η canonical μορφή βοηθά ώστε η ίδια χημική δομή να έχει
    μία ενιαία αναπαράσταση, ακόμη και αν αρχικά εμφανίζεται
    με διαφορετική σύνταξη SMILES.
    """
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        
        if mol is None:
            return None
        
        return Chem.MolToSmiles(
            mol,
            isomericSmiles=True,
            canonical=True
        )
    
    except Exception:
        return None


def smiles_to_selfies(smiles):
    """
    Μετατρέπει ένα canonical SMILES σε SELFIES.
    
    Αν η μετατροπή αποτύχει, επιστρέφει None.
    """
    try:
        return sf.encoder(smiles)
    
    except Exception:
        return None


# Ενεργοποίηση μπάρας προόδου για τις pandas εφαρμογές
tqdm.pandas()


# ==========================================
# 1. Κανονικοποίηση SMILES
# ==========================================

print("1/2: Κανονικοποίηση SMILES με την RDKit...")

df_all["Clean_SMILES"] = df_all["smiles"].progress_apply(
    canonicalize_smiles
)

# Αφαιρούμε όσα μόρια δεν μπόρεσε να διαβάσει η RDKit
df_all = df_all.dropna(subset=["Clean_SMILES"]).copy()

print(f"Έγκυρα μόρια μετά την κανονικοποίηση: {len(df_all)}")


# ==========================================
# 2. Μετατροπή σε SELFIES
# ==========================================

print("2/2: Μετατροπή των canonical SMILES σε SELFIES...")

df_all["SELFIES"] = df_all["Clean_SMILES"].progress_apply(
    smiles_to_selfies
)

# Αφαιρούμε τυχόν αποτυχίες στη μετατροπή σε SELFIES
df_all = df_all.dropna(subset=["SELFIES"]).copy()

print(f"Έγκυρα μόρια μετά τη μετατροπή σε SELFIES: {len(df_all)}")


# ==========================================
# 3. Αφαίρεση διπλότυπων μορίων
# ==========================================

before_dedup = len(df_all)

# Αφαιρούμε διπλότυπα με βάση το canonical SMILES.
# Έτσι κάθε χημική δομή εμφανίζεται μόνο μία φορά στο τελικό dataset.
df_all = df_all.drop_duplicates(
    subset=["Clean_SMILES"],
    keep="first"
).reset_index(drop=True)

after_dedup = len(df_all)

print("-" * 40)
print("Αφαίρεση διπλοτύπων ολοκληρώθηκε.")
print(f"Μόρια πριν την αφαίρεση διπλοτύπων: {before_dedup}")
print(f"Μόρια μετά την αφαίρεση διπλοτύπων: {after_dedup}")
print(f"Αφαιρέθηκαν διπλότυπα: {before_dedup - after_dedup}")


# ==========================================
# 4. Αποθήκευση καθαρισμένου dataset
# ==========================================

df_all.to_csv("processed_data.csv", index=False)

print("-" * 40)
print("Το preprocessing ολοκληρώθηκε.")
print(f"Τελικό πλήθος καθαρών και μοναδικών μορίων: {len(df_all)}")
print("Το αρχείο processed_data.csv αποθηκεύτηκε.")


# Εμφάνιση πρώτων αποτελεσμάτων
df_all[["Clean_SMILES", "SELFIES"]].head()

### 4. Διαχωρισμός Δεδομένων 
Για την ορθή εκπαίδευση και αξιολόγηση του Generative Transformer, χωρίζουμε το σύνολο των δεδομένων σε δύο ανεξάρτητα υποσύνολα. Αυτό μας επιτρέπει να ελέγξουμε αν το μοντέλο έμαθε πραγματικά τους κανόνες της χημείας ή αν απλώς "αποστήθισε" το dataset.

Training Set (90%): Το κύριο σώμα των δεδομένων, αποτελούμενο από περίπου 1.350.000+ μόρια. Αυτό το υποσύνολο τροφοδοτείται στον αλγόριθμο Backpropagation για τη συνεχή ενημέρωση των βαρών του νευρωνικού δικτύου.

Validation/Test Set (10%): Ένα ανεξάρτητο υποσύνολο περίπου 150.000+ μορίων, το οποίο το μοντέλο δεν "βλέπει" κατά τη διαδικασία της μάθησης. Χρησιμοποιείται για την παρακολούθηση του Validation Loss και την αξιολόγηση της ικανότητας του μοντέλου να παράγει έγκυρες χημικές δομές που δεν περιλαμβάνονταν στο training set.

Τεχνικές Λεπτομέρειες:
Random Seed (42): Χρησιμοποιήθηκε σταθερή τιμή τυχαιότητας (seed) για τη διασφάλιση της αναπαραγωγιμότητας των αποτελεσμάτων.

Shuffle: Πραγματοποιήθηκε πλήρης ανακατανομή των δεδομένων πριν από τον διαχωρισμό, ώστε να εξαλειφθεί οποιαδήποτε μεροληψία (bias) που θα μπορούσε να προκύψει από την αρχική σειρά καταχώρησης των μορίων στις βάσεις δεδομένων (π.χ. ταξινόμηση ανά μοριακό βάρος ή πηγή προέλευσης).

Scalability: Η επιλογή της αναλογίας 90/10 θεωρείται βέλτιστη για datasets τέτοιας κλίμακας (Big Data), καθώς το 10% προσφέρει ένα στατιστικά ισχυρό δείγμα 150.000 μορίων για τον έλεγχο της αξιοπιστίας του μοντέλου.

In [ ]:
# ==========================================
# Διαχωρισμός σε Train / Validation / Test
# ==========================================

print(f"Συνολικά επεξεργασμένα μόρια: {len(df_all)}")

# 1. Πρώτος διαχωρισμός:
# Κρατάμε το 80% των δεδομένων για εκπαίδευση
# και το υπόλοιπο 20% προσωρινά για validation και test.
train_df, temp_df = train_test_split(
    df_all,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# 2. Δεύτερος διαχωρισμός:
# Χωρίζουμε το προσωρινό 20% σε δύο ίσα μέρη:
# 10% validation και 10% test.
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

# 3. Επιβεβαίωση μεγεθών
print("Ο διαχωρισμός έγινε σωστά σε train, validation και test set.")
print(f"Train Set:      {len(train_df)}")
print(f"Validation Set: {len(val_df)}")
print(f"Test Set:       {len(test_df)}")

# 4. Γρήγορος έλεγχος των δεδομένων εκπαίδευσης
train_df[["SELFIES", "logP"]].head()

### 5. Tokenization (Τεμαχισμός) και Δημιουργία Λεξικού
Οι αρχιτεκτονικές Transformer είναι σχεδιασμένες να επεξεργάζονται αριθμητικές ακολουθίες και όχι απευθείας κείμενο. Για τον λόγο αυτό, υλοποιούμε τη διαδικασία του Tokenization, μετατρέποντας τις συμβολοσειρές SELFIES σε μια μορφή που μπορεί να κατανοήσει το νευρωνικό δίκτυο.

**5.1** Ορισμός Λεξικού (Vocabulary)
Το Λεξικό μας αποτελείται από το σύνολο των μοναδικών χαρακτήρων/συμβόλων που εμφανίζονται στο dataset (π.χ. [C], [N], [O], [Branch1], κλπ.). Κάθε σύμβολο αντιστοιχίζεται σε έναν μοναδικό ακέραιο αριθμό (ID).

**5.2** Ειδικά Tokens (Special Tokens)
Για τον έλεγχο της παραγωγής από τον Generative Decoder, εισάγουμε τρία κρίσιμα tokens:

 <START>: Λειτουργεί ως το "έναυσμα" για το μοντέλο ώστε να ξεκινήσει τη δημιουργία ενός νέου μορίου.

 <END>: Σηματοδοτεί την ολοκλήρωση της χημικής δομής.

 <PAD>: Χρησιμοποιείται για την εξίσωση του μήκους όλων των ακολουθιών σε έναν πίνακα (Batch Tensor), επιτρέποντας την παράλληλη επεξεργασία στην GPU.

**5.3** Numerical Encoding (Αριθμητική Κωδικοποίηση)
Σε αυτό το στάδιο, κάθε μόριο μετατρέπεται από μια σειρά χαρακτήρων σε ένα διάνυσμα ακεραίων.

Παράδειγμα: Αν το [C] έχει ID 5 και το [O] έχει ID 12, η αλυσίδα [C][O] μετατρέπεται σε [5, 12].

Status: Το Λεξικό μας περιλαμβάνει 59 μοναδικά tokens. Το μέγιστο μήκος ακολουθίας (MAX_LEN) ορίστηκε στα 74 tokens, διασφαλίζοντας ότι καλύπτεται το 100% των μορίων του ZINC-250k χωρίς περικοπές.

In [ ]:
# ==========================================
# Δημιουργία λεξικού από τα SELFIES του train set
# ==========================================

# Παίρνουμε μόνο τα SELFIES του train set.
# Το vocabulary πρέπει να δημιουργείται μόνο από τα δεδομένα εκπαίδευσης,
# ώστε να αποφεύγεται διαρροή πληροφορίας από validation ή test set.
all_selfies = train_df["SELFIES"].astype(str).tolist()

print("Εξαγωγή και ανάλυση συχνότητας των SELFIES tokens...")


# ==========================================
# Ανάλυση συχνότητας tokens
# ==========================================

all_tokens_list = []

for selfies_string in all_selfies:
    tokens = list(sf.split_selfies(selfies_string))
    all_tokens_list.extend(tokens)

token_counts = Counter(all_tokens_list)


# ==========================================
# Φιλτράρισμα σπάνιων tokens
# ==========================================

# Κρατάμε μόνο tokens που εμφανίζονται τουλάχιστον 5 φορές.
# Τα σπάνια tokens θα αντικατασταθούν αργότερα με <UNK>.
min_freq = 5

alphabet = {
    token
    for token, count in token_counts.items()
    if count >= min_freq
}

print(
    f"Αρχικά μοναδικά tokens: {len(token_counts)} | "
    f"Μετά το φιλτράρισμα freq >= {min_freq}: {len(alphabet)}"
)


# ==========================================
# Προσθήκη ειδικών tokens
# ==========================================

# <START>: δηλώνει την αρχή μιας ακολουθίας.
# <END>: δηλώνει το τέλος μιας ακολουθίας.
# <PAD>: χρησιμοποιείται για padding, ώστε όλες οι ακολουθίες να έχουν ίδιο μήκος.
# <UNK>: χρησιμοποιείται για σπάνια ή άγνωστα tokens.
special_tokens = ["<START>", "<END>", "<PAD>", "<UNK>"]

alphabet.update(special_tokens)


# ==========================================
# Δημιουργία vocabulary και mappings
# ==========================================

vocab = sorted(list(alphabet))

token_to_id = {
    token: idx
    for idx, token in enumerate(vocab)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}


# ==========================================
# Ορισμός μέγιστου μήκους ακολουθίας
# ==========================================

# Το MAX_LEN καθορίζει το μέγιστο μήκος κάθε encoded SELFIES ακολουθίας.
# Περιλαμβάνει και τα ειδικά tokens <START> και <END>.
MAX_LEN = 90


# ==========================================
# Έλεγχοι λεξικού
# ==========================================

print("-" * 40)
print("Το λεξικό δημιουργήθηκε επιτυχώς.")
print(f"Μέγεθος λεξικού: {len(vocab)}")
print(f"Μέγιστο μήκος ακολουθίας MAX_LEN: {MAX_LEN}")
print(f"Πρώτα 15 tokens: {vocab[:15]}")
print("-" * 40)


# ==========================================
# Έλεγχος σε ένα παράδειγμα SELFIES
# ==========================================

sample_selfie = all_selfies[0]

sample_tokens = [
    token if token in token_to_id else "<UNK>"
    for token in sf.split_selfies(sample_selfie)
]

print(f"Παράδειγμα SELFIES:\n{sample_selfie}")
print(f"\nTokens:\n{sample_tokens}")

### 6. Μετατροπή Δεδομένων σε Tensors (Numerical Encoding)
Σε αυτό το στάδιο, ολοκληρώνουμε τη μετάβαση από τη χημική αναπαράσταση στην υπολογιστική δομή. Το Numerical Encoding μετατρέπει τις ακολουθίες SELFIES σε αριθμητικούς πίνακες (Tensors), οι οποίοι αποτελούν τη βασική μονάδα επεξεργασίας για τη βιβλιοθήκη PyTorch.

Τα Στάδια του Encoding:
Προσθήκη Οριοθετών (Boundary Tokens): Κάθε μόριο πλαισιώνεται από το token έναρξης <START> και το token τερματισμού <END>. Αυτό επιτρέπει στο μοντέλο να μαθαίνει πότε μια χημική δομή είναι πλήρης.

Mapping σε Ακεραίους: Αντικαθιστούμε κάθε σύμβολο SELFIES με το αντίστοιχο Index από το Λεξικό μας. Η διαδικασία αυτή μετατρέπει τη χημεία σε μια γραμμική αλληλουχία αριθμών.

Εφαρμογή Padding: Επειδή τα μόρια έχουν διαφορετικά μήκη (π.χ. η αιθανόλη είναι μικρή, ενώ ένα φάρμακο μεγάλο), προσθέτουμε το token <PAD> (συνήθως το 0) στο τέλος κάθε ακολουθίας. Με αυτόν τον τρόπο, όλες οι είσοδοι αποκτούν σταθερό μήκος MAX_LEN.

Δημιουργία PyTorch Tensors: Οι λίστες ακεραίων μετατρέπονται σε αντικείμενα torch.Tensor. Αυτό είναι το κρίσιμο βήμα που επιτρέπει τη μεταφορά των δεδομένων στη μνήμη της GPU (CUDA) για την παράλληλη εκτέλεση των μαθηματικών πράξεων του Transformer.

In [ ]:
# ==========================================
# Encoding Function
# ==========================================

def encode_selfies(selfies, token_to_id, max_len):
    """
    Μετατρέπει μία SELFIES ακολουθία σε λίστα από token IDs.
    
    Προσθέτει:
    - <START> στην αρχή
    - <END> στο τέλος
    - <PAD> μέχρι να φτάσει στο max_len
    
    Αν η ακολουθία είναι μεγαλύτερη από max_len,
    κόβεται και διασφαλίζεται ότι τελειώνει με <END>.
    """
    
    tokens = list(sf.split_selfies(selfies))
    tokens = ["<START>"] + tokens + ["<END>"]

    ids = [
        token_to_id.get(t, token_to_id["<UNK>"])
        for t in tokens
    ]

    padding_length = max_len - len(ids)

    if padding_length > 0:
        ids = ids + [token_to_id["<PAD>"]] * padding_length
    else:
        ids = ids[:max_len - 1] + [token_to_id["<END>"]]

    return ids


# ==========================================
# Encoding Train / Validation / Test Sets
# ==========================================

print("Encoding Train Set...")
train_encoded = [
    encode_selfies(s, token_to_id, MAX_LEN)
    for s in tqdm(train_df["SELFIES"])
]
full_X_train = torch.tensor(train_encoded, dtype=torch.long)


print("Encoding Validation Set...")
val_encoded = [
    encode_selfies(s, token_to_id, MAX_LEN)
    for s in tqdm(val_df["SELFIES"])
]
full_X_val = torch.tensor(val_encoded, dtype=torch.long)


print("Encoding Test Set...")
test_encoded = [
    encode_selfies(s, token_to_id, MAX_LEN)
    for s in tqdm(test_df["SELFIES"])
]
full_X_test = torch.tensor(test_encoded, dtype=torch.long)


# ==========================================
# Inputs and Targets for Language Modeling
# ==========================================

# Train
# Input: από το <START> έως το προτελευταίο token
X_train_input = full_X_train[:, :-1]

# Target: από το δεύτερο token έως το <END>
Y_train_target = full_X_train[:, 1:]


# Validation
X_val_input = full_X_val[:, :-1]
Y_val_target = full_X_val[:, 1:]


# Test
X_test_input = full_X_test[:, :-1]
Y_test_target = full_X_test[:, 1:]


# ==========================================
# logP Condition Tensors
# ==========================================

Y_train_logP = torch.tensor(
    train_df["logP"].values,
    dtype=torch.float32
).unsqueeze(1)

Y_val_logP = torch.tensor(
    val_df["logP"].values,
    dtype=torch.float32
).unsqueeze(1)

Y_test_logP = torch.tensor(
    test_df["logP"].values,
    dtype=torch.float32
).unsqueeze(1)


# ==========================================
# DataLoaders
# ==========================================

# Κάθε DataLoader επιστρέφει:
# 1. input sequence
# 2. target sequence
# 3. logP condition

train_dataset = TensorDataset(
    X_train_input,
    Y_train_target,
    Y_train_logP
)

val_dataset = TensorDataset(
    X_val_input,
    Y_val_target,
    Y_val_logP
)

test_dataset = TensorDataset(
    X_test_input,
    Y_test_target,
    Y_test_logP
)


BATCH_SIZE = 1200

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)


# ==========================================
# Final Checks
# ==========================================

print(f"\nBatch Size: {BATCH_SIZE}")
print(f"Μήκος αλληλουχίας εισόδου: {X_train_input.shape[1]}")
print(f"Train Batches:      {len(train_loader)}")
print(f"Validation Batches: {len(val_loader)}")
print(f"Test Batches:       {len(test_loader)}")
print("-" * 30)
print("Οι DataLoaders είναι έτοιμοι για Conditional Training.")

### 8. Ορισμός Αρχιτεκτονικής Transformer (Generative Decoder)
Για την παραγωγή νέων χημικών δομών, υλοποιούμε έναν Transformer Decoder. Σε αντίθεση με τα παραδοσιακά δίκτυα RNN/LSTM, ο Transformer βασίζεται αποκλειστικά σε μηχανισμούς Attention (Προσοχής), επιτρέποντας την παράλληλη επεξεργασία και την κατανόηση μακροπρόθεσμων εξαρτήσεων μέσα στο μόριο.

**Κύρια Αρχιτεκτονικά Στοιχεία:**
**Positional Encoding (Θεσειακή Κωδικοποίηση)** : Καθώς ο Transformer στερείται εγγενούς γνώσης για τη σειρά των tokens, προσθέτουμε ένα ημιτονοειδές διάνυσμα (sinusoidal signal) σε κάθε embedding. Αυτό επιτρέπει στο μοντέλο να γνωρίζει τη σχετική και απόλυτη θέση κάθε ατόμου μέσα στην αλυσίδα SELFIES.

**Multi-Head Self-Attention** : Είναι ο "εγκέφαλος" του μοντέλου. Επιτρέπει στον Transformer να συσχετίζει κάθε άτομο με όλα τα υπόλοιπα ταυτόχρονα. Για παράδειγμα, μπορεί να "προσέχει" ταυτόχρονα την αρχή και το τέλος ενός αρωματικού δακτυλίου, διασφαλίζοντας ότι η χημική δομή θα κλείσει σωστά.

**Causal Masking (Τριγωνική Μάσκα)**: Επειδή το μοντέλο είναι παραγωγικό (Generative), χρησιμοποιούμε μια μάσκα που εμποδίζει την πληροφορία να ρέει από το "μέλλον" προς το παρελθόν. Κατά την εκπαίδευση, το μοντέλο προβλέπει το επόμενο άτομο βασιζόμενο αποκλειστικά στους χαρακτήρες που έχουν ήδη εμφανιστεί.

Τεχνικές Προδιαγραφές Μοντέλου:
Embedding Size: 256 (Διάσταση αναπαράστασης κάθε token).

Attention Heads: 8 (Διαφορετικές "ματιές" πάνω στη δομή).

Decoder Layers: 4 (Βάθος του δικτύου για την εκμάθηση σύνθετων χημικών κανόνων).

In [ ]:
# ==========================================
# Ορισμός αρχιτεκτονικής Conditional Transformer
# ==========================================

class MoleculeTransformer(nn.Module):
    """
    Conditional Transformer μοντέλο για παραγωγή μοριακών ακολουθιών SELFIES.

    Το μοντέλο δέχεται:
    - x: ακολουθία από token IDs
    - prop: αριθμητική ιδιότητα-στόχο, εδώ το logP

    Η αρχιτεκτονική βασίζεται σε causal self-attention, ώστε κάθε θέση
    να μπορεί να χρησιμοποιεί μόνο τα προηγούμενα tokens και όχι μελλοντική πληροφορία.
    """

    def __init__(self, vocab_size, embed_size, num_heads, num_layers, max_len, pad_idx):
        super(MoleculeTransformer, self).__init__()

        self.pad_idx = pad_idx

        # Μετατροπή token IDs σε dense διανύσματα αναπαράστασης
        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_size,
            padding_idx=pad_idx
        )

        # Εκμάθηση positional embeddings για τη θέση κάθε token στην ακολουθία
        self.position_embedding = nn.Embedding(
            max_len,
            embed_size
        )

        # Μετατροπή της αριθμητικής ιδιότητας logP σε embedding ίδιου μεγέθους
        # με τα token embeddings, ώστε να μπορεί να προστεθεί στην ακολουθία.
        self.property_fc = nn.Sequential(
            nn.Linear(1, embed_size),
            nn.SiLU(),
            nn.Linear(embed_size, embed_size)
        )

        # Transformer block με self-attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_size,
            nhead=num_heads,
            batch_first=True,
            norm_first=True,
            dropout=0.1,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        # Τελικό linear layer που προβλέπει πιθανότητες για κάθε token του λεξικού
        self.fc_out = nn.Linear(embed_size, vocab_size)

    def forward(self, x, prop):
        """
        Εκτελεί forward pass του μοντέλου.

        Parameters
        ----------
        x : torch.Tensor
            Tensor διαστάσεων [batch_size, sequence_length] με τα token IDs.

        prop : torch.Tensor
            Tensor διαστάσεων [batch_size, 1] με την τιμή logP.

        Returns
        -------
        torch.Tensor
            Logits διαστάσεων [batch_size, sequence_length, vocab_size].
        """

        seq_len = x.size(1)

        # Δημιουργία causal mask.
        # Η μάσκα εμποδίζει το μοντέλο να βλέπει μελλοντικά tokens.
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool),
            diagonal=1
        )

        # Padding mask ώστε το μοντέλο να αγνοεί τα <PAD> tokens.
        key_padding_mask = x == self.pad_idx

        # Δημιουργία positional IDs: 0, 1, 2, ..., seq_len-1
        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0)

        # Token embeddings + positional embeddings
        x_embed = self.token_embedding(x) + self.position_embedding(positions)

        # Property embedding για το logP
        prop_embed = self.property_fc(prop).unsqueeze(1)

        # Προσθήκη του logP condition σε κάθε θέση της ακολουθίας
        out = x_embed + prop_embed

        # Transformer encoder με causal self-attention
        out = self.transformer(
            out,
            mask=causal_mask,
            src_key_padding_mask=key_padding_mask
        )

        # Πρόβλεψη επόμενου token
        logits = self.fc_out(out)

        return logits


# ==========================================
# Αρχικοποίηση μοντέλου
# ==========================================

PAD_IDX = token_to_id["<PAD>"]

model = MoleculeTransformer(
    vocab_size=len(vocab),
    embed_size=256,
    num_heads=8,
    num_layers=4,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX
).to(device)

print(f"Το μοντέλο μεταφέρθηκε στη συσκευή: {device}")
print(f"Συνολικές παράμετροι: {sum(p.numel() for p in model.parameters()):,}")

### 10. Διαδικασία Εκπαίδευσης (Training Loop)

Μηχανισμός Εκπαίδευσης:
Teacher Forcing: Σε κάθε χρονικό βήμα, το μοντέλο δέχεται ως είσοδο την πραγματική αλληλουχία από το dataset (ground truth) και όχι τη δική του προηγούμενη πρόβλεψη. Αυτό αποτρέπει τη συσσώρευση σφαλμάτων στην αρχή της εκπαίδευσης και επιτρέπει στο μοντέλο να μαθαίνει ταχύτερα τους σωστούς χημικούς κανόνες.

Πρόβλεψη Επόμενου Token (Next Token Prediction): Για μια είσοδο όπως η <START> [C][C], ο Transformer παράγει ταυτόχρονα προβλέψεις για κάθε θέση. Η συνάρτηση σφάλματος αξιολογεί αν το μοντέλο "μαντεύει" σωστά ότι μετά το δεύτερο [C] ακολουθεί, για παράδειγμα, ένας δεσμός ή ένα [O].

Backpropagation & Gradient Clipping: * Ο αλγόριθμος **Backpropagation** υπολογίζει την κλίση του σφάλματος σε σχέση με κάθε βάρος του δικτύου.

Εφαρμόζω Gradient Clipping (περικοπή κλίσεων) στην τιμή 1.0, ώστε να αποφύγουμε απότομες μεταβολές που θα μπορούσαν να "αποσταθεροποιήσουν" τη μνήμη της GPU.

Αξιολόγηση (Validation Phase): Στο τέλος κάθε εποχής, το μοντέλο τίθεται σε κατάσταση eval(). Υπολογίζω το σφάλμα στο Validation Set για να βεβαιωθώ ότι το μοντέλο αποκτά γενικευτική ικανότητα και δεν περιορίζεται στην αποστήθιση **(overfitting)** των δεδομένων εκπαίδευσης.

Στρατηγική Αποθήκευσης (Checkpointing):
Best Model: Αποθηκεύω αυτόματα το μοντέλο με το χαμηλότερο Validation Loss (best_molecule_model.pth).

Periodic Checkpoints: Ανά 5 εποχές αποθηκεύω την πλήρη κατάσταση (μοντέλο + optimizer) για λόγους ασφάλειας και δυνατότητα μελλοντικής συνέχισης της εκπαίδευσης.

In [ ]:
# ==========================================
# Εκπαίδευση Conditional Transformer
# ==========================================

# Μεταφέρουμε το μοντέλο στη συσκευή που έχει ήδη οριστεί στο πρώτο κελί.
model = model.to(device)

# ==========================================
# Ρυθμίσεις εκπαίδευσης
# ==========================================

EPOCHS = 100
checkpoint_interval = 5
patience = 7
learning_rate = 0.0003

vocab_size = len(vocab)
pad_id = token_to_id["<PAD>"]

# Ονόματα αρχείων για να μη μπερδευτούν με παλιά checkpoints.
best_model_path = "best_molecule_transformer_clean_split.pth"
checkpoint_prefix = "checkpoint_training_epoch"

# Χρήση mixed precision μόνο αν υπάρχει GPU με CUDA.
use_amp = device.type == "cuda"

# Συνάρτηση σφάλματος.
# Το ignore_index αγνοεί τα <PAD> tokens.
criterion = nn.CrossEntropyLoss(
    ignore_index=pad_id,
    label_smoothing=0.1
)

# Optimizer.
optimizer = optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.01
)

# Scheduler για μείωση του learning rate όταν το validation loss δεν βελτιώνεται.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3,
    factor=0.5
)

# GradScaler για mixed precision training.
scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp
)

# Λίστες για αποθήκευση των losses ανά εποχή.
train_losses = []
val_losses = []

best_val_loss = float("inf")
counter = 0

print(f"Έναρξη εκπαίδευσης για {EPOCHS} εποχές στη συσκευή: {device}")


# ==========================================
# Κύριο loop εκπαίδευσης
# ==========================================

for epoch in range(EPOCHS):
    start_time = time.time()

    # ======================================
    # Φάση εκπαίδευσης
    # ======================================

    model.train()
    total_train_loss = 0.0

    train_pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]"
    )

    for batch in train_pbar:
        inputs, targets, properties = batch

        inputs = inputs.to(device)
        targets = targets.to(device).contiguous()
        properties = properties.to(device).float()

        optimizer.zero_grad()

        with torch.amp.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            logits = model(inputs, properties)

            loss = criterion(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1)
            )

        scaler.scale(loss).backward()

        # Gradient clipping για πιο σταθερή εκπαίδευση.
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        total_train_loss += loss.item()
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)


    # ======================================
    # Φάση validation
    # ======================================

    model.eval()
    total_val_loss = 0.0

    # Εδώ χρησιμοποιούμε val_loader, όχι test_loader.
    val_pbar = tqdm(
        val_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS} [Val]"
    )

    with torch.no_grad():
        for batch in val_pbar:
            inputs, targets, properties = batch

            inputs = inputs.to(device)
            targets = targets.to(device).contiguous()
            properties = properties.to(device).float()

            with torch.amp.autocast(
                device_type=device.type,
                enabled=use_amp
            ):
                logits = model(inputs, properties)

                loss = criterion(
                    logits.reshape(-1, vocab_size),
                    targets.reshape(-1)
                )

            total_val_loss += loss.item()
            val_pbar.set_postfix(val_loss=f"{loss.item():.4f}")

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    scheduler.step(avg_val_loss)


    # ======================================
    # Αποθήκευση και ενημέρωση προόδου
    # ======================================

    epoch_time = time.time() - start_time

    print(f"Εποχή {epoch + 1} ολοκληρώθηκε σε {epoch_time:.1f}s")
    print(f"Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f}")

    # Αποθήκευση του καλύτερου μοντέλου με βάση το validation loss.
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_losses": train_losses,
                "val_losses": val_losses,
                "best_val_loss": best_val_loss,
                "vocab": vocab,
                "token_to_id": token_to_id,
                "id_to_token": id_to_token,
                "max_len": MAX_LEN
            },
            best_model_path
        )

        print(f"Νέο καλύτερο μοντέλο αποθηκεύτηκε: {best_model_path}")

    else:
        counter += 1

    # Περιοδική αποθήκευση checkpoint.
    if (epoch + 1) % checkpoint_interval == 0:
        checkpoint_path = f"{checkpoint_prefix}_{epoch + 1}.pth"

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_losses": train_losses,
                "val_losses": val_losses,
                "best_val_loss": best_val_loss,
                "vocab": vocab,
                "token_to_id": token_to_id,
                "id_to_token": id_to_token,
                "max_len": MAX_LEN
            },
            checkpoint_path
        )

        print(f"Checkpoint αποθηκεύτηκε: {checkpoint_path}")

    # Early stopping αν το validation loss δεν βελτιώνεται.
    if counter >= patience:
        print(f"Early stopping στην εποχή {epoch + 1}.")
        break


print("Η εκπαίδευση ολοκληρώθηκε.")

In [ ]:
# ==========================================
# Φόρτωση του καλύτερου εκπαιδευμένου μοντέλου
# ==========================================

checkpoint_path = "best_molecule_transformer_clean_split.pth"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Δεν βρέθηκε το checkpoint: {checkpoint_path}")

# Φόρτωση checkpoint
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

# Ανάκτηση vocabulary και mappings από το checkpoint
vocab = checkpoint["vocab"]
token_to_id = checkpoint["token_to_id"]
id_to_token = checkpoint["id_to_token"]
MAX_LEN = checkpoint["max_len"]

# Αρχικοποίηση του μοντέλου με την ίδια αρχιτεκτονική που χρησιμοποιήθηκε στην εκπαίδευση
PAD_IDX = token_to_id["<PAD>"]

model = MoleculeTransformer(
    vocab_size=len(vocab),
    embed_size=256,
    num_heads=8,
    num_layers=4,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX
).to(device)

# Φόρτωση των βαρών του μοντέλου
model.load_state_dict(checkpoint["model_state_dict"])

# Θέτουμε το μοντέλο σε evaluation mode για παραγωγή μορίων
model.eval()

print(f"Το μοντέλο φορτώθηκε από: {checkpoint_path}")
print(f"Epoch καλύτερου μοντέλου: {checkpoint['epoch']}")
print(f"Best Validation Loss: {checkpoint['best_val_loss']:.4f}")
print(f"Συνολικές παράμετροι: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ==========================================
# Held-Out Test Evaluation
# ==========================================

import math

# Φορτώνουμε ρητά το καλύτερο μοντέλο
checkpoint_path = "best_molecule_transformer_clean_split.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

vocab = checkpoint["vocab"]
token_to_id = checkpoint["token_to_id"]
id_to_token = checkpoint["id_to_token"]
MAX_LEN = checkpoint["max_len"]

PAD_IDX = token_to_id["<PAD>"]

# Ίδια ακριβώς αρχιτεκτονική με το training
model = MoleculeTransformer(
    vocab_size=len(vocab),
    embed_size=256,
    num_heads=8,
    num_layers=4,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Ίδιο ακριβώς loss με Train / Validation
criterion_test = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX,
    label_smoothing=0.1
)

use_amp = device.type == "cuda"

total_test_loss = 0.0
correct_tokens = 0
total_tokens = 0

with torch.no_grad():

    test_pbar = tqdm(
        test_loader,
        desc="Held-Out Test"
    )

    for inputs, targets, properties in test_pbar:

        inputs = inputs.to(device)
        targets = targets.to(device).contiguous()

        # Διατηρείται [batch_size, 1]
        properties = properties.to(device).float()

        with torch.amp.autocast(
            device_type=device.type,
            enabled=use_amp
        ):
            logits = model(inputs, properties)

            loss = criterion_test(
                logits.reshape(-1, len(vocab)),
                targets.reshape(-1)
            )

        total_test_loss += loss.item()

        # Token accuracy μόνο στα πραγματικά tokens
        predictions = logits.argmax(dim=-1)
        valid_mask = targets != PAD_IDX

        correct_tokens += (
            (predictions == targets) & valid_mask
        ).sum().item()

        total_tokens += valid_mask.sum().item()


# Ίδιος τρόπος averaging με το validation loop
avg_test_loss = total_test_loss / len(test_loader)

test_perplexity = math.exp(avg_test_loss)

test_token_accuracy = (
    100.0 * correct_tokens / total_tokens
)

print(f"Checkpoint: {checkpoint_path}")
print(f"Checkpoint Epoch: {checkpoint['epoch']}")
print(f"PAD ID: {PAD_IDX}")
print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Perplexity: {test_perplexity:.4f}")
print(f"Test Token Accuracy: {test_token_accuracy:.4f}%")

In [ ]:
# 1. Ορισμός διαδρομής
checkpoint_path = 'checkpoint_training_epoch_100.pth' 

if os.path.exists(checkpoint_path):
    # 2. Φόρτωση των πάντων (Losses + Βάρη Μοντέλου)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    train_losses = checkpoint['train_losses']
    val_losses = checkpoint['val_losses']
    
    # Φόρτωση των βαρών στο μοντέλο 
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval() # Θέτουμε το μοντέλο σε evaluation mode
    
    print(f"Το μοντέλο και τα losses ανακτήθηκαν από το {checkpoint_path}")

    # 3. Δημιουργία γραφήματος
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss', color='#1f77b4', lw=2)
    plt.plot(val_losses, label='Validation Loss', color='#ff7f0e', lw=2)
    plt.title('Transformer Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.savefig('learning_curve_final_clean_split.png', dpi=300)
    plt.show()
else:
    print(f"Το αρχείο {checkpoint_path} δεν βρέθηκε!")

### 11. Διαδικασία Παραγωγής Μορίων (Inference & De Novo Generation)
Η συνάρτηση generate_diverse_molecules αποτελεί τον μηχανισμό με τον οποίο το εκπαιδευμένο μοντέλο "σχεδιάζει" νέες χημικές δομές. Η διαδικασία αυτή ακολουθεί τη λογική της καθοδηγούμενης παραγωγής (Conditional Generation):

Property Injection (Έγχυση Ιδιότητας): Σε αντίθεση με την απλή παραγωγή, εδώ τροφοδοτούμε το μοντέλο με έναν συγκεκριμένο αριθμητικό στόχο (target_logp). Αυτή η τιμή λειτουργεί ως "κατευθυντήριο σήμα" σε κάθε βήμα πρόβλεψης, αναγκάζοντας τον Transformer να επιλέξει tokens που οδηγούν στο επιθυμητό φυσικοχημικό προφίλ.

Αυτοπαλινδρομική Δειγματοληψία (Autoregressive Sampling): Η παραγωγή ξεκινά με το token <START>. Το μοντέλο προβλέπει το επόμενο χημικό σύμβολο, το οποίο στη συνέχεια προστίθεται στην είσοδο για να προβλεφθεί το μεθεπόμενο, μέχρι να παραχθεί το token τερματισμού <END> ή να φτάσουμε το max_length.

**Ρυθμίσεις Δημιουργικότητας & Εγκυρότητας:**

Temperature Scaling: Χρησιμοποιώ την παράμετρο temperature (π.χ. 0.8) για να ελέγξω την κατανομή πιθανοτήτων. Χαμηλές τιμές κάνουν το μοντέλο πιο "σίγουρο" και συντηρητικό, ενώ υψηλές τιμές αυξάνουν τη δημιουργικότητα και την ποικιλομορφία (diversity), ρισκάροντας όμως τη χημική εγκυρότητα.

Multinomial Sampling: Αντί για την επιλογή του συμβόλου με την απόλυτα μεγαλύτερη πιθανότητα, χρησιμοποιώ τυχαία δειγματοληψία βάσει της κατανομής πιθανοτήτων. Αυτό διασφαλίζει ότι το μοντέλο δεν θα παράγει το ίδιο μόριο επανειλημμένα για τον ίδιο στόχο.

**Μετα-επεξεργασία & Επαλήθευση (RDKit & SELFIES):**

Robust Decoding: Χρησιμοποιώ τη βιβλιοθήκη SELFIES για τη μετατροπή των tokens σε SMILES, καθώς εγγυάται ότι οι αλληλουχίες αντιστοιχούν σε μαθηματικά έγκυρους γράφους.

Χημικός Έλεγχος: Μέσω της RDKit, κάθε παραγόμενο μόριο ελέγχεται για τη χημική του εγκυρότητα (Valency check). Αν το μόριο είναι έγκυρο, υπολογίζεται η πραγματική του λιποφιλία (Actual_logP) για να συγκριθεί αργότερα με τον στόχο που δώσαμε.

**Βελτιστοποίηση Πόρων:**

model.eval() & torch.no_grad(): Θέτω το μοντέλο σε κατάσταση αξιολόγησης και απενεργοποιώ τον υπολογισμό κλίσεων. Αυτό μειώνει δραστικά την κατανάλωση μνήμης της GPU και επιταχύνει τη διαδικασία παραγωγής χιλιάδων μορίων.

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 1. ΠΡΟΕΤΟΙΜΑΣΙΑ ΜΟΝΤΕΛΟΥ & ΣΥΣΚΕΥΗΣ ---

# Φορτώνουμε το καλύτερο μοντέλο από το νέο training.
checkpoint_path = "best_molecule_transformer_clean_split.pth"

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Δεν βρέθηκε το checkpoint: {checkpoint_path}")

print(f"Προετοιμασία φόρτωσης από: {checkpoint_path}")

# Φόρτωση checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
# Ανάκτηση vocabulary και mappings από το checkpoint
vocab = checkpoint["vocab"]
token_to_id = checkpoint["token_to_id"]
id_to_token = checkpoint["id_to_token"]
MAX_LEN = checkpoint["max_len"]

# Αρχικοποίηση μοντέλου με τις ίδιες παραμέτρους που χρησιμοποιήθηκαν στην εκπαίδευση
PAD_IDX = token_to_id["<PAD>"]

model = MoleculeTransformer(
    vocab_size=len(vocab),
    embed_size=256,
    num_heads=8,
    num_layers=4,
    max_len=MAX_LEN,
    pad_idx=PAD_IDX
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Το μοντέλο φορτώθηκε επιτυχώς στη συσκευή: {device}")
print(f"Checkpoint epoch: {checkpoint['epoch']}")
print(f"Best Validation Loss: {checkpoint['best_val_loss']:.4f}")


# --- 2. ΣΥΝΑΡΤΗΣΗ ΜΑΖΙΚΗΣ ΠΑΡΑΓΩΓΗΣ ---

def mass_generate_library(model, total_count=150000, batch_size=128, max_len=90):
    output_file = "GENERATED_LIBRARY_150K.csv"
    
    # Δημιουργία τυχαίων στόχων logP στο εύρος που θέλουμε να ελέγξει το μοντέλο.
    target_logps = np.random.uniform(-2.0, 6.0, total_count)
    
    start_id = token_to_id["<START>"]
    end_id = token_to_id["<END>"]
    pad_id = token_to_id["<PAD>"]
    
    all_data = []
    pbar = tqdm(total=total_count, desc="Παραγωγή Μορίων")

    # Επεξεργασία ανά batch
    for i in range(0, total_count, batch_size):
        current_batch_targets = target_logps[i : i + batch_size]
        actual_bs = len(current_batch_targets)
        
        # Αρχικοποίηση κάθε ακολουθίας με το <START> token
        current_tokens = torch.full(
            (actual_bs, 1),
            start_id,
            device=device
        ).long()
        
        prop_tensor = torch.tensor(
            current_batch_targets,
            device=device
        ).float().reshape(-1, 1)
        
        finished = torch.zeros(actual_bs, dtype=torch.bool, device=device)
        
        with torch.no_grad():
            for _ in range(max_len - 1):
                # Forward pass
                logits = model(current_tokens, prop_tensor)
                
                # Sampling από το τελευταίο token με temperature
                next_token_logits = logits[:, -1, :] / 0.8
                
                # Top-K Sampling με K=10
                v, _ = torch.topk(next_token_logits, 10)
                next_token_logits[next_token_logits < v[:, [-1]]] = -float("Inf")
                
                probs = torch.softmax(next_token_logits, dim=-1)
                next_tokens = torch.multinomial(probs, num_samples=1)
                
                # Όσες ακολουθίες έχουν ήδη τελειώσει, συμπληρώνονται με <PAD>
                next_tokens[finished] = pad_id
                
                # Προσθήκη του νέου token στην ακολουθία
                current_tokens = torch.cat([current_tokens, next_tokens], dim=1)
                
                # Ενημέρωση για το ποιες ακολουθίες έφτασαν στο <END>
                finished |= (next_tokens.squeeze(1) == end_id)
                
                if finished.all():
                    break

        # Decoding και υπολογισμός πραγματικού logP με RDKit
        indices_np = current_tokens.cpu().numpy()
        
        for idx in range(actual_bs):
            ids = indices_np[idx]
            
            clean_tokens = []
            
            # Κρατάμε tokens μέχρι το πρώτο <END>.
            # Αγνοούμε <START> και <PAD>.
            for tid in ids:
                if tid == end_id:
                    break
                
                if tid not in [start_id, pad_id]:
                    clean_tokens.append(id_to_token[tid])
            
            selfie_str = "".join(clean_tokens)
            
            try:
                smiles_str = sf.decoder(selfie_str)
                mol = Chem.MolFromSmiles(smiles_str)
                
                if mol:
                    actual_logp = Descriptors.MolLogP(mol)
                    
                    all_data.append({
                        "Target_logP": round(float(current_batch_targets[idx]), 3),
                        "Actual_logP": round(actual_logp, 3),
                        "SMILES": smiles_str,
                        "SELFIES": selfie_str
                    })
            
            except Exception:
                continue
        
        pbar.update(actual_bs)

        # Ενδιάμεση αποθήκευση ανά 10.000 έγκυρα μόρια
        if len(all_data) > 0 and len(all_data) % 10000 == 0:
            pd.DataFrame(all_data).to_csv(output_file, index=False)

    # Τελικό DataFrame και αφαίρεση διπλοτύπων
        final_df = pd.DataFrame(all_data)
        initial_len = len(final_df)

        final_df = final_df.drop_duplicates(subset=["SMILES"])

        final_df.to_csv(output_file, index=False)
    pbar.close()
    
    print(f"\nΗ παραγωγή ολοκληρώθηκε!")
    print(f"Σύνολο έγκυρων μορίων: {initial_len}")
    print(f"Μοναδικά μόρια (Unique): {len(final_df)}")
    print(f"Το αρχείο '{output_file}' αποθηκεύτηκε.")
    
    return final_df


# --- 3. ΕΝΑΡΞΗ ΕΚΤΕΛΕΣΗΣ ---

final_library_df = mass_generate_library(
    model,
    total_count=150000,
    batch_size=512,
    max_len=MAX_LEN
)

In [ ]:
# ==========================================
# Conditional logP Calibration by Target Bins
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

input_file = "CLEAN_GENERATED_LIBRARY_FINAL_clean_split.csv"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Δεν βρέθηκε το αρχείο: {input_file}")

calibration_df = pd.read_csv(input_file)

required_columns = ["Target_logP", "Actual_logP"]

missing_columns = [
    col for col in required_columns
    if col not in calibration_df.columns
]

if missing_columns:
    raise ValueError(
        f"Λείπουν απαραίτητες στήλες: {missing_columns}"
    )

calibration_df["Target_logP"] = pd.to_numeric(
    calibration_df["Target_logP"],
    errors="coerce"
)

calibration_df["Actual_logP"] = pd.to_numeric(
    calibration_df["Actual_logP"],
    errors="coerce"
)

calibration_df = calibration_df.dropna(
    subset=["Target_logP", "Actual_logP"]
).copy()


# ==========================================
# Errors
# ==========================================

calibration_df["Error"] = (
    calibration_df["Actual_logP"]
    - calibration_df["Target_logP"]
)

calibration_df["Absolute_Error"] = (
    calibration_df["Error"].abs()
)

calibration_df["Squared_Error"] = (
    calibration_df["Error"] ** 2
)

calibration_df["Within_0_5"] = (
    calibration_df["Absolute_Error"] <= 0.5
)

calibration_df["Within_1_0"] = (
    calibration_df["Absolute_Error"] <= 1.0
)


# ==========================================
# Fixed target-logP bins
# ==========================================

bin_edges = [
    -2.0,
    -1.0,
     0.0,
     1.0,
     2.0,
     3.0,
     4.0,
     5.0,
     6.000001
]

bin_labels = [
    "[-2, -1)",
    "[-1, 0)",
    "[0, 1)",
    "[1, 2)",
    "[2, 3)",
    "[3, 4)",
    "[4, 5)",
    "[5, 6]"
]

calibration_df["Target_Bin"] = pd.cut(
    calibration_df["Target_logP"],
    bins=bin_edges,
    labels=bin_labels,
    include_lowest=True,
    right=False
)


# ==========================================
# Calibration summary
# ==========================================

calibration_summary = (
    calibration_df
    .groupby("Target_Bin", observed=True)
    .agg(
        N=("Target_logP", "size"),
        Mean_Target_logP=("Target_logP", "mean"),
        Mean_Actual_logP=("Actual_logP", "mean"),
        Bias=("Error", "mean"),
        MAE=("Absolute_Error", "mean"),
        MSE=("Squared_Error", "mean"),
        Within_0_5=("Within_0_5", "mean"),
        Within_1_0=("Within_1_0", "mean")
    )
    .reset_index()
)

calibration_summary["RMSE"] = np.sqrt(
    calibration_summary["MSE"]
)

calibration_summary["Within_0_5_Percent"] = (
    calibration_summary["Within_0_5"] * 100
)

calibration_summary["Within_1_0_Percent"] = (
    calibration_summary["Within_1_0"] * 100
)

calibration_summary = calibration_summary[
    [
        "Target_Bin",
        "N",
        "Mean_Target_logP",
        "Mean_Actual_logP",
        "Bias",
        "MAE",
        "RMSE",
        "Within_0_5_Percent",
        "Within_1_0_Percent"
    ]
]

numeric_columns = calibration_summary.select_dtypes(
    include=[np.number]
).columns

calibration_summary[numeric_columns] = (
    calibration_summary[numeric_columns].round(4)
)


# ==========================================
# Overall target-output alignment
# ==========================================

overall_mae = calibration_df["Absolute_Error"].mean()
overall_bias = calibration_df["Error"].mean()
overall_rmse = np.sqrt(
    calibration_df["Squared_Error"].mean()
)

pearson_r = calibration_df[
    ["Target_logP", "Actual_logP"]
].corr().iloc[0, 1]

slope, intercept = np.polyfit(
    calibration_df["Target_logP"],
    calibration_df["Actual_logP"],
    1
)


print("=" * 65)
print("CONDITIONAL logP CALIBRATION")
print("=" * 65)

print(
    f"Canonical-unique molecules evaluated: "
    f"{len(calibration_df):,}"
)

print(f"Overall MAE: {overall_mae:.4f}")
print(f"Overall RMSE: {overall_rmse:.4f}")
print(f"Overall Bias: {overall_bias:.4f}")
print(f"Target–Actual Pearson r: {pearson_r:.4f}")
print(f"Calibration slope: {slope:.4f}")
print(f"Calibration intercept: {intercept:.4f}")

print("\nCalibration by target logP bin:")
display(calibration_summary)


# ==========================================
# Calibration plot
# ==========================================

plt.figure(figsize=(8, 6))

plt.plot(
    calibration_summary["Mean_Target_logP"],
    calibration_summary["Mean_Actual_logP"],
    marker="o",
    linewidth=2,
    label="Generated molecules"
)

identity_min = min(
    calibration_summary["Mean_Target_logP"].min(),
    calibration_summary["Mean_Actual_logP"].min()
)

identity_max = max(
    calibration_summary["Mean_Target_logP"].max(),
    calibration_summary["Mean_Actual_logP"].max()
)

plt.plot(
    [identity_min, identity_max],
    [identity_min, identity_max],
    linestyle="--",
    label="Ideal calibration"
)

plt.xlabel("Mean Target logP")
plt.ylabel("Mean Actual logP")
plt.title("Conditional logP Calibration by Target Bin")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "CONDITIONAL_LOGP_CALIBRATION_BY_BIN.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ==========================================
# Save table
# ==========================================

output_file = "CONDITIONAL_LOGP_CALIBRATION_BY_BIN.csv"

calibration_summary.to_csv(
    output_file,
    index=False
)

print(f"\nΑποθηκεύτηκε: {output_file}")
print(
    "Αποθηκεύτηκε: "
    "CONDITIONAL_LOGP_CALIBRATION_BY_BIN.png"
)

In [ ]:
# ==========================================
# Permutation / Null Test for logP Alignment
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

input_file = "CLEAN_GENERATED_LIBRARY_FINAL_clean_split.csv"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"Δεν βρέθηκε το αρχείο: {input_file}")

null_df = pd.read_csv(input_file)

null_df["Target_logP"] = pd.to_numeric(
    null_df["Target_logP"],
    errors="coerce"
)

null_df["Actual_logP"] = pd.to_numeric(
    null_df["Actual_logP"],
    errors="coerce"
)

null_df = null_df.dropna(
    subset=["Target_logP", "Actual_logP"]
).copy()

target = null_df["Target_logP"].to_numpy()
actual = null_df["Actual_logP"].to_numpy()

# ------------------------------------------
# Observed alignment
# ------------------------------------------

observed_r = np.corrcoef(
    target,
    actual
)[0, 1]

observed_mae = np.mean(
    np.abs(actual - target)
)

# ------------------------------------------
# Permutation null distribution
# ------------------------------------------

N_PERMUTATIONS = 1000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)

null_correlations = np.empty(N_PERMUTATIONS)
null_maes = np.empty(N_PERMUTATIONS)

for i in range(N_PERMUTATIONS):

    shuffled_target = rng.permutation(target)

    null_correlations[i] = np.corrcoef(
        shuffled_target,
        actual
    )[0, 1]

    null_maes[i] = np.mean(
        np.abs(actual - shuffled_target)
    )

# ------------------------------------------
# Empirical permutation p-values
# ------------------------------------------

p_value_r = (
    1
    + np.sum(
        np.abs(null_correlations)
        >= abs(observed_r)
    )
) / (N_PERMUTATIONS + 1)

p_value_mae = (
    1
    + np.sum(
        null_maes <= observed_mae
    )
) / (N_PERMUTATIONS + 1)

# ------------------------------------------
# Results
# ------------------------------------------

print("=" * 65)
print("PERMUTATION / NULL TEST — TARGET–OUTPUT ALIGNMENT")
print("=" * 65)

print(f"Molecules evaluated: {len(null_df):,}")
print(f"Permutations: {N_PERMUTATIONS:,}")

print("\nObserved:")
print(f"Pearson r: {observed_r:.4f}")
print(f"MAE:       {observed_mae:.4f}")

print("\nNull distribution:")
print(
    f"Pearson r: "
    f"{null_correlations.mean():.4f} "
    f"± {null_correlations.std():.4f}"
)

print(
    f"MAE:       "
    f"{null_maes.mean():.4f} "
    f"± {null_maes.std():.4f}"
)

print("\nPermutation significance:")
print(f"Pearson r empirical p-value: {p_value_r:.6f}")
print(f"MAE empirical p-value:       {p_value_mae:.6f}")


# ------------------------------------------
# Plot
# ------------------------------------------

plt.figure(figsize=(8, 6))

plt.hist(
    null_correlations,
    bins=40,
    alpha=0.8,
    label="Permuted target-output pairs"
)

plt.axvline(
    observed_r,
    linestyle="--",
    linewidth=2,
    label=f"Observed r = {observed_r:.4f}"
)

plt.xlabel("Pearson Correlation")
plt.ylabel("Frequency")
plt.title("Permutation Test for Conditional logP Alignment")
plt.legend()
plt.tight_layout()

plt.savefig(
    "LOGP_ALIGNMENT_PERMUTATION_TEST.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------
# Save results
# ------------------------------------------

permutation_summary = pd.DataFrame({
    "Metric": [
        "Observed_Pearson_r",
        "Null_Mean_Pearson_r",
        "Null_SD_Pearson_r",
        "Pearson_r_Empirical_p",
        "Observed_MAE",
        "Null_Mean_MAE",
        "Null_SD_MAE",
        "MAE_Empirical_p"
    ],
    "Value": [
        observed_r,
        null_correlations.mean(),
        null_correlations.std(),
        p_value_r,
        observed_mae,
        null_maes.mean(),
        null_maes.std(),
        p_value_mae
    ]
})

permutation_summary.to_csv(
    "LOGP_ALIGNMENT_PERMUTATION_TEST.csv",
    index=False
)

print(
    "\nΑποθηκεύτηκε: "
    "LOGP_ALIGNMENT_PERMUTATION_TEST.csv"
)

print(
    "Αποθηκεύτηκε: "
    "LOGP_ALIGNMENT_PERMUTATION_TEST.png"
)

In [ ]:
# ==========================================
# Sampling Sensitivity Analysis
# Temperatures: 0.6 / 0.8 / 1.0
# ==========================================

import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors
from tqdm import tqdm
import selfies as sf

# ------------------------------------------
# Settings
# ------------------------------------------

TEMPERATURES = [0.6, 0.8, 1.0]

N_SAMPLES = 10000
BATCH_SIZE = 512
TOP_K = 10
RANDOM_SEED = 42

start_id = token_to_id["<START>"]
end_id = token_to_id["<END>"]
pad_id = token_to_id["<PAD>"]

# Ίδιοι ακριβώς target logP στόχοι
# για όλα τα temperatures
rng = np.random.default_rng(RANDOM_SEED)

shared_target_logps = rng.uniform(
    -2.0,
    6.0,
    N_SAMPLES
)


# ==========================================
# Generation function
# ==========================================

def generate_temperature_sample(
    model,
    target_logps,
    temperature,
    batch_size=512,
    max_len=90,
    top_k=10
):

    generated_rows = []
    valid_count = 0

    pbar = tqdm(
        total=len(target_logps),
        desc=f"Temperature {temperature}"
    )

    for i in range(
        0,
        len(target_logps),
        batch_size
    ):

        current_batch_targets = (
            target_logps[i:i + batch_size]
        )

        actual_bs = len(
            current_batch_targets
        )

        current_tokens = torch.full(
            (actual_bs, 1),
            start_id,
            device=device
        ).long()

        prop_tensor = torch.tensor(
            current_batch_targets,
            device=device
        ).float().reshape(-1, 1)

        finished = torch.zeros(
            actual_bs,
            dtype=torch.bool,
            device=device
        )

        with torch.no_grad():

            for _ in range(max_len - 1):

                logits = model(
                    current_tokens,
                    prop_tensor
                )

                # ΜΟΝΗ αλλαγή από το original generation:
                # διαφορετικό temperature
                next_token_logits = (
                    logits[:, -1, :]
                    / temperature
                )

                # Ίδιο Top-K = 10
                top_values, _ = torch.topk(
                    next_token_logits,
                    top_k
                )

                next_token_logits[
                    next_token_logits
                    < top_values[:, [-1]]
                ] = -float("Inf")

                probs = torch.softmax(
                    next_token_logits,
                    dim=-1
                )

                next_tokens = torch.multinomial(
                    probs,
                    num_samples=1
                )

                next_tokens[finished] = pad_id

                current_tokens = torch.cat(
                    [
                        current_tokens,
                        next_tokens
                    ],
                    dim=1
                )

                finished |= (
                    next_tokens.squeeze(1)
                    == end_id
                )

                if finished.all():
                    break

        indices_np = (
            current_tokens
            .cpu()
            .numpy()
        )

        for idx in range(actual_bs):

            ids = indices_np[idx]

            clean_tokens = []

            for tid in ids:

                if tid == end_id:
                    break

                if tid not in [
                    start_id,
                    pad_id
                ]:
                    clean_tokens.append(
                        id_to_token[tid]
                    )

            selfie_str = "".join(
                clean_tokens
            )

            try:

                smiles_str = sf.decoder(
                    selfie_str
                )

                mol = Chem.MolFromSmiles(
                    smiles_str
                )

                if mol is None:
                    continue

                valid_count += 1

                actual_logp = (
                    Descriptors.MolLogP(mol)
                )

                canonical_smiles = (
                    Chem.MolToSmiles(
                        mol,
                        canonical=True
                    )
                )

                generated_rows.append({
                    "Temperature": temperature,
                    "Target_logP": float(
                        current_batch_targets[idx]
                    ),
                    "Actual_logP": float(
                        actual_logp
                    ),
                    "SMILES": canonical_smiles,
                    "SELFIES": selfie_str
                })

            except Exception:
                continue

        pbar.update(actual_bs)

    pbar.close()

    result_df = pd.DataFrame(
        generated_rows
    )

    return result_df, valid_count


# ==========================================
# Run temperatures
# ==========================================

model.eval()

all_temperature_results = []
summary_rows = []

for temperature in TEMPERATURES:

    # Ίδιο seed πριν από κάθε temperature
    # για reproducible comparison
    torch.manual_seed(RANDOM_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            RANDOM_SEED
        )

    temp_df, valid_count = (
        generate_temperature_sample(
            model=model,
            target_logps=shared_target_logps,
            temperature=temperature,
            batch_size=BATCH_SIZE,
            max_len=MAX_LEN,
            top_k=TOP_K
        )
    )

    # --------------------------------------
    # Metrics
    # --------------------------------------

    validity = (
        valid_count
        / N_SAMPLES
    )

    unique_count = (
        temp_df["SMILES"]
        .nunique()
    )

    uniqueness = (
        unique_count
        / valid_count
        if valid_count > 0
        else np.nan
    )

    temp_df["Absolute_Error"] = (
        temp_df["Actual_logP"]
        - temp_df["Target_logP"]
    ).abs()

    mae = (
        temp_df["Absolute_Error"]
        .mean()
    )

    pearson_r = (
        temp_df[
            [
                "Target_logP",
                "Actual_logP"
            ]
        ]
        .corr()
        .iloc[0, 1]
    )

    summary_rows.append({
        "Temperature": temperature,
        "Attempts": N_SAMPLES,
        "Valid": valid_count,
        "Validity_Percent":
            100 * validity,
        "Unique": unique_count,
        "Uniqueness_Percent":
            100 * uniqueness,
        "logP_MAE": mae,
        "Target_Actual_Pearson_r":
            pearson_r
    })

    all_temperature_results.append(
        temp_df
    )


# ==========================================
# Final summary
# ==========================================

temperature_summary = pd.DataFrame(
    summary_rows
)

numeric_cols = (
    temperature_summary
    .select_dtypes(
        include=[np.number]
    )
    .columns
)

temperature_summary[
    numeric_cols
] = temperature_summary[
    numeric_cols
].round(4)


print("=" * 70)
print("SAMPLING TEMPERATURE SENSITIVITY")
print("=" * 70)

display(
    temperature_summary
)


# ==========================================
# Save results
# ==========================================

temperature_summary.to_csv(
    "SAMPLING_TEMPERATURE_SENSITIVITY.csv",
    index=False
)

all_temperature_df = pd.concat(
    all_temperature_results,
    ignore_index=True
)

all_temperature_df.to_csv(
    "SAMPLING_TEMPERATURE_GENERATED_MOLECULES.csv",
    index=False
)

print(
    "\nΑποθηκεύτηκε: "
    "SAMPLING_TEMPERATURE_SENSITIVITY.csv"
)

print(
    "Αποθηκεύτηκε: "
    "SAMPLING_TEMPERATURE_GENERATED_MOLECULES.csv"
)

In [ ]:
# ==========================================
# Multi-Seed Sampling Stability
# Temperature = 0.8
# ==========================================

import numpy as np
import pandas as pd
import torch

# ------------------------------------------
# Settings
# ------------------------------------------

SEEDS = [42, 123, 2026]

N_SAMPLES = 10000
TEMPERATURE = 0.8
BATCH_SIZE = 512
TOP_K = 10

# Ίδιοι ακριβώς target logP στόχοι
# για όλα τα seeds
target_rng = np.random.default_rng(42)

stability_target_logps = target_rng.uniform(
    -2.0,
    6.0,
    N_SAMPLES
)

# Βεβαιωνόμαστε ότι χρησιμοποιείται
# το best checkpoint
checkpoint_path = "best_molecule_transformer_clean_split.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(
    f"Checkpoint: {checkpoint_path} "
    f"(epoch {checkpoint['epoch']})"
)


# ==========================================
# Run generation for each seed
# ==========================================

seed_summary_rows = []
all_seed_results = []

for seed in SEEDS:

    # Αλλάζουμε μόνο το sampling randomness
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_df, valid_count = generate_temperature_sample(
        model=model,
        target_logps=stability_target_logps,
        temperature=TEMPERATURE,
        batch_size=BATCH_SIZE,
        max_len=MAX_LEN,
        top_k=TOP_K
    )

    seed_df["Seed"] = seed

    # --------------------------------------
    # Metrics
    # --------------------------------------

    validity_percent = (
        100.0
        * valid_count
        / N_SAMPLES
    )

    unique_count = (
        seed_df["SMILES"]
        .nunique()
    )

    uniqueness_percent = (
        100.0
        * unique_count
        / valid_count
        if valid_count > 0
        else np.nan
    )

    absolute_error = (
        seed_df["Actual_logP"]
        - seed_df["Target_logP"]
    ).abs()

    logp_mae = absolute_error.mean()

    pearson_r = (
        seed_df[
            ["Target_logP", "Actual_logP"]
        ]
        .corr()
        .iloc[0, 1]
    )

    seed_summary_rows.append({
        "Seed": seed,
        "Attempts": N_SAMPLES,
        "Valid": valid_count,
        "Validity_Percent": validity_percent,
        "Unique": unique_count,
        "Uniqueness_Percent": uniqueness_percent,
        "logP_MAE": logp_mae,
        "Target_Actual_Pearson_r": pearson_r
    })

    all_seed_results.append(seed_df)


# ==========================================
# Per-seed results
# ==========================================

seed_stability_summary = pd.DataFrame(
    seed_summary_rows
)

display_summary = seed_stability_summary.copy()

numeric_columns = [
    "Validity_Percent",
    "Uniqueness_Percent",
    "logP_MAE",
    "Target_Actual_Pearson_r"
]

display_summary[numeric_columns] = (
    display_summary[numeric_columns]
    .round(4)
)

print("\n" + "=" * 70)
print("MULTI-SEED SAMPLING STABILITY")
print("=" * 70)

display(display_summary)


# ==========================================
# Mean ± SD across seeds
# ==========================================

stability_metrics = [
    "Validity_Percent",
    "Uniqueness_Percent",
    "logP_MAE",
    "Target_Actual_Pearson_r"
]

stability_statistics = pd.DataFrame({
    "Metric": stability_metrics,
    "Mean": [
        seed_stability_summary[col].mean()
        for col in stability_metrics
    ],
    "SD": [
        seed_stability_summary[col].std(ddof=1)
        for col in stability_metrics
    ]
})

stability_statistics[
    ["Mean", "SD"]
] = stability_statistics[
    ["Mean", "SD"]
].round(4)

print("\nMean ± SD across seeds:")
display(stability_statistics)


# ==========================================
# Save results
# ==========================================

seed_stability_summary.to_csv(
    "MULTI_SEED_STABILITY_SUMMARY.csv",
    index=False
)

stability_statistics.to_csv(
    "MULTI_SEED_STABILITY_MEAN_SD.csv",
    index=False
)

all_seed_df = pd.concat(
    all_seed_results,
    ignore_index=True
)

all_seed_df.to_csv(
    "MULTI_SEED_STABILITY_GENERATED_MOLECULES.csv",
    index=False
)

print(
    "\nΑποθηκεύτηκε: "
    "MULTI_SEED_STABILITY_SUMMARY.csv"
)

print(
    "Αποθηκεύτηκε: "
    "MULTI_SEED_STABILITY_MEAN_SD.csv"
)

print(
    "Αποθηκεύτηκε: "
    "MULTI_SEED_STABILITY_GENERATED_MOLECULES.csv"
)

In [ ]:
# ==========================================
# Thesis methodological checks
# logP range, truncation and <UNK> statistics
# ==========================================

import numpy as np
import pandas as pd
import selfies as sf

print("=" * 70)
print("1. TRAINING logP DISTRIBUTION")
print("=" * 70)

train_logp = pd.to_numeric(train_df["logP"], errors="coerce").dropna()

print(f"Training molecules: {len(train_logp):,}")
print(f"Min logP: {train_logp.min():.4f}")
print(f"Max logP: {train_logp.max():.4f}")
print(f"Mean logP: {train_logp.mean():.4f}")
print(f"Median logP: {train_logp.median():.4f}")
print(f"Std logP: {train_logp.std():.4f}")

percentiles = train_logp.quantile(
    [0.01, 0.025, 0.05, 0.25, 0.50, 0.75, 0.95, 0.975, 0.99]
)

print("\nPercentiles:")
for q, value in percentiles.items():
    print(f"{q*100:5.1f}% : {value:.4f}")

inside_range = ((train_logp >= -2) & (train_logp <= 6)).sum()
below_range = (train_logp < -2).sum()
above_range = (train_logp > 6).sum()

print("\nGeneration target range [-2, 6]:")
print(
    f"Inside [-2, 6]: {inside_range:,} "
    f"({inside_range / len(train_logp) * 100:.4f}%)"
)
print(
    f"Below -2:       {below_range:,} "
    f"({below_range / len(train_logp) * 100:.4f}%)"
)
print(
    f"Above 6:        {above_range:,} "
    f"({above_range / len(train_logp) * 100:.4f}%)"
)


print("\n" + "=" * 70)
print("2. SEQUENCE LENGTH / TRUNCATION")
print("=" * 70)

def sequence_length_with_special_tokens(selfies_string):
    return len(list(sf.split_selfies(str(selfies_string)))) + 2
    # +2 for <START> and <END>

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    lengths = df["SELFIES"].apply(sequence_length_with_special_tokens)

    truncated = (lengths > MAX_LEN).sum()

    print(f"\n{name}:")
    print(f"Molecules: {len(lengths):,}")
    print(f"Min length: {lengths.min()}")
    print(f"Mean length: {lengths.mean():.2f}")
    print(f"Median length: {lengths.median():.2f}")
    print(f"95th percentile: {lengths.quantile(0.95):.2f}")
    print(f"99th percentile: {lengths.quantile(0.99):.2f}")
    print(f"Max length: {lengths.max()}")
    print(
        f"Sequences > MAX_LEN={MAX_LEN}: "
        f"{truncated:,} ({truncated / len(lengths) * 100:.4f}%)"
    )


print("\n" + "=" * 70)
print("3. <UNK> TOKEN STATISTICS")
print("=" * 70)

def count_unknown_tokens(df):
    total_tokens = 0
    unk_tokens = 0
    molecules_with_unk = 0

    for selfies_string in df["SELFIES"].astype(str):
        tokens = list(sf.split_selfies(selfies_string))

        current_unk = sum(
            token not in token_to_id
            for token in tokens
        )

        total_tokens += len(tokens)
        unk_tokens += current_unk

        if current_unk > 0:
            molecules_with_unk += 1

    return total_tokens, unk_tokens, molecules_with_unk


for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    total_tokens, unk_tokens, molecules_with_unk = count_unknown_tokens(df)

    print(f"\n{name}:")
    print(f"SELFIES tokens: {total_tokens:,}")
    print(
        f"<UNK> tokens: {unk_tokens:,} "
        f"({unk_tokens / total_tokens * 100:.6f}%)"
    )
    print(
        f"Molecules containing >=1 <UNK>: "
        f"{molecules_with_unk:,} "
        f"({molecules_with_unk / len(df) * 100:.6f}%)"
    )

print("\n" + "=" * 70)
print("CHECK COMPLETED")
print("=" * 70)

In [ ]:
import selfies as sf

print("=== TRUNCATION ===")

def seq_len(s):
    return len(list(sf.split_selfies(str(s)))) + 2  # START + END

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    lengths = df["SELFIES"].apply(seq_len)
    n = (lengths > MAX_LEN).sum()

    print(
        f"{name}: >{MAX_LEN} = {n:,} "
        f"({100*n/len(df):.6f}%), max={lengths.max()}, "
        f"p99={lengths.quantile(0.99):.1f}"
    )


print("\n=== UNKNOWN TOKENS ===")

for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    total = 0
    unk = 0
    mol_unk = 0

    for s in df["SELFIES"].astype(str):
        tokens = list(sf.split_selfies(s))
        u = sum(t not in token_to_id for t in tokens)

        total += len(tokens)
        unk += u
        mol_unk += (u > 0)

    print(
        f"{name}: UNK tokens={unk:,}/{total:,} "
        f"({100*unk/total:.6f}%), "
        f"molecules with UNK={mol_unk:,}/{len(df):,} "
        f"({100*mol_unk/len(df):.6f}%)"
    )

### 12. Καθαρισμός Δεδομένων και Τελική Στατιστική Αξιολόγηση

Μετά τη μαζική παραγωγή, ακολουθεί η διαδικασία του Data Cleaning και ο υπολογισμός των τελικών δεικτών απόδοσης. 

**Στάδια Επεξεργασίας:**
Φιλτράρισμα Εγκυρότητας (Validity Filtering): Αφαιρούνται τυχόν αποτυχημένες προσπάθειες παραγωγής ή μόρια που δεν μπόρεσαν να αναγνωριστούν από την RDKit. Το ποσοστό Validity αντικατοπτρίζει τη γνώση των χημικών κανόνων από το μοντέλο.

**Ανάλυση Μοναδικότητας (Uniqueness Analysis):** Εντοπίζονται και αφαιρούνται τα διπλότυπα SMILES. Η μοναδικότητα είναι ο σημαντικότερος δείκτης για την αποφυγή του Mode Collapse, διασφαλίζοντας ότι ο Transformer εξερεύνησε νέες περιοχές του χημικού χώρου αντί να επαναλαμβάνει γνώριμες δομές.

**Ερμηνεία των Metrics:**
Validity Rate: Δείχνει πόσο "γραμματισμένο" είναι το μοντέλο στη γλώσσα των SELFIES/SMILES. Τιμές >95% είναι εξαιρετικές.
Uniqueness Rate: Δείχνει την ποικιλομορφία της "φαντασίας" του μοντέλου.

 Στόχος είναι η παραγωγή μιας βιβλιοθήκης με τη μέγιστη δυνατή πρωτοτυπία.
 Final MAE (Mean Absolute Error): Είναι η τελική επιβεβαίωση της ακρίβειας. Μετρά πόσο κοντά έπεσε το μοντέλο στον στόχο $logP$ που του δώσαμε, χρησιμοποιώντας πλέον μόνο τα μοναδικά και έγκυρα μόρια.

In [ ]:
# ==========================================
# Τελική στατιστική αξιολόγηση generated molecules
# ==========================================

import os
import pandas as pd
from sklearn.model_selection import train_test_split
from rdkit import Chem

# ==========================================
# 1. Βασικές ρυθμίσεις από το generation log
# ==========================================

# Αυτές οι τιμές προέρχονται από το αποτέλεσμα της μαζικής παραγωγής:
# Συνολικές προσπάθειες: 150000
# Έγκυρα μόρια: 149995
TOTAL_ATTEMPTS = 150000
VALID_MOLECULES_FROM_GENERATION = 149995

generated_file = "GENERATED_LIBRARY_150K.csv"
processed_file = "processed_data.csv"

if not os.path.exists(generated_file):
    raise FileNotFoundError(f"Δεν βρέθηκε το αρχείο: {generated_file}")

if not os.path.exists(processed_file):
    raise FileNotFoundError(f"Δεν βρέθηκε το αρχείο: {processed_file}")


# ==========================================
# 2. Φόρτωση generated molecules
# ==========================================

# Το GENERATED_LIBRARY_150K.csv περιέχει τα μοναδικά μόρια
# που αποθηκεύτηκαν μετά την πρώτη αφαίρεση διπλοτύπων με βάση το SMILES.
df_generated = pd.read_csv(generated_file)

# Αφαιρούμε τυχόν κενές γραμμές σε βασικές στήλες
df_generated = df_generated.dropna(
    subset=["SMILES", "Actual_logP", "Target_logP"]
).copy()

unique_smiles_molecules = len(df_generated)


# ==========================================
# 3. Ανάκτηση train_df για υπολογισμό Novelty
# ==========================================

# Φορτώνουμε το καθαρισμένο dataset που χρησιμοποιήθηκε πριν το split.
df_all = pd.read_csv(processed_file)

# Αναδημιουργούμε ακριβώς το ίδιο train / validation / test split
# με το ίδιο random_state που χρησιμοποιήθηκε στην εκπαίδευση.
train_df, temp_df = train_test_split(
    df_all,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Το train / validation / test split αναδημιουργήθηκε επιτυχώς.")
print(f"Train Set:      {len(train_df)}")
print(f"Validation Set: {len(val_df)}")
print(f"Test Set:       {len(test_df)}")


# ==========================================
# 4. Canonicalization των generated SMILES
# ==========================================

def canonicalize_generated_smiles(smiles):
    """
    Μετατρέπει ένα generated SMILES σε canonical SMILES.
    Αυτό επιτρέπει αυστηρότερη αναγνώριση διπλότυπων μορίων.
    """
    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return None

        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

    except Exception:
        return None


df_generated["Canonical_SMILES"] = df_generated["SMILES"].apply(
    canonicalize_generated_smiles
)

df_generated = df_generated.dropna(subset=["Canonical_SMILES"]).copy()


# ==========================================
# 5. Αυστηρή μοναδικότητα με βάση canonical SMILES
# ==========================================

df_final = df_generated.drop_duplicates(
    subset=["Canonical_SMILES"]
).reset_index(drop=True)

canonical_unique_molecules = len(df_final)


# ==========================================
# 6. Υπολογισμός Validity, Uniqueness, Novelty και MAE
# ==========================================

# Validity:
# Πόσα από τα 150000 generation attempts έδωσαν έγκυρο μόριο.
validity_rate = (
    VALID_MOLECULES_FROM_GENERATION / TOTAL_ATTEMPTS
) * 100

# Uniqueness με βάση το αρχικό SMILES-based generated file.
uniqueness_rate_smiles = (
    unique_smiles_molecules / VALID_MOLECULES_FROM_GENERATION
) * 100

# Αυστηρότερη uniqueness μετά από canonicalization.
uniqueness_rate_canonical = (
    canonical_unique_molecules / VALID_MOLECULES_FROM_GENERATION
) * 100

# Novelty:
# Πόσα από τα unique generated molecules δεν υπάρχουν στο training set.
train_smiles_set = set(train_df["Clean_SMILES"].astype(str))

novel_mask = ~df_final["Canonical_SMILES"].isin(train_smiles_set)
novel_molecules = int(novel_mask.sum())

novelty_rate = (
    novel_molecules / canonical_unique_molecules
) * 100

# logP MAE:
# Μέση απόλυτη απόκλιση μεταξύ target logP και πραγματικού RDKit logP.
final_mae = (
    df_final["Actual_logP"] - df_final["Target_logP"]
).abs().mean()


# ==========================================
# 7. Εμφάνιση αποτελεσμάτων
# ==========================================

print("\nΤΕΛΙΚΗ ΣΤΑΤΙΣΤΙΚΗ ΑΝΑΛΥΣΗ")
print("-" * 50)
print(f"Συνολικές προσπάθειες παραγωγής: {TOTAL_ATTEMPTS}")
print(f"Έγκυρα μόρια: {VALID_MOLECULES_FROM_GENERATION}")
print(f"Μοναδικά μόρια βάσει SMILES: {unique_smiles_molecules}")
print(f"Μοναδικά μόρια βάσει canonical SMILES: {canonical_unique_molecules}")
print(f"Novel μόρια: {novel_molecules}")
print("-" * 50)
print(f"Ποσοστό Validity: {validity_rate:.2f}%")
print(f"Ποσοστό Uniqueness βάσει SMILES: {uniqueness_rate_smiles:.2f}%")
print(f"Ποσοστό Uniqueness βάσει canonical SMILES: {uniqueness_rate_canonical:.2f}%")
print(f"Ποσοστό Novelty: {novelty_rate:.2f}%")
print(f"Μέσο Απόλυτο Σφάλμα logP (MAE): {final_mae:.4f}")


# ==========================================
# 8. Αποθήκευση τελικής καθαρής βιβλιοθήκης
# ==========================================

output_final_file = "CLEAN_GENERATED_LIBRARY_FINAL_clean_split.csv"

df_final.to_csv(output_final_file, index=False)

print("-" * 50)
print(f"Η καθαρή τελική βιβλιοθήκη αποθηκεύτηκε ως: {output_final_file}")

## 13. Φυσικοχημική αξιολόγηση των παραγόμενων μορίων


In [ ]:
# ==========================================
# Υπολογισμός φυσικοχημικών ιδιοτήτων
# παραγόμενων και training μορίων
# ==========================================

import os
import sys
import numpy as np
import pandas as pd

from tqdm.notebook import tqdm

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, QED
from rdkit import RDConfig
from sklearn.model_selection import train_test_split

# ==========================================
# 1. Ενεργοποίηση progress_apply
# ==========================================

tqdm.pandas()


# ==========================================
# 2. Φόρτωση του Synthetic Accessibility scorer
# ==========================================

# Το SA Score δεν βρίσκεται πάντα στα βασικά imports της RDKit.
# Συνήθως παρέχεται στον φάκελο Contrib/SA_Score.

sa_score_available = False
sascorer = None

try:
    from rdkit.Contrib.SA_Score import sascorer
    sa_score_available = True

except ImportError:
    try:
        sa_score_path = os.path.join(
            RDConfig.RDContribDir,
            "SA_Score"
        )

        if sa_score_path not in sys.path:
            sys.path.append(sa_score_path)

        import sascorer
        sa_score_available = True

    except Exception:
        sa_score_available = False


if sa_score_available:
    print("Το Synthetic Accessibility Score είναι διαθέσιμο.")
else:
    print(
        "ΠΡΟΕΙΔΟΠΟΙΗΣΗ: Δεν βρέθηκε το SA_Score module.\n"
        "Οι υπόλοιπες ιδιότητες θα υπολογιστούν κανονικά,\n"
        "αλλά η στήλη SAS θα περιέχει NaN."
    )


# ==========================================
# 3. Ορισμός αρχείων
# ==========================================

generated_file = "CLEAN_GENERATED_LIBRARY_FINAL_clean_split.csv"
processed_file = "processed_data.csv"

generated_output_file = (
    "GENERATED_LIBRARY_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"
)

training_output_file = (
    "TRAINING_SAMPLE_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"
)


if not os.path.exists(generated_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το αρχείο παραγόμενων μορίων: {generated_file}"
    )

if not os.path.exists(processed_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το αρχείο εκπαίδευσης: {processed_file}"
    )


# ==========================================
# 4. Φόρτωση παραγόμενων μορίων
# ==========================================

generated_df = pd.read_csv(generated_file)

if "Canonical_SMILES" in generated_df.columns:
    generated_df["Evaluation_SMILES"] = (
        generated_df["Canonical_SMILES"]
    )
else:
    generated_df["Evaluation_SMILES"] = (
        generated_df["SMILES"]
    )

generated_df = generated_df.dropna(
    subset=["Evaluation_SMILES"]
).copy()

print(
    f"Παραγόμενα canonical unique μόρια: "
    f"{len(generated_df):,}"
)


# ==========================================
# 5. Αναδημιουργία του ίδιου training set
# ==========================================

df_all = pd.read_csv(processed_file)

df_all = df_all.dropna(
    subset=["Clean_SMILES", "SELFIES", "logP"]
).copy()

train_df, temp_df = train_test_split(
    df_all,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print(
    f"Συνολικά μόρια training set: "
    f"{len(train_df):,}"
)


# ==========================================
# 6. Επιλογή αντιπροσωπευτικού training sample
# ==========================================

# Το training set έχει πάνω από 1,2 εκατομμύρια μόρια.
# Για σύγκριση κατανομών δεν χρειάζεται να υπολογίσουμε descriptors
# και στα 1,2 εκατομμύρια. Χρησιμοποιούμε μεγάλο, σταθερό δείγμα.

TRAIN_SAMPLE_SIZE = min(
    150000,
    len(train_df)
)

training_sample_df = train_df.sample(
    n=TRAIN_SAMPLE_SIZE,
    random_state=42
).copy()

training_sample_df["Evaluation_SMILES"] = (
    training_sample_df["Clean_SMILES"]
)

print(
    f"Training μόρια που θα χρησιμοποιηθούν στις κατανομές: "
    f"{len(training_sample_df):,}"
)


# ==========================================
# 7. Συνάρτηση υπολογισμού ιδιοτήτων
# ==========================================

def calculate_physicochemical_properties(smiles):
    """
    Υπολογίζει τις φυσικοχημικές ιδιότητες ενός μορίου.

    Επιστρέφει:
    - RDKit logP
    - Μοριακό βάρος
    - H-bond donors
    - H-bond acceptors
    - Περιστρεφόμενους δεσμούς
    - TPSA
    - QED
    - Synthetic Accessibility Score
    - Αριθμό βαρέων ατόμων
    - Μήκος canonical SMILES
    - Canonical SMILES
    """

    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return pd.Series({
                "Canonical_SMILES_Check": None,
                "LogP_RDKit": np.nan,
                "Molecular_Weight": np.nan,
                "HBD": np.nan,
                "HBA": np.nan,
                "Rotatable_Bonds": np.nan,
                "TPSA": np.nan,
                "QED": np.nan,
                "SAS": np.nan,
                "Heavy_Atoms": np.nan,
                "SMILES_Length": np.nan
            })

        canonical_smiles = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

        if sa_score_available:
            sas_value = float(
                sascorer.calculateScore(mol)
            )
        else:
            sas_value = np.nan

        return pd.Series({
            "Canonical_SMILES_Check": canonical_smiles,
            "LogP_RDKit": Descriptors.MolLogP(mol),
            "Molecular_Weight": Descriptors.MolWt(mol),
            "HBD": Lipinski.NumHDonors(mol),
            "HBA": Lipinski.NumHAcceptors(mol),
            "Rotatable_Bonds": Lipinski.NumRotatableBonds(mol),
            "TPSA": Descriptors.TPSA(mol),
            "QED": QED.qed(mol),
            "SAS": sas_value,
            "Heavy_Atoms": mol.GetNumHeavyAtoms(),
            "SMILES_Length": len(canonical_smiles)
        })

    except Exception:
        return pd.Series({
            "Canonical_SMILES_Check": None,
            "LogP_RDKit": np.nan,
            "Molecular_Weight": np.nan,
            "HBD": np.nan,
            "HBA": np.nan,
            "Rotatable_Bonds": np.nan,
            "TPSA": np.nan,
            "QED": np.nan,
            "SAS": np.nan,
            "Heavy_Atoms": np.nan,
            "SMILES_Length": np.nan
        })


# ==========================================
# 8. Υπολογισμός ιδιοτήτων generated molecules
# ==========================================

print(
    "\nΥπολογισμός ιδιοτήτων για τα παραγόμενα μόρια..."
)

generated_properties = (
    generated_df["Evaluation_SMILES"]
    .progress_apply(
        calculate_physicochemical_properties
    )
)

generated_profile_df = pd.concat(
    [
        generated_df.reset_index(drop=True),
        generated_properties.reset_index(drop=True)
    ],
    axis=1
)

generated_profile_df = generated_profile_df.dropna(
    subset=["Canonical_SMILES_Check"]
).copy()

generated_profile_df["Dataset"] = "Generated"


# ==========================================
# 9. Υπολογισμός ιδιοτήτων training sample
# ==========================================

print(
    "\nΥπολογισμός ιδιοτήτων για το training sample..."
)

training_properties = (
    training_sample_df["Evaluation_SMILES"]
    .progress_apply(
        calculate_physicochemical_properties
    )
)

training_profile_df = pd.concat(
    [
        training_sample_df.reset_index(drop=True),
        training_properties.reset_index(drop=True)
    ],
    axis=1
)

training_profile_df = training_profile_df.dropna(
    subset=["Canonical_SMILES_Check"]
).copy()

training_profile_df["Dataset"] = "Training"


# ==========================================
# 10. Έλεγχος συνέπειας του Actual logP
# ==========================================

if "Actual_logP" in generated_profile_df.columns:
    generated_profile_df["Stored_vs_RDKit_logP_Difference"] = (
        generated_profile_df["Actual_logP"] -
        generated_profile_df["LogP_RDKit"]
    ).abs()

    mean_logp_difference = (
        generated_profile_df[
            "Stored_vs_RDKit_logP_Difference"
        ].mean()
    )

    max_logp_difference = (
        generated_profile_df[
            "Stored_vs_RDKit_logP_Difference"
        ].max()
    )

    print("\nΈλεγχος αποθηκευμένου Actual_logP:")
    print(
        f"Μέση διαφορά από νέο RDKit logP: "
        f"{mean_logp_difference:.6f}"
    )
    print(
        f"Μέγιστη διαφορά από νέο RDKit logP: "
        f"{max_logp_difference:.6f}"
    )


# ==========================================
# 11. Αποθήκευση εμπλουτισμένων datasets
# ==========================================

generated_profile_df.to_csv(
    generated_output_file,
    index=False
)

training_profile_df.to_csv(
    training_output_file,
    index=False
)


# ==========================================
# 12. Συνοπτικός πίνακας ιδιοτήτων
# ==========================================

properties_to_summarize = [
    "LogP_RDKit",
    "Molecular_Weight",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "TPSA",
    "QED",
    "SAS",
    "Heavy_Atoms",
    "SMILES_Length"
]

combined_profile_df = pd.concat(
    [
        generated_profile_df[
            properties_to_summarize + ["Dataset"]
        ],
        training_profile_df[
            properties_to_summarize + ["Dataset"]
        ]
    ],
    ignore_index=True
)

summary_table = (
    combined_profile_df
    .groupby("Dataset")[properties_to_summarize]
    .agg(["mean", "std", "median", "min", "max"])
    .round(4)
)

print("\n" + "=" * 65)
print("ΣΥΝΟΠΤΙΚΗ ΣΤΑΤΙΣΤΙΚΗ ΦΥΣΙΚΟΧΗΜΙΚΩΝ ΙΔΙΟΤΗΤΩΝ")
print("=" * 65)

display(summary_table)


# ==========================================
# 13. Τελική αναφορά
# ==========================================

print("\n" + "-" * 65)
print(
    f"Generated molecules με υπολογισμένες ιδιότητες: "
    f"{len(generated_profile_df):,}"
)

print(
    f"Training molecules με υπολογισμένες ιδιότητες: "
    f"{len(training_profile_df):,}"
)

print("-" * 65)
print(
    f"Αποθηκεύτηκε: {generated_output_file}"
)

print(
    f"Αποθηκεύτηκε: {training_output_file}"
)

if not sa_score_available:
    print(
        "\nΣημείωση: Το SAS δεν υπολογίστηκε επειδή δεν ήταν "
        "διαθέσιμο το αντίστοιχο RDKit module."
    )

In [ ]:
# ==========================================
# Έλεγχος αποκλίσεων μεταξύ αποθηκευμένου
# Actual_logP και νέου RDKit logP
# ==========================================

import os
import numpy as np
import pandas as pd

# ==========================================
# 1. Φόρτωση εμπλουτισμένου generated dataset
# ==========================================

input_file = "GENERATED_LIBRARY_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το αρχείο: {input_file}"
    )

logp_check_df = pd.read_csv(input_file)

required_columns = [
    "SMILES",
    "Actual_logP",
    "LogP_RDKit",
    "Stored_vs_RDKit_logP_Difference"
]

missing_columns = [
    column
    for column in required_columns
    if column not in logp_check_df.columns
]

if missing_columns:
    raise ValueError(
        f"Λείπουν απαραίτητες στήλες: {missing_columns}"
    )

print(f"Συνολικά generated μόρια: {len(logp_check_df):,}")


# ==========================================
# 2. Μετατροπή σε αριθμητικές τιμές
# ==========================================

logp_check_df["Actual_logP"] = pd.to_numeric(
    logp_check_df["Actual_logP"],
    errors="coerce"
)

logp_check_df["LogP_RDKit"] = pd.to_numeric(
    logp_check_df["LogP_RDKit"],
    errors="coerce"
)

logp_check_df["Stored_vs_RDKit_logP_Difference"] = pd.to_numeric(
    logp_check_df["Stored_vs_RDKit_logP_Difference"],
    errors="coerce"
)

logp_check_df = logp_check_df.dropna(
    subset=[
        "Actual_logP",
        "LogP_RDKit",
        "Stored_vs_RDKit_logP_Difference"
    ]
).copy()


# ==========================================
# 3. Καταμέτρηση αποκλίσεων ανά όριο
# ==========================================

tolerances = [
    0.001,
    0.005,
    0.010,
    0.050,
    0.100,
    0.500
]

print("\nΑΠΟΚΛΙΣΕΙΣ ΑΝΑ ΟΡΙΟ")
print("-" * 55)

for tolerance in tolerances:
    count = int(
        (
            logp_check_df[
                "Stored_vs_RDKit_logP_Difference"
            ] > tolerance
        ).sum()
    )

    percentage = (
        count / len(logp_check_df)
    ) * 100

    print(
        f"Διαφορά > {tolerance:.3f}: "
        f"{count:,} μόρια "
        f"({percentage:.6f}%)"
    )


# ==========================================
# 4. Εμφάνιση των μεγαλύτερων αποκλίσεων
# ==========================================

largest_differences_df = (
    logp_check_df
    .sort_values(
        by="Stored_vs_RDKit_logP_Difference",
        ascending=False
    )
    .head(20)
    .copy()
)

columns_to_display = [
    "SMILES"
]

if "Canonical_SMILES" in largest_differences_df.columns:
    columns_to_display.append("Canonical_SMILES")

if "Canonical_SMILES_Check" in largest_differences_df.columns:
    columns_to_display.append("Canonical_SMILES_Check")

columns_to_display.extend([
    "Target_logP",
    "Actual_logP",
    "LogP_RDKit",
    "Stored_vs_RDKit_logP_Difference"
])

print("\n20 ΜΕΓΑΛΥΤΕΡΕΣ ΑΠΟΚΛΙΣΕΙΣ")
print("-" * 55)

display(
    largest_differences_df[
        columns_to_display
    ]
)


# ==========================================
# 5. Περιγραφικά στατιστικά της απόκλισης
# ==========================================

difference_series = (
    logp_check_df[
        "Stored_vs_RDKit_logP_Difference"
    ]
)

print("\nΠΕΡΙΓΡΑΦΙΚΑ ΣΤΑΤΙΣΤΙΚΑ ΑΠΟΚΛΙΣΗΣ")
print("-" * 55)
print(
    difference_series.describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99,
            0.999
        ]
    )
)


# ==========================================
# 6. Έλεγχος αν η απόκλιση επηρεάζει το MAE
# ==========================================

mae_stored = (
    logp_check_df["Actual_logP"] -
    logp_check_df["Target_logP"]
).abs().mean()

mae_recalculated = (
    logp_check_df["LogP_RDKit"] -
    logp_check_df["Target_logP"]
).abs().mean()

mae_difference = abs(
    mae_stored - mae_recalculated
)

print("\nΕΠΙΔΡΑΣΗ ΣΤΟ MAE")
print("-" * 55)
print(
    f"MAE με αποθηκευμένο Actual_logP: "
    f"{mae_stored:.6f}"
)
print(
    f"MAE με νέο RDKit logP: "
    f"{mae_recalculated:.6f}"
)
print(
    f"Απόλυτη διαφορά μεταξύ των δύο MAE: "
    f"{mae_difference:.8f}"
)


# ==========================================
# 7. Αποθήκευση προβληματικών γραμμών
# ==========================================

problematic_rows_df = logp_check_df[
    logp_check_df[
        "Stored_vs_RDKit_logP_Difference"
    ] > 0.01
].copy()

problematic_output_file = (
    "LOGP_DISCREPANCIES_ABOVE_0_01.csv"
)

problematic_rows_df.to_csv(
    problematic_output_file,
    index=False
)

print("\n" + "-" * 55)
print(
    f"Γραμμές με απόκλιση > 0.01: "
    f"{len(problematic_rows_df):,}"
)
print(
    f"Αποθηκεύτηκαν ως: "
    f"{problematic_output_file}"
)


# ==========================================
# 8. Τελικό συμπέρασμα
# ==========================================

print("\nΤΕΛΙΚΟΣ ΕΛΕΓΧΟΣ")
print("=" * 55)

if mae_difference < 0.001:
    print(
        "Οι αποκλίσεις δεν επηρεάζουν ουσιαστικά "
        "το συνολικό MAE."
    )
else:
    print(
        "Οι αποκλίσεις επηρεάζουν το συνολικό MAE "
        "και χρειάζονται διόρθωση πριν την τελική αξιολόγηση."
    )

In [ ]:
# ============================================================
# Φυσικοχημικές κατανομές Training–Generated
# Προσαρμογή της λογικής της Figure 5 του Grechishnikova (2021)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. Ορισμός αρχείων εισόδου και εξόδου
# ============================================================

generated_file = (
    "GENERATED_LIBRARY_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"
)

training_file = (
    "TRAINING_SAMPLE_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"
)

figure_output_file = (
    "PHYSICOCHEMICAL_DISTRIBUTIONS_3x3.png"
)

summary_output_file = (
    "PHYSICOCHEMICAL_SUMMARY_MEAN_MEDIAN.csv"
)


# ============================================================
# 2. Έλεγχος ότι υπάρχουν τα απαιτούμενα αρχεία
# ============================================================

if not os.path.exists(generated_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το generated αρχείο: {generated_file}"
    )

if not os.path.exists(training_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το training αρχείο: {training_file}"
    )


# ============================================================
# 3. Φόρτωση των δύο datasets
# ============================================================

generated_df = pd.read_csv(generated_file)
training_df = pd.read_csv(training_file)

print("=" * 75)
print("ΦΟΡΤΩΣΗ ΑΡΧΕΙΩΝ")
print("=" * 75)

print(
    f"Generated dataset: "
    f"{generated_df.shape[0]:,} γραμμές × "
    f"{generated_df.shape[1]} στήλες"
)

print(
    f"Training dataset:  "
    f"{training_df.shape[0]:,} γραμμές × "
    f"{training_df.shape[1]} στήλες"
)


# ============================================================
# 4. Ορισμός των υποχρεωτικών στηλών
# ============================================================

# Στο generated dataset ελέγχουμε και το Actual_logP,
# παρότι στο γράφημα χρησιμοποιούμε το επανυπολογισμένο
# LogP_RDKit, το οποίο διατηρεί μεγαλύτερη αριθμητική ακρίβεια.

required_generated_columns = [
    "Target_logP",
    "Actual_logP",
    "LogP_RDKit",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "QED",
    "SAS",
    "TPSA",
    "Molecular_Weight",
    "SMILES_Length"
]

required_training_columns = [
    "LogP_RDKit",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "QED",
    "SAS",
    "TPSA",
    "Molecular_Weight",
    "SMILES_Length"
]


# ============================================================
# 5. Έλεγχος ύπαρξης στηλών
# ============================================================

missing_generated_columns = [
    column
    for column in required_generated_columns
    if column not in generated_df.columns
]

missing_training_columns = [
    column
    for column in required_training_columns
    if column not in training_df.columns
]

if missing_generated_columns:
    raise ValueError(
        "Λείπουν οι παρακάτω στήλες από το generated dataset:\n"
        f"{missing_generated_columns}\n\n"
        f"Διαθέσιμες στήλες:\n{generated_df.columns.tolist()}"
    )

if missing_training_columns:
    raise ValueError(
        "Λείπουν οι παρακάτω στήλες από το training dataset:\n"
        f"{missing_training_columns}\n\n"
        f"Διαθέσιμες στήλες:\n{training_df.columns.tolist()}"
    )

print("\nΟ έλεγχος στηλών ολοκληρώθηκε επιτυχώς.")
print("Όλες οι απαιτούμενες στήλες είναι διαθέσιμες.")


# ============================================================
# 6. Μετατροπή των ιδιοτήτων σε αριθμητική μορφή
# ============================================================

# Μετατρέπουμε όλες τις απαιτούμενες στήλες σε αριθμητικές.
# Τυχόν μη αριθμητικές τιμές μετατρέπονται σε NaN και
# αφαιρούνται μόνο από την αντίστοιχη ιδιότητα κατά το plotting.

for column in required_generated_columns:
    generated_df[column] = pd.to_numeric(
        generated_df[column],
        errors="coerce"
    )

for column in required_training_columns:
    training_df[column] = pd.to_numeric(
        training_df[column],
        errors="coerce"
    )


# ============================================================
# 7. Βοηθητική συνάρτηση καθαρισμού αριθμητικών τιμών
# ============================================================

def get_clean_numeric_values(dataframe, column):
    """
    Επιστρέφει τις έγκυρες και πεπερασμένες αριθμητικές τιμές
    μιας συγκεκριμένης στήλης.

    Δεν εφαρμόζει:
    - clipping,
    - απομάκρυνση outliers,
    - αλλαγή της κατανομής.
    """

    values = pd.to_numeric(
        dataframe[column],
        errors="coerce"
    )

    values = values.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return values.dropna()


# ============================================================
# 8. Βοηθητική συνάρτηση δημιουργίας κοινών bins
# ============================================================

CONTINUOUS_BIN_COUNT = 50


def create_common_bins(
    value_groups,
    number_of_bins=50,
    discrete=False
):
    """
    Δημιουργεί ακριβώς τα ίδια histogram bins για όλες τις
    ομάδες που συγκρίνονται στο ίδιο panel.

    Για διακριτές ακέραιες ιδιότητες, όπως HBD και HBA,
    δημιουργούνται bins κεντραρισμένα στους ακέραιους.

    Για συνεχείς ιδιότητες δημιουργούνται ισομεγέθη bins
    από το συνολικό ελάχιστο έως το συνολικό μέγιστο όλων
    των συγκρινόμενων ομάδων.
    """

    available_arrays = [
        values.to_numpy(dtype=float)
        for values in value_groups
        if len(values) > 0
    ]

    if not available_arrays:
        raise ValueError(
            "Δεν υπάρχουν διαθέσιμες αριθμητικές τιμές "
            "για τη δημιουργία histogram bins."
        )

    combined_values = np.concatenate(
        available_arrays
    )

    minimum_value = float(
        np.min(combined_values)
    )

    maximum_value = float(
        np.max(combined_values)
    )

    # Για ακέραιες ιδιότητες δημιουργούμε bins
    # με κέντρο κάθε ακέραιη τιμή.
    if discrete:
        return np.arange(
            np.floor(minimum_value) - 0.5,
            np.ceil(maximum_value) + 1.5,
            1.0
        )

    # Προστασία για την ειδική περίπτωση όπου όλες
    # οι τιμές μιας ιδιότητας είναι ίδιες.
    if np.isclose(
        minimum_value,
        maximum_value
    ):
        margin = max(
            abs(minimum_value) * 0.05,
            0.05
        )

        minimum_value -= margin
        maximum_value += margin

    return np.linspace(
        minimum_value,
        maximum_value,
        number_of_bins + 1
    )


# ============================================================
# 9. Έλεγχος διαθέσιμων αριθμητικών τιμών
# ============================================================

print("\n" + "=" * 75)
print("ΕΛΕΓΧΟΣ ΑΡΙΘΜΗΤΙΚΩΝ ΤΙΜΩΝ")
print("=" * 75)

columns_for_report = [
    "LogP_RDKit",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "QED",
    "SAS",
    "TPSA",
    "Molecular_Weight",
    "SMILES_Length"
]

for column in columns_for_report:
    training_valid_count = len(
        get_clean_numeric_values(
            training_df,
            column
        )
    )

    generated_valid_count = len(
        get_clean_numeric_values(
            generated_df,
            column
        )
    )

    print(
        f"{column:<20} | "
        f"Training: {training_valid_count:>8,} | "
        f"Generated: {generated_valid_count:>8,}"
    )

    if training_valid_count == 0:
        raise ValueError(
            f"Η στήλη '{column}' του training dataset "
            "δεν περιέχει καμία έγκυρη αριθμητική τιμή."
        )

    if generated_valid_count == 0:
        raise ValueError(
            f"Η στήλη '{column}' του generated dataset "
            "δεν περιέχει καμία έγκυρη αριθμητική τιμή."
        )


target_logp_count = len(
    get_clean_numeric_values(
        generated_df,
        "Target_logP"
    )
)

actual_logp_count = len(
    get_clean_numeric_values(
        generated_df,
        "Actual_logP"
    )
)

print(
    f"{'Target_logP':<20} | "
    f"Generated: {target_logp_count:>8,}"
)

print(
    f"{'Actual_logP':<20} | "
    f"Generated: {actual_logp_count:>8,}"
)

if target_logp_count == 0:
    raise ValueError(
        "Η στήλη Target_logP δεν περιέχει "
        "καμία έγκυρη αριθμητική τιμή."
    )


# ============================================================
# 10. Προετοιμασία συνοπτικού πίνακα mean και median
# ============================================================

summary_rows = []


def add_summary_row(
    property_name,
    dataset_name,
    values,
    order
):
    """
    Προσθέτει μία γραμμή στον συνοπτικό πίνακα.
    """

    summary_rows.append({
        "Order": order,
        "Property": property_name,
        "Dataset": dataset_name,
        "N": int(len(values)),
        "Mean": float(values.mean()),
        "Median": float(values.median())
    })


# ============================================================
# 11. Δημιουργία του σχήματος 3 × 3
# ============================================================

fig, axes = plt.subplots(
    nrows=3,
    ncols=3,
    figsize=(18, 14),
    constrained_layout=True
)

axes = axes.ravel()


# ============================================================
# PANEL 1: Training vs Target vs Actual Generated logP
# ============================================================

# Training logP:
# Επανυπολογισμένη τιμή RDKit για το training sample.
training_logp = get_clean_numeric_values(
    training_df,
    "LogP_RDKit"
)

# Target logP:
# Targets που αντιστοιχούν στα canonical unique generated
# molecules του τελικού generated dataset.
target_logp = get_clean_numeric_values(
    generated_df,
    "Target_logP"
)

# Actual generated logP:
# Χρησιμοποιούμε το πλήρους ακρίβειας επανυπολογισμένο
# RDKit logP και όχι τη στρογγυλοποιημένη αποθηκευμένη τιμή.
generated_actual_logp = get_clean_numeric_values(
    generated_df,
    "LogP_RDKit"
)

logp_bins = create_common_bins(
    [
        training_logp,
        target_logp,
        generated_actual_logp
    ],
    number_of_bins=CONTINUOUS_BIN_COUNT,
    discrete=False
)

axes[0].hist(
    training_logp,
    bins=logp_bins,
    density=True,
    alpha=0.45,
    label=(
        f"Training logP "
        f"(n={len(training_logp):,})"
    )
)

axes[0].hist(
    target_logp,
    bins=logp_bins,
    density=True,
    alpha=0.45,
    label=(
        f"Target logP "
        f"(n={len(target_logp):,})"
    )
)

axes[0].hist(
    generated_actual_logp,
    bins=logp_bins,
    density=True,
    alpha=0.45,
    label=(
        f"Actual Generated logP "
        f"(n={len(generated_actual_logp):,})"
    )
)

axes[0].set_title(
    "logP Distribution"
)

axes[0].set_xlabel(
    "logP"
)

axes[0].set_ylabel(
    "Density"
)

axes[0].legend(
    fontsize=8
)

axes[0].grid(
    alpha=0.25
)

add_summary_row(
    property_name="logP",
    dataset_name="Training",
    values=training_logp,
    order=1
)

add_summary_row(
    property_name="logP",
    dataset_name="Target",
    values=target_logp,
    order=2
)

add_summary_row(
    property_name="logP",
    dataset_name="Generated Actual RDKit",
    values=generated_actual_logp,
    order=3
)


# ============================================================
# 12. Ορισμός των υπόλοιπων οκτώ panels
# ============================================================

# Η Boolean τιμή discrete δηλώνει αν η ιδιότητα
# παίρνει ακέραιες τιμές και χρειάζεται integer-centered bins.

plot_specifications = [
    {
        "column": "HBD",
        "title": "Hydrogen-Bond Donors",
        "xlabel": "Number of H-bond donors",
        "property_name": "HBD",
        "discrete": True
    },
    {
        "column": "HBA",
        "title": "Hydrogen-Bond Acceptors",
        "xlabel": "Number of H-bond acceptors",
        "property_name": "HBA",
        "discrete": True
    },
    {
        "column": "Rotatable_Bonds",
        "title": "Rotatable Bonds",
        "xlabel": "Number of rotatable bonds",
        "property_name": "Rotatable Bonds",
        "discrete": True
    },
    {
        "column": "QED",
        "title": "Quantitative Estimate of Drug-likeness",
        "xlabel": "QED",
        "property_name": "QED",
        "discrete": False
    },
    {
        "column": "SAS",
        "title": "Synthetic Accessibility Score",
        "xlabel": "SAS",
        "property_name": "SAS",
        "discrete": False
    },
    {
        "column": "TPSA",
        "title": "Topological Polar Surface Area",
        "xlabel": "TPSA (Å²)",
        "property_name": "TPSA",
        "discrete": False
    },
    {
        "column": "Molecular_Weight",
        "title": "Molecular Weight",
        "xlabel": "Molecular weight (Da)",
        "property_name": "Molecular Weight",
        "discrete": False
    },
    {
        "column": "SMILES_Length",
        "title": "Molecular Length",
        "xlabel": "Canonical SMILES length (characters)",
        "property_name": "Canonical SMILES Length",
        "discrete": True
    }
]


# ============================================================
# 13. Δημιουργία των οκτώ Training–Generated histograms
# ============================================================

summary_order = 4

for axis, plot_info in zip(
    axes[1:],
    plot_specifications
):
    column = plot_info["column"]

    training_values = get_clean_numeric_values(
        training_df,
        column
    )

    generated_values = get_clean_numeric_values(
        generated_df,
        column
    )

    common_bins = create_common_bins(
        [
            training_values,
            generated_values
        ],
        number_of_bins=CONTINUOUS_BIN_COUNT,
        discrete=plot_info["discrete"]
    )

    axis.hist(
        training_values,
        bins=common_bins,
        density=True,
        alpha=0.50,
        label=(
            f"Training "
            f"(n={len(training_values):,})"
        )
    )

    axis.hist(
        generated_values,
        bins=common_bins,
        density=True,
        alpha=0.50,
        label=(
            f"Generated "
            f"(n={len(generated_values):,})"
        )
    )

    axis.set_title(
        plot_info["title"]
    )

    axis.set_xlabel(
        plot_info["xlabel"]
    )

    axis.set_ylabel(
        "Density"
    )

    axis.legend(
        fontsize=8
    )

    axis.grid(
        alpha=0.25
    )

    add_summary_row(
        property_name=plot_info["property_name"],
        dataset_name="Training",
        values=training_values,
        order=summary_order
    )

    summary_order += 1

    add_summary_row(
        property_name=plot_info["property_name"],
        dataset_name="Generated",
        values=generated_values,
        order=summary_order
    )

    summary_order += 1


# ============================================================
# 14. Τελική μορφοποίηση και αποθήκευση του PNG
# ============================================================

fig.suptitle(
    (
        "Physicochemical Property Distributions: "
        "Training, Targets and Generated Molecules"
    ),
    fontsize=16
)

fig.savefig(
    figure_output_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 15. Δημιουργία και αποθήκευση συνοπτικού CSV
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

summary_df = (
    summary_df
    .sort_values("Order")
    .drop(columns="Order")
    .reset_index(drop=True)
)

summary_df[
    ["Mean", "Median"]
] = summary_df[
    ["Mean", "Median"]
].round(6)

summary_df.to_csv(
    summary_output_file,
    index=False
)


# ============================================================
# 16. Εμφάνιση τελικών αποτελεσμάτων
# ============================================================

print("\n" + "=" * 75)
print("Η ΦΥΣΙΚΟΧΗΜΙΚΗ ΑΞΙΟΛΟΓΗΣΗ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("=" * 75)

print(
    f"Το σχήμα αποθηκεύτηκε ως:\n"
    f"{figure_output_file}"
)

print(
    f"\nΟ συνοπτικός πίνακας αποθηκεύτηκε ως:\n"
    f"{summary_output_file}"
)

print("\nΣυνοπτικός πίνακας mean και median:")

display(
    summary_df
)

In [ ]:
# ============================================================
# ΒΗΜΑ 3:
# Ανάλυση Drug-likeness και Lipinski Violations
#
# Μεθοδολογική βάση:
# Grechishnikova (2021), ενότητα
# "Physicochemical properties and metrics" και Table 5.
# ============================================================

import os
import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. Ορισμός αρχείων εισόδου
# ============================================================

generated_file = (
    "GENERATED_LIBRARY_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"
)

training_file = (
    "TRAINING_SAMPLE_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"
)


# ============================================================
# 2. Ορισμός αρχείων εξόδου
# ============================================================

annotated_generated_output = (
    "GENERATED_LIBRARY_WITH_DRUG_LIKENESS_FLAGS.csv"
)

compliance_summary_output = (
    "DRUG_LIKENESS_COMPLIANCE_SUMMARY.csv"
)

lipinski_distribution_output = (
    "LIPINSKI_VIOLATION_DISTRIBUTION.csv"
)

qed_sas_summary_output = (
    "QED_SAS_DRUG_LIKENESS_SUMMARY.csv"
)


# ============================================================
# 3. Έλεγχος ύπαρξης των αρχείων
# ============================================================

if not os.path.exists(generated_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το generated αρχείο: {generated_file}"
    )

if not os.path.exists(training_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το training αρχείο: {training_file}"
    )


# ============================================================
# 4. Φόρτωση των datasets
# ============================================================

generated_df = pd.read_csv(generated_file)
training_df = pd.read_csv(training_file)

print("=" * 78)
print("ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ ΓΙΑ DRUG-LIKENESS ANALYSIS")
print("=" * 78)

print(
    f"Generated molecules: "
    f"{len(generated_df):,}"
)

print(
    f"Training molecules:  "
    f"{len(training_df):,}"
)


# ============================================================
# 5. Απαιτούμενες στήλες
# ============================================================

required_columns = [
    "LogP_RDKit",
    "Molecular_Weight",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "TPSA",
    "QED",
    "SAS"
]


# ============================================================
# 6. Έλεγχος ότι υπάρχουν όλες οι απαιτούμενες στήλες
# ============================================================

missing_generated_columns = [
    column
    for column in required_columns
    if column not in generated_df.columns
]

missing_training_columns = [
    column
    for column in required_columns
    if column not in training_df.columns
]

if missing_generated_columns:
    raise ValueError(
        "Λείπουν οι ακόλουθες στήλες από το generated αρχείο:\n"
        f"{missing_generated_columns}"
    )

if missing_training_columns:
    raise ValueError(
        "Λείπουν οι ακόλουθες στήλες από το training αρχείο:\n"
        f"{missing_training_columns}"
    )

print("\nΌλες οι απαιτούμενες στήλες είναι διαθέσιμες.")


# ============================================================
# 7. Μετατροπή των descriptor στηλών σε αριθμητικές τιμές
# ============================================================

for column in required_columns:
    generated_df[column] = pd.to_numeric(
        generated_df[column],
        errors="coerce"
    )

    training_df[column] = pd.to_numeric(
        training_df[column],
        errors="coerce"
    )


# Αντικαθιστούμε τυχόν άπειρες τιμές με NaN.
generated_df[required_columns] = (
    generated_df[required_columns]
    .replace([np.inf, -np.inf], np.nan)
)

training_df[required_columns] = (
    training_df[required_columns]
    .replace([np.inf, -np.inf], np.nan)
)


# ============================================================
# 8. Αυστηρός έλεγχος για ελλιπείς αριθμητικές τιμές
# ============================================================

generated_missing_counts = (
    generated_df[required_columns]
    .isna()
    .sum()
)

training_missing_counts = (
    training_df[required_columns]
    .isna()
    .sum()
)

if generated_missing_counts.sum() > 0:
    raise ValueError(
        "Βρέθηκαν κενές ή μη αριθμητικές τιμές "
        "στο generated dataset:\n"
        f"{generated_missing_counts[generated_missing_counts > 0]}"
    )

if training_missing_counts.sum() > 0:
    raise ValueError(
        "Βρέθηκαν κενές ή μη αριθμητικές τιμές "
        "στο training dataset:\n"
        f"{training_missing_counts[training_missing_counts > 0]}"
    )

print(
    "Δεν βρέθηκαν κενές ή μη αριθμητικές τιμές "
    "στις απαιτούμενες ιδιότητες."
)


# ============================================================
# 9. Συνάρτηση προσθήκης drug-likeness flags
# ============================================================

def add_drug_likeness_flags(dataframe):
    """
    Προσθέτει flags συμμόρφωσης με τα κριτήρια που
    χρησιμοποιούνται στην ανάλυση drug-likeness.

    Αυστηρά Lipinski κριτήρια:
    - logP <= 5
    - Molecular Weight < 500
    - HBD <= 5
    - HBA <= 10

    Πρόσθετα κριτήρια του paper:
    - Rotatable Bonds <= 10
    - TPSA <= 140
    - SAS <= 6

    Το QED δεν μετατρέπεται σε pass/fail, επειδή το βασικό
    paper δεν ορίζει συγκεκριμένο QED threshold.
    """

    result_df = dataframe.copy()


    # --------------------------------------------------------
    # Ατομικά κριτήρια συμμόρφωσης
    # --------------------------------------------------------

    result_df["Pass_LogP_le_5"] = (
        result_df["LogP_RDKit"] <= 5
    )

    result_df["Pass_MW_lt_500"] = (
        result_df["Molecular_Weight"] < 500
    )

    result_df["Pass_HBD_le_5"] = (
        result_df["HBD"] <= 5
    )

    result_df["Pass_HBA_le_10"] = (
        result_df["HBA"] <= 10
    )

    result_df["Pass_Rotatable_Bonds_le_10"] = (
        result_df["Rotatable_Bonds"] <= 10
    )

    result_df["Pass_TPSA_le_140"] = (
        result_df["TPSA"] <= 140
    )

    result_df["Pass_SAS_le_6"] = (
        result_df["SAS"] <= 6
    )


    # --------------------------------------------------------
    # Αυστηρή καταμέτρηση Lipinski violations
    #
    # Χρησιμοποιούνται μόνο:
    # logP, MW, HBD και HBA.
    #
    # Τα Rotatable Bonds, TPSA και SAS δεν προσμετρώνται
    # ως Lipinski violations.
    # --------------------------------------------------------

    result_df["Lipinski_Violations"] = (
        (~result_df["Pass_LogP_le_5"]).astype(int)
        + (~result_df["Pass_MW_lt_500"]).astype(int)
        + (~result_df["Pass_HBD_le_5"]).astype(int)
        + (~result_df["Pass_HBA_le_10"]).astype(int)
    )


    # Αυστηρό flag μηδενικών Lipinski violations.
    # Θα μπορεί να χρησιμοποιηθεί αργότερα ως ένα από
    # τα προκαθορισμένα κριτήρια επιλογής υποψηφίων.
    result_df["Pass_Zero_Lipinski_Violations"] = (
        result_df["Lipinski_Violations"] == 0
    )


    # --------------------------------------------------------
    # Συνδυαστική συμμόρφωση με όλα τα όρια του paper
    #
    # Αυτό αποτελεί ρητή προσαρμογή της πτυχιακής.
    # Το paper παρουσιάζει κυρίως τα ποσοστά κάθε
    # επιμέρους κριτηρίου.
    # --------------------------------------------------------

    paper_threshold_columns = [
        "Pass_LogP_le_5",
        "Pass_MW_lt_500",
        "Pass_HBD_le_5",
        "Pass_HBA_le_10",
        "Pass_Rotatable_Bonds_le_10",
        "Pass_TPSA_le_140",
        "Pass_SAS_le_6"
    ]

    result_df["Pass_All_Paper_Aligned_Thresholds"] = (
        result_df[paper_threshold_columns]
        .all(axis=1)
    )

    return result_df


# ============================================================
# 10. Προσθήκη flags σε generated και training molecules
# ============================================================

generated_annotated_df = add_drug_likeness_flags(
    generated_df
)

training_annotated_df = add_drug_likeness_flags(
    training_df
)


# ============================================================
# 11. Ορισμός κριτηρίων για τον συνοπτικό πίνακα
# ============================================================

criteria_definitions = [
    {
        "Criterion": "logP <= 5",
        "Column": "Pass_LogP_le_5",
        "Methodological_Status": (
            "Paper-aligned individual threshold"
        )
    },
    {
        "Criterion": "Molecular Weight < 500 Da",
        "Column": "Pass_MW_lt_500",
        "Methodological_Status": (
            "Paper-aligned individual threshold"
        )
    },
    {
        "Criterion": "HBD <= 5",
        "Column": "Pass_HBD_le_5",
        "Methodological_Status": (
            "Paper-aligned individual threshold"
        )
    },
    {
        "Criterion": "HBA <= 10",
        "Column": "Pass_HBA_le_10",
        "Methodological_Status": (
            "Paper-aligned individual threshold"
        )
    },
    {
        "Criterion": "Rotatable Bonds <= 10",
        "Column": "Pass_Rotatable_Bonds_le_10",
        "Methodological_Status": (
            "Paper-aligned individual threshold"
        )
    },
    {
        "Criterion": "TPSA <= 140 A^2",
        "Column": "Pass_TPSA_le_140",
        "Methodological_Status": (
            "Paper-aligned individual threshold"
        )
    },
    {
        "Criterion": "SAS <= 6",
        "Column": "Pass_SAS_le_6",
        "Methodological_Status": (
            "Paper-aligned individual threshold"
        )
    },
    {
        "Criterion": "Zero strict Lipinski violations",
        "Column": "Pass_Zero_Lipinski_Violations",
        "Methodological_Status": (
            "Explicit thesis summary criterion"
        )
    },
    {
        "Criterion": "All seven paper-aligned thresholds",
        "Column": "Pass_All_Paper_Aligned_Thresholds",
        "Methodological_Status": (
            "Explicit thesis combined criterion"
        )
    }
]


# ============================================================
# 12. Δημιουργία συνοπτικού πίνακα συμμόρφωσης
# ============================================================

compliance_rows = []

datasets_for_analysis = [
    (
        "Training",
        training_annotated_df
    ),
    (
        "Generated",
        generated_annotated_df
    )
]

for dataset_name, dataframe in datasets_for_analysis:

    total_molecules = len(dataframe)

    for criterion_info in criteria_definitions:

        criterion_column = criterion_info["Column"]

        passed_molecules = int(
            dataframe[criterion_column].sum()
        )

        failed_molecules = (
            total_molecules - passed_molecules
        )

        pass_percentage = (
            passed_molecules / total_molecules
        ) * 100

        compliance_rows.append({
            "Dataset": dataset_name,
            "Criterion": criterion_info["Criterion"],
            "Methodological_Status": (
                criterion_info["Methodological_Status"]
            ),
            "Total_Molecules": total_molecules,
            "Passed_Molecules": passed_molecules,
            "Failed_Molecules": failed_molecules,
            "Pass_Percentage": pass_percentage
        })


compliance_summary_df = pd.DataFrame(
    compliance_rows
)

compliance_summary_df["Pass_Percentage"] = (
    compliance_summary_df["Pass_Percentage"]
    .round(4)
)


# ============================================================
# 13. Κατανομή αριθμού Lipinski violations
# ============================================================

lipinski_distribution_rows = []

for dataset_name, dataframe in datasets_for_analysis:

    total_molecules = len(dataframe)

    violation_counts = (
        dataframe["Lipinski_Violations"]
        .value_counts()
        .reindex(
            [0, 1, 2, 3, 4],
            fill_value=0
        )
        .sort_index()
    )

    for number_of_violations, count in violation_counts.items():

        percentage = (
            count / total_molecules
        ) * 100

        lipinski_distribution_rows.append({
            "Dataset": dataset_name,
            "Lipinski_Violations": int(
                number_of_violations
            ),
            "Molecule_Count": int(count),
            "Percentage": round(
                percentage,
                4
            )
        })


lipinski_distribution_df = pd.DataFrame(
    lipinski_distribution_rows
)


# ============================================================
# 14. Περιγραφικά στατιστικά QED και SAS
# ============================================================

qed_sas_summary_rows = []

for dataset_name, dataframe in datasets_for_analysis:

    for property_name in ["QED", "SAS"]:

        property_values = dataframe[property_name]

        qed_sas_summary_rows.append({
            "Dataset": dataset_name,
            "Property": property_name,
            "N": int(property_values.count()),
            "Mean": float(property_values.mean()),
            "Median": float(property_values.median()),
            "Standard_Deviation": float(
                property_values.std()
            ),
            "Q1_25_Percent": float(
                property_values.quantile(0.25)
            ),
            "Q3_75_Percent": float(
                property_values.quantile(0.75)
            ),
            "Minimum": float(property_values.min()),
            "Maximum": float(property_values.max())
        })


qed_sas_summary_df = pd.DataFrame(
    qed_sas_summary_rows
)

numeric_summary_columns = [
    "Mean",
    "Median",
    "Standard_Deviation",
    "Q1_25_Percent",
    "Q3_75_Percent",
    "Minimum",
    "Maximum"
]

qed_sas_summary_df[numeric_summary_columns] = (
    qed_sas_summary_df[numeric_summary_columns]
    .round(6)
)


# ============================================================
# 15. Αποθήκευση αποτελεσμάτων
# ============================================================

# Αποθηκεύουμε μόνο το generated dataset με όλα τα νέα flags,
# επειδή αυτό θα χρησιμοποιηθεί αργότερα για την επιλογή
# generated candidates.
generated_annotated_df.to_csv(
    annotated_generated_output,
    index=False
)

compliance_summary_df.to_csv(
    compliance_summary_output,
    index=False
)

lipinski_distribution_df.to_csv(
    lipinski_distribution_output,
    index=False
)

qed_sas_summary_df.to_csv(
    qed_sas_summary_output,
    index=False
)


# ============================================================
# 16. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 78)
print("ΠΟΣΟΣΤΑ ΣΥΜΜΟΡΦΩΣΗΣ DRUG-LIKENESS")
print("=" * 78)

display(
    compliance_summary_df
)


print("\n" + "=" * 78)
print("ΚΑΤΑΝΟΜΗ ΑΡΙΘΜΟΥ LIPINSKI VIOLATIONS")
print("=" * 78)

display(
    lipinski_distribution_df
)


print("\n" + "=" * 78)
print("ΠΕΡΙΓΡΑΦΙΚΑ ΣΤΑΤΙΣΤΙΚΑ QED ΚΑΙ SAS")
print("=" * 78)

display(
    qed_sas_summary_df
)


# ============================================================
# 17. Τελική αναφορά αρχείων
# ============================================================

print("\n" + "-" * 78)
print("Η ανάλυση drug-likeness ολοκληρώθηκε.")
print("-" * 78)

print(
    f"Generated dataset με flags:\n"
    f"{annotated_generated_output}"
)

print(
    f"\nΠίνακας ποσοστών συμμόρφωσης:\n"
    f"{compliance_summary_output}"
)

print(
    f"\nΚατανομή Lipinski violations:\n"
    f"{lipinski_distribution_output}"
)

print(
    f"\nΠεριγραφικά στατιστικά QED και SAS:\n"
    f"{qed_sas_summary_output}"
)

In [ ]:
# ============================================================
# ΒΗΜΑ 4:
# Δομική ομοιότητα με Nearest-Neighbor Tanimoto
#
# Μεθοδολογική βάση:
# Grechishnikova (2021), Figures 6 και 7.
#
# Fingerprint σύμφωνα με τον διαθέσιμο κώδικα των συγγραφέων:
# - Morgan fingerprint
# - radius = 2
# - 1024 bits
#
# Αναγκαία προσαρμογή:
# Χρησιμοποιείται αντιπροσωπευτικό και αναπαραγώγιμο
# sampling λόγω του πολύ μεγάλου μεγέθους των datasets.
# ============================================================

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem

from IPython.display import display


# ============================================================
# 1. Ρυθμίσεις RDKit και προειδοποιήσεων
# ============================================================

RDLogger.DisableLog("rdApp.*")

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)


# ============================================================
# 2. Αρχεία εισόδου
# ============================================================

generated_file = (
    "GENERATED_LIBRARY_WITH_DRUG_LIKENESS_FLAGS.csv"
)

training_file = (
    "TRAINING_SAMPLE_WITH_PHYSICOCHEMICAL_PROPERTIES.csv"
)


# ============================================================
# 3. Αρχεία εξόδου
# ============================================================

generated_similarity_output = (
    "GENERATED_TO_TRAINING_NEAREST_NEIGHBOR_TANIMOTO.csv"
)

training_similarity_output = (
    "TRAINING_TO_TRAINING_NEAREST_NEIGHBOR_TANIMOTO.csv"
)

summary_output = (
    "TANIMOTO_NEAREST_NEIGHBOR_SUMMARY.csv"
)

metadata_output = (
    "TANIMOTO_ANALYSIS_METADATA.csv"
)

figure_output = (
    "TANIMOTO_NEAREST_NEIGHBOR_DISTRIBUTIONS.png"
)


# ============================================================
# 4. Έλεγχος ύπαρξης αρχείων
# ============================================================

if not os.path.exists(generated_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το generated αρχείο: {generated_file}"
    )

if not os.path.exists(training_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το training αρχείο: {training_file}"
    )


# ============================================================
# 5. Φόρτωση δεδομένων
# ============================================================

generated_df = pd.read_csv(
    generated_file
)

training_df = pd.read_csv(
    training_file
)

print("=" * 80)
print("ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ ΓΙΑ TANIMOTO ANALYSIS")
print("=" * 80)

print(
    f"Generated molecules στο αρχείο: "
    f"{len(generated_df):,}"
)

print(
    f"Training molecules στο αρχείο:  "
    f"{len(training_df):,}"
)


# ============================================================
# 6. Εντοπισμός της κατάλληλης στήλης SMILES
# ============================================================

def find_smiles_column(
    dataframe,
    candidate_columns,
    dataset_name
):
    """
    Επιστρέφει την πρώτη διαθέσιμη στήλη SMILES από
    μια προκαθορισμένη λίστα υποψηφίων.
    """

    for column in candidate_columns:
        if column in dataframe.columns:
            print(
                f"Στήλη SMILES για {dataset_name}: "
                f"{column}"
            )

            return column

    raise ValueError(
        f"Δεν βρέθηκε κατάλληλη στήλη SMILES "
        f"στο dataset '{dataset_name}'.\n"
        f"Διαθέσιμες στήλες:\n"
        f"{dataframe.columns.tolist()}"
    )


generated_smiles_column = find_smiles_column(
    dataframe=generated_df,
    candidate_columns=[
        "Canonical_SMILES_Check",
        "Canonical_SMILES",
        "Evaluation_SMILES",
        "SMILES"
    ],
    dataset_name="Generated"
)

training_smiles_column = find_smiles_column(
    dataframe=training_df,
    candidate_columns=[
        "Canonical_SMILES_Check",
        "Clean_SMILES",
        "Evaluation_SMILES",
        "SMILES"
    ],
    dataset_name="Training"
)


# ============================================================
# 7. Προετοιμασία canonical unique δομών
# ============================================================

generated_structures_df = (
    generated_df[
        [generated_smiles_column]
    ]
    .dropna()
    .copy()
)

generated_structures_df.columns = [
    "Canonical_SMILES"
]

generated_structures_df[
    "Canonical_SMILES"
] = generated_structures_df[
    "Canonical_SMILES"
].astype(str)

generated_structures_df = (
    generated_structures_df
    .drop_duplicates(
        subset=["Canonical_SMILES"]
    )
    .reset_index(drop=True)
)


training_structures_df = (
    training_df[
        [training_smiles_column]
    ]
    .dropna()
    .copy()
)

training_structures_df.columns = [
    "Canonical_SMILES"
]

training_structures_df[
    "Canonical_SMILES"
] = training_structures_df[
    "Canonical_SMILES"
].astype(str)

training_structures_df = (
    training_structures_df
    .drop_duplicates(
        subset=["Canonical_SMILES"]
    )
    .reset_index(drop=True)
)


print("\n" + "-" * 80)

print(
    f"Canonical unique generated structures: "
    f"{len(generated_structures_df):,}"
)

print(
    f"Canonical unique training structures:  "
    f"{len(training_structures_df):,}"
)


# ============================================================
# 8. Προκαθορισμένες παράμετροι sampling
# ============================================================

# Χρησιμοποιούμε 1.000 query molecules για κάθε κατανομή.
#
# Το training reference set αποτελείται από όλα τα υπόλοιπα
# μόρια του ήδη αποθηκευμένου training sample.
#
# Το ίδιο training reference χρησιμοποιείται:
# - για Generated → Training
# - για Training → Training
#
# Με αυτόν τον τρόπο οι δύο κατανομές είναι συγκρίσιμες.

RANDOM_STATE = 42

GENERATED_QUERY_SIZE = 1000
TRAINING_QUERY_SIZE = 1000

MORGAN_RADIUS = 2
FINGERPRINT_BITS = 1024

LOW_SIMILARITY_THRESHOLD = 0.50
HIGH_SIMILARITY_THRESHOLD = 0.85


if len(generated_structures_df) < GENERATED_QUERY_SIZE:
    raise ValueError(
        "Δεν υπάρχουν αρκετά generated molecules "
        f"για δείγμα {GENERATED_QUERY_SIZE:,}."
    )

if len(training_structures_df) <= TRAINING_QUERY_SIZE:
    raise ValueError(
        "Δεν υπάρχουν αρκετά training molecules για τη "
        "δημιουργία ανεξάρτητων query και reference sets."
    )


# ============================================================
# 9. Δημιουργία Generated query sample
# ============================================================

generated_query_df = (
    generated_structures_df
    .sample(
        n=GENERATED_QUERY_SIZE,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)


# ============================================================
# 10. Δημιουργία διακριτών Training query/reference sets
# ============================================================

# Ανακατεύουμε αναπαραγώγιμα το training sample.
training_shuffled_df = (
    training_structures_df
    .sample(
        frac=1.0,
        random_state=RANDOM_STATE
    )
    .reset_index(drop=True)
)

# Τα πρώτα 1.000 training molecules χρησιμοποιούνται ως queries.
training_query_df = (
    training_shuffled_df
    .iloc[:TRAINING_QUERY_SIZE]
    .copy()
    .reset_index(drop=True)
)

# Όλα τα υπόλοιπα χρησιμοποιούνται ως κοινό reference set.
# Έτσι δεν υπάρχουν self-matches στο Training → Training.
training_reference_df = (
    training_shuffled_df
    .iloc[TRAINING_QUERY_SIZE:]
    .copy()
    .reset_index(drop=True)
)


print("\n" + "=" * 80)
print("SAMPLING ΓΙΑ ΤΗ ΔΟΜΙΚΗ ΑΝΑΛΥΣΗ")
print("=" * 80)

print(
    f"Generated query molecules: "
    f"{len(generated_query_df):,}"
)

print(
    f"Training query molecules:  "
    f"{len(training_query_df):,}"
)

print(
    f"Training reference molecules: "
    f"{len(training_reference_df):,}"
)


# ============================================================
# 11. Συνάρτηση δημιουργίας Morgan fingerprint
# ============================================================

def calculate_morgan_fingerprint(smiles):
    """
    Δημιουργεί Morgan bit fingerprint σύμφωνα με τον
    διαθέσιμο κώδικα των συγγραφέων του paper:

    - radius = 2
    - nBits = 1024

    Επιστρέφει None αν το SMILES δεν μπορεί να διαβαστεί.
    """

    try:
        mol = Chem.MolFromSmiles(
            str(smiles)
        )

        if mol is None:
            return None

        fingerprint = (
            AllChem.GetMorganFingerprintAsBitVect(
                mol,
                radius=MORGAN_RADIUS,
                nBits=FINGERPRINT_BITS
            )
        )

        return fingerprint

    except Exception:
        return None


# ============================================================
# 12. Συνάρτηση υπολογισμού fingerprints για DataFrame
# ============================================================

def prepare_fingerprints(
    structures_dataframe,
    dataset_name
):
    """
    Υπολογίζει fingerprints και διατηρεί μόνο τις δομές
    για τις οποίες δημιουργήθηκε έγκυρο fingerprint.
    """

    valid_rows = []
    valid_fingerprints = []

    iterator = tqdm(
        structures_dataframe[
            "Canonical_SMILES"
        ].tolist(),
        desc=f"Fingerprints: {dataset_name}"
    )

    for smiles in iterator:

        fingerprint = (
            calculate_morgan_fingerprint(
                smiles
            )
        )

        if fingerprint is not None:

            valid_rows.append({
                "Canonical_SMILES": smiles
            })

            valid_fingerprints.append(
                fingerprint
            )

    valid_dataframe = pd.DataFrame(
        valid_rows
    )

    if len(valid_dataframe) == 0:
        raise ValueError(
            f"Δεν δημιουργήθηκε κανένα έγκυρο fingerprint "
            f"για το dataset: {dataset_name}"
        )

    print(
        f"{dataset_name}: "
        f"{len(valid_dataframe):,} έγκυρα fingerprints"
    )

    return (
        valid_dataframe,
        valid_fingerprints
    )


# ============================================================
# 13. Υπολογισμός fingerprints
# ============================================================

start_fingerprint_time = time.time()

(
    training_reference_valid_df,
    training_reference_fingerprints
) = prepare_fingerprints(
    structures_dataframe=training_reference_df,
    dataset_name="Training Reference"
)


(
    generated_query_valid_df,
    generated_query_fingerprints
) = prepare_fingerprints(
    structures_dataframe=generated_query_df,
    dataset_name="Generated Query"
)


(
    training_query_valid_df,
    training_query_fingerprints
) = prepare_fingerprints(
    structures_dataframe=training_query_df,
    dataset_name="Training Query"
)


fingerprint_elapsed_time = (
    time.time() - start_fingerprint_time
)

print(
    f"\nΧρόνος δημιουργίας fingerprints: "
    f"{fingerprint_elapsed_time:.1f} δευτερόλεπτα"
)


# ============================================================
# 14. Εκτίμηση συνολικού αριθμού συγκρίσεων
# ============================================================

estimated_comparisons = (
    len(training_reference_fingerprints)
    * (
        len(generated_query_fingerprints)
        + len(training_query_fingerprints)
    )
)

print(
    f"Εκτιμώμενες Tanimoto συγκρίσεις: "
    f"{estimated_comparisons:,}"
)


# ============================================================
# 15. Συνάρτηση Nearest-Neighbor Tanimoto
# ============================================================

def calculate_nearest_neighbor_tanimoto(
    query_dataframe,
    query_fingerprints,
    reference_dataframe,
    reference_fingerprints,
    comparison_name
):
    """
    Για κάθε query fingerprint:

    1. υπολογίζει τη Tanimoto similarity προς όλα τα
       fingerprints του reference set,
    2. βρίσκει τη μέγιστη similarity,
    3. αποθηκεύει το αντίστοιχο nearest-neighbor SMILES.

    Ο υπολογισμός είναι ακριβής μέσα στο καθορισμένο
    training reference sample.
    """

    result_rows = []

    progress_bar = tqdm(
        range(len(query_fingerprints)),
        desc=comparison_name
    )

    for query_index in progress_bar:

        query_fingerprint = (
            query_fingerprints[
                query_index
            ]
        )

        similarities = (
            DataStructs.BulkTanimotoSimilarity(
                query_fingerprint,
                reference_fingerprints
            )
        )

        similarities_array = np.asarray(
            similarities,
            dtype=float
        )

        nearest_neighbor_index = int(
            np.argmax(
                similarities_array
            )
        )

        maximum_similarity = float(
            similarities_array[
                nearest_neighbor_index
            ]
        )

        result_rows.append({
            "Query_SMILES": (
                query_dataframe
                .iloc[query_index][
                    "Canonical_SMILES"
                ]
            ),
            "Nearest_Training_SMILES": (
                reference_dataframe
                .iloc[nearest_neighbor_index][
                    "Canonical_SMILES"
                ]
            ),
            "Nearest_Neighbor_Tanimoto": (
                maximum_similarity
            )
        })

    return pd.DataFrame(
        result_rows
    )


# ============================================================
# 16. Generated → Training nearest-neighbor similarity
# ============================================================

similarity_start_time = time.time()

generated_to_training_df = (
    calculate_nearest_neighbor_tanimoto(
        query_dataframe=generated_query_valid_df,
        query_fingerprints=generated_query_fingerprints,
        reference_dataframe=training_reference_valid_df,
        reference_fingerprints=training_reference_fingerprints,
        comparison_name=(
            "Generated → Training Tanimoto"
        )
    )
)


# ============================================================
# 17. Training → Training nearest-neighbor similarity
# ============================================================

training_to_training_df = (
    calculate_nearest_neighbor_tanimoto(
        query_dataframe=training_query_valid_df,
        query_fingerprints=training_query_fingerprints,
        reference_dataframe=training_reference_valid_df,
        reference_fingerprints=training_reference_fingerprints,
        comparison_name=(
            "Training → Training Tanimoto"
        )
    )
)


similarity_elapsed_time = (
    time.time() - similarity_start_time
)

print(
    f"\nΧρόνος Tanimoto comparisons: "
    f"{similarity_elapsed_time:.1f} δευτερόλεπτα"
)


# ============================================================
# 18. Αποθήκευση αποτελεσμάτων ανά query molecule
# ============================================================

generated_to_training_df.to_csv(
    generated_similarity_output,
    index=False
)

training_to_training_df.to_csv(
    training_similarity_output,
    index=False
)


# ============================================================
# 19. Συνάρτηση συνοπτικών στατιστικών
# ============================================================

def summarize_tanimoto_distribution(
    result_dataframe,
    comparison_name,
    reference_size
):
    """
    Υπολογίζει τα στατιστικά που χρειάζονται για
    τη σύγκριση των δύο distributions.

    Τα thresholds ακολουθούν την ερμηνεία του paper:
    - similarity < 0.50
    - similarity > 0.85
    """

    values = result_dataframe[
        "Nearest_Neighbor_Tanimoto"
    ].astype(float)

    percentage_below_050 = (
        (
            values
            < LOW_SIMILARITY_THRESHOLD
        ).mean()
        * 100
    )

    percentage_above_085 = (
        (
            values
            > HIGH_SIMILARITY_THRESHOLD
        ).mean()
        * 100
    )

    return {
        "Comparison": comparison_name,
        "Query_Molecules": int(
            len(values)
        ),
        "Reference_Molecules": int(
            reference_size
        ),
        "Mean_Tanimoto": float(
            values.mean()
        ),
        "Standard_Deviation": float(
            np.std(
                values,
                ddof=0
            )
        ),
        "Median_Tanimoto": float(
            values.median()
        ),
        "Minimum_Tanimoto": float(
            values.min()
        ),
        "Maximum_Tanimoto": float(
            values.max()
        ),
        "Percentage_Below_0_50": float(
            percentage_below_050
        ),
        "Percentage_Above_0_85": float(
            percentage_above_085
        )
    }


# ============================================================
# 20. Δημιουργία συνοπτικού πίνακα
# ============================================================

summary_rows = [
    summarize_tanimoto_distribution(
        result_dataframe=generated_to_training_df,
        comparison_name="Generated_to_Training",
        reference_size=len(
            training_reference_valid_df
        )
    ),
    summarize_tanimoto_distribution(
        result_dataframe=training_to_training_df,
        comparison_name="Training_to_Training",
        reference_size=len(
            training_reference_valid_df
        )
    )
]

tanimoto_summary_df = pd.DataFrame(
    summary_rows
)

numeric_summary_columns = [
    "Mean_Tanimoto",
    "Standard_Deviation",
    "Median_Tanimoto",
    "Minimum_Tanimoto",
    "Maximum_Tanimoto",
    "Percentage_Below_0_50",
    "Percentage_Above_0_85"
]

tanimoto_summary_df[
    numeric_summary_columns
] = tanimoto_summary_df[
    numeric_summary_columns
].round(6)

tanimoto_summary_df.to_csv(
    summary_output,
    index=False
)


# ============================================================
# 21. Αποθήκευση metadata μεθοδολογίας
# ============================================================

metadata_df = pd.DataFrame([
    {
        "Parameter": "Fingerprint_Type",
        "Value": "Morgan bit fingerprint"
    },
    {
        "Parameter": "Morgan_Radius",
        "Value": MORGAN_RADIUS
    },
    {
        "Parameter": "Fingerprint_Bits",
        "Value": FINGERPRINT_BITS
    },
    {
        "Parameter": "Similarity_Metric",
        "Value": "Tanimoto"
    },
    {
        "Parameter": "Generated_Query_Size",
        "Value": len(
            generated_query_valid_df
        )
    },
    {
        "Parameter": "Training_Query_Size",
        "Value": len(
            training_query_valid_df
        )
    },
    {
        "Parameter": "Training_Reference_Size",
        "Value": len(
            training_reference_valid_df
        )
    },
    {
        "Parameter": "Random_State",
        "Value": RANDOM_STATE
    },
    {
        "Parameter": "Low_Similarity_Threshold",
        "Value": LOW_SIMILARITY_THRESHOLD
    },
    {
        "Parameter": "High_Similarity_Threshold",
        "Value": HIGH_SIMILARITY_THRESHOLD
    },
    {
        "Parameter": "Methodological_Note",
        "Value": (
            "Sample-based adaptation due to the scale "
            "of the thesis datasets"
        )
    }
])

metadata_df.to_csv(
    metadata_output,
    index=False
)


# ============================================================
# 22. Δημιουργία των δύο histogram panels
# ============================================================

# Κοινά bins από 0 έως 1 ώστε οι δύο distributions
# να μπορούν να συγκριθούν άμεσα.
common_bins = np.linspace(
    0.0,
    1.0,
    21
)

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(14, 5),
    constrained_layout=True
)


# ------------------------------------------------------------
# Panel 1: Generated → Training
# ------------------------------------------------------------

axes[0].hist(
    generated_to_training_df[
        "Nearest_Neighbor_Tanimoto"
    ],
    bins=common_bins,
    density=True,
    alpha=0.75,
    edgecolor="black"
)

axes[0].axvline(
    LOW_SIMILARITY_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label="Similarity = 0.50"
)

axes[0].axvline(
    HIGH_SIMILARITY_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label="Similarity = 0.85"
)

axes[0].set_title(
    "Generated Molecules to Nearest Training Neighbor"
)

axes[0].set_xlabel(
    "Nearest-neighbor Tanimoto similarity"
)

axes[0].set_ylabel(
    "Density"
)

axes[0].set_xlim(
    0,
    1
)

axes[0].grid(
    alpha=0.25
)

axes[0].legend(
    fontsize=8
)


# ------------------------------------------------------------
# Panel 2: Training → Training
# ------------------------------------------------------------

axes[1].hist(
    training_to_training_df[
        "Nearest_Neighbor_Tanimoto"
    ],
    bins=common_bins,
    density=True,
    alpha=0.75,
    edgecolor="black"
)

axes[1].axvline(
    LOW_SIMILARITY_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label="Similarity = 0.50"
)

axes[1].axvline(
    HIGH_SIMILARITY_THRESHOLD,
    linestyle="--",
    linewidth=1.5,
    label="Similarity = 0.85"
)

axes[1].set_title(
    "Training Molecules to Nearest Training Neighbor"
)

axes[1].set_xlabel(
    "Nearest-neighbor Tanimoto similarity"
)

axes[1].set_ylabel(
    "Density"
)

axes[1].set_xlim(
    0,
    1
)

axes[1].grid(
    alpha=0.25
)

axes[1].legend(
    fontsize=8
)


fig.suptitle(
    (
        "Nearest-Neighbor Structural Similarity "
        "(Morgan Radius 2, 1024 Bits)"
    ),
    fontsize=14
)

fig.savefig(
    figure_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 23. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 80)
print("ΣΥΝΟΠΤΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ TANIMOTO")
print("=" * 80)

display(
    tanimoto_summary_df
)


print("\n" + "=" * 80)
print("METADATA ΑΝΑΛΥΣΗΣ")
print("=" * 80)

display(
    metadata_df
)


# ============================================================
# 24. Τελική αναφορά αρχείων
# ============================================================

print("\n" + "-" * 80)
print("Η nearest-neighbor Tanimoto analysis ολοκληρώθηκε.")
print("-" * 80)

print(
    f"Generated → Training αποτελέσματα:\n"
    f"{generated_similarity_output}"
)

print(
    f"\nTraining → Training αποτελέσματα:\n"
    f"{training_similarity_output}"
)

print(
    f"\nΣυνοπτικά στατιστικά:\n"
    f"{summary_output}"
)

print(
    f"\nMetadata μεθοδολογίας:\n"
    f"{metadata_output}"
)

print(
    f"\nΣχήμα κατανομών 300 dpi:\n"
    f"{figure_output}"
)

print(
    "\nΣημαντική σημείωση: "
    "Τα αποτελέσματα είναι sample-based estimates "
    "ως προς το training sample των 150.000 μορίων."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 5A:
# Προκαθορισμός κριτηρίων και δημιουργία του
# Property-Qualified Generated Candidate Pool
#
# Πιστή βάση από το paper:
# - logP <= 5
# - Molecular Weight < 500 Da
# - HBD <= 5
# - HBA <= 10
# - Rotatable Bonds <= 10
# - TPSA <= 140 Å²
# - SAS <= 6
#
# Ρητές προσαρμογές της πτυχιακής:
# - ακριβής canonical novelty έναντι ολόκληρου του training set
# - absolute logP error <= 0.5 λόγω logP conditioning
#
# Δεν εφαρμόζεται ακόμη:
# - QED cutoff
# - composite score
# - structural diversity μεταξύ τελικών candidates
# - docking ή target-specific screening
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from IPython.display import display


# ============================================================
# 1. Ορισμός αρχείων εισόδου
# ============================================================

generated_file = (
    "GENERATED_LIBRARY_WITH_DRUG_LIKENESS_FLAGS.csv"
)

processed_file = (
    "processed_data.csv"
)


# ============================================================
# 2. Ορισμός αρχείων εξόδου
# ============================================================

candidate_pool_output = (
    "PROPERTY_QUALIFIED_GENERATED_CANDIDATE_POOL.csv"
)

filter_flow_output = (
    "CANDIDATE_PREFILTER_FLOW_SUMMARY.csv"
)

criteria_output = (
    "CANDIDATE_SELECTION_CRITERIA.csv"
)


# ============================================================
# 3. Έλεγχος ύπαρξης αρχείων
# ============================================================

if not os.path.exists(generated_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το generated αρχείο: {generated_file}"
    )

if not os.path.exists(processed_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το processed αρχείο: {processed_file}"
    )


# ============================================================
# 4. Φόρτωση δεδομένων
# ============================================================

generated_df = pd.read_csv(
    generated_file
)

processed_df = pd.read_csv(
    processed_file
)

print("=" * 82)
print("ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ ΓΙΑ CANDIDATE PREFILTERING")
print("=" * 82)

print(
    f"Generated canonical unique molecules: "
    f"{len(generated_df):,}"
)

print(
    f"Συνολικά processed molecules πριν το split: "
    f"{len(processed_df):,}"
)


# ============================================================
# 5. Έλεγχος απαιτούμενων στηλών
# ============================================================

required_generated_columns = [
    "Target_logP",
    "LogP_RDKit",
    "Molecular_Weight",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "TPSA",
    "QED",
    "SAS",
    "Pass_Zero_Lipinski_Violations"
]

required_processed_columns = [
    "Clean_SMILES",
    "SELFIES",
    "logP"
]


missing_generated_columns = [
    column
    for column in required_generated_columns
    if column not in generated_df.columns
]

missing_processed_columns = [
    column
    for column in required_processed_columns
    if column not in processed_df.columns
]

if missing_generated_columns:
    raise ValueError(
        "Λείπουν οι ακόλουθες στήλες από το generated αρχείο:\n"
        f"{missing_generated_columns}\n\n"
        f"Διαθέσιμες στήλες:\n"
        f"{generated_df.columns.tolist()}"
    )

if missing_processed_columns:
    raise ValueError(
        "Λείπουν οι ακόλουθες στήλες από το processed αρχείο:\n"
        f"{missing_processed_columns}"
    )


# ============================================================
# 6. Εντοπισμός canonical SMILES στήλης
# ============================================================

generated_smiles_candidates = [
    "Canonical_SMILES_Check",
    "Canonical_SMILES",
    "Evaluation_SMILES",
    "SMILES"
]

generated_smiles_column = None

for candidate_column in generated_smiles_candidates:
    if candidate_column in generated_df.columns:
        generated_smiles_column = candidate_column
        break

if generated_smiles_column is None:
    raise ValueError(
        "Δεν βρέθηκε κατάλληλη canonical SMILES στήλη "
        "στο generated dataset."
    )

print(
    f"\nCanonical SMILES στήλη generated dataset: "
    f"{generated_smiles_column}"
)


# ============================================================
# 7. Μετατροπή αριθμητικών στηλών
# ============================================================

numeric_columns = [
    "Target_logP",
    "LogP_RDKit",
    "Molecular_Weight",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "TPSA",
    "QED",
    "SAS"
]

for column in numeric_columns:
    generated_df[column] = pd.to_numeric(
        generated_df[column],
        errors="coerce"
    )

generated_df[numeric_columns] = (
    generated_df[numeric_columns]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


# ============================================================
# 8. Έλεγχος ελλιπών τιμών
# ============================================================

missing_numeric_counts = (
    generated_df[numeric_columns]
    .isna()
    .sum()
)

if missing_numeric_counts.sum() > 0:
    raise ValueError(
        "Βρέθηκαν κενές ή μη αριθμητικές τιμές:\n"
        f"{missing_numeric_counts[missing_numeric_counts > 0]}"
    )

generated_df[generated_smiles_column] = (
    generated_df[generated_smiles_column]
    .astype(str)
)

generated_df = generated_df.dropna(
    subset=[generated_smiles_column]
).copy()


# ============================================================
# 9. Αναδημιουργία ακριβώς του πλήρους training set
# ============================================================

processed_df = processed_df.dropna(
    subset=[
        "Clean_SMILES",
        "SELFIES",
        "logP"
    ]
).copy()

train_df, temporary_df = train_test_split(
    processed_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("\nΤο αρχικό split αναδημιουργήθηκε.")

print(
    f"Train set:      {len(train_df):,}"
)

print(
    f"Validation set: {len(validation_df):,}"
)

print(
    f"Test set:       {len(test_df):,}"
)


# ============================================================
# 10. Δημιουργία set όλων των training canonical SMILES
# ============================================================

training_canonical_smiles_set = set(
    train_df["Clean_SMILES"]
    .astype(str)
)

print(
    f"Canonical SMILES στο πλήρες training reference: "
    f"{len(training_canonical_smiles_set):,}"
)


# ============================================================
# 11. Προκαθορισμένες τιμές κριτηρίων
# ============================================================

MAX_LOGP = 5.0
MAX_MOLECULAR_WEIGHT_EXCLUSIVE = 500.0
MAX_HBD = 5
MAX_HBA = 10
MAX_ROTATABLE_BONDS = 10
MAX_TPSA = 140.0
MAX_SAS = 6.0

# Ρητή προσαρμογή λόγω conditional generation.
MAX_ABSOLUTE_LOGP_ERROR = 0.5


# ============================================================
# 12. Αποθήκευση των κριτηρίων πριν την επιλογή
# ============================================================

criteria_df = pd.DataFrame([
    {
        "Order": 1,
        "Criterion": "Canonical novelty",
        "Rule": (
            "Generated canonical SMILES must not exist "
            "in the full recreated training set"
        ),
        "Threshold": "Exact non-membership",
        "Methodological_Status": (
            "Thesis adaptation using exact training membership"
        )
    },
    {
        "Order": 2,
        "Criterion": "Strict Lipinski compliance",
        "Rule": (
            "Zero violations across logP, MW, HBD and HBA"
        ),
        "Threshold": "0 violations",
        "Methodological_Status": (
            "Paper-aligned physicochemical constraints"
        )
    },
    {
        "Order": 3,
        "Criterion": "Rotatable bonds",
        "Rule": "Rotatable_Bonds <= 10",
        "Threshold": "<= 10",
        "Methodological_Status": (
            "Paper-aligned physicochemical constraint"
        )
    },
    {
        "Order": 4,
        "Criterion": "TPSA",
        "Rule": "TPSA <= 140 square Angstrom",
        "Threshold": "<= 140",
        "Methodological_Status": (
            "Paper-aligned physicochemical constraint"
        )
    },
    {
        "Order": 5,
        "Criterion": "Synthetic accessibility",
        "Rule": "SAS <= 6",
        "Threshold": "<= 6",
        "Methodological_Status": (
            "Paper-aligned physicochemical constraint"
        )
    },
    {
        "Order": 6,
        "Criterion": "Conditional logP fidelity",
        "Rule": (
            "Absolute difference between RDKit logP "
            "and target logP <= 0.5"
        ),
        "Threshold": "<= 0.5",
        "Methodological_Status": (
            "Explicit thesis adaptation for logP conditioning"
        )
    },
    {
        "Order": 7,
        "Criterion": "QED",
        "Rule": (
            "No hard cutoff; higher values will be preferred "
            "during the subsequent ranking stage"
        ),
        "Threshold": "No hard threshold",
        "Methodological_Status": (
            "Continuous paper-aligned metric; thesis ranking adaptation"
        )
    }
])

criteria_df.to_csv(
    criteria_output,
    index=False
)


# ============================================================
# 13. Υπολογισμός ακριβούς canonical novelty
# ============================================================

generated_df["Is_Canonically_Novel"] = (
    ~generated_df[generated_smiles_column]
    .isin(training_canonical_smiles_set)
)


# ============================================================
# 14. Υπολογισμός absolute logP error
# ============================================================

generated_df["Absolute_logP_Error"] = (
    generated_df["LogP_RDKit"]
    - generated_df["Target_logP"]
).abs()


# ============================================================
# 15. Επαναϋπολογισμός των επιμέρους flags
#
# Δεν βασιζόμαστε μόνο στα flags του προηγούμενου κελιού.
# Τα κριτήρια επανυπολογίζονται άμεσα από τις descriptor τιμές
# για ανεξάρτητη επιβεβαίωση της επιλογής.
# ============================================================

generated_df["Candidate_Pass_LogP"] = (
    generated_df["LogP_RDKit"]
    <= MAX_LOGP
)

generated_df["Candidate_Pass_MW"] = (
    generated_df["Molecular_Weight"]
    < MAX_MOLECULAR_WEIGHT_EXCLUSIVE
)

generated_df["Candidate_Pass_HBD"] = (
    generated_df["HBD"]
    <= MAX_HBD
)

generated_df["Candidate_Pass_HBA"] = (
    generated_df["HBA"]
    <= MAX_HBA
)

generated_df["Candidate_Pass_Zero_Lipinski"] = (
    generated_df[
        [
            "Candidate_Pass_LogP",
            "Candidate_Pass_MW",
            "Candidate_Pass_HBD",
            "Candidate_Pass_HBA"
        ]
    ]
    .all(axis=1)
)

generated_df["Candidate_Pass_Rotatable_Bonds"] = (
    generated_df["Rotatable_Bonds"]
    <= MAX_ROTATABLE_BONDS
)

generated_df["Candidate_Pass_TPSA"] = (
    generated_df["TPSA"]
    <= MAX_TPSA
)

generated_df["Candidate_Pass_SAS"] = (
    generated_df["SAS"]
    <= MAX_SAS
)

generated_df["Candidate_Pass_logP_Error"] = (
    generated_df["Absolute_logP_Error"]
    <= MAX_ABSOLUTE_LOGP_ERROR
)


# ============================================================
# 16. Έλεγχος συμφωνίας με το προηγούμενο Lipinski flag
# ============================================================

previous_lipinski_flag = (
    generated_df[
        "Pass_Zero_Lipinski_Violations"
    ]
    .astype(bool)
)

recalculated_lipinski_flag = (
    generated_df[
        "Candidate_Pass_Zero_Lipinski"
    ]
)

lipinski_flag_disagreements = int(
    (
        previous_lipinski_flag
        != recalculated_lipinski_flag
    ).sum()
)

print(
    f"\nΔιαφωνίες με το προηγούμενο Lipinski flag: "
    f"{lipinski_flag_disagreements:,}"
)

if lipinski_flag_disagreements > 0:
    raise ValueError(
        "Ο επανυπολογισμός των Lipinski criteria "
        "δεν συμφωνεί με το προηγούμενο αρχείο."
    )


# ============================================================
# 17. Συνάρτηση καταγραφής της ροής φιλτραρίσματος
# ============================================================

filter_flow_rows = []


def record_filter_stage(
    stage_number,
    stage_name,
    dataframe_before,
    dataframe_after
):
    """
    Καταγράφει πόσα μόρια παραμένουν και πόσα αποκλείονται
    σε κάθε διαδοχικό στάδιο φιλτραρίσματος.
    """

    molecules_before = len(
        dataframe_before
    )

    molecules_after = len(
        dataframe_after
    )

    excluded_at_stage = (
        molecules_before
        - molecules_after
    )

    percentage_of_initial = (
        molecules_after
        / len(generated_df)
    ) * 100

    percentage_retained_from_previous = (
        molecules_after
        / molecules_before
    ) * 100 if molecules_before > 0 else 0.0

    filter_flow_rows.append({
        "Stage": stage_number,
        "Filter": stage_name,
        "Molecules_Before": molecules_before,
        "Molecules_After": molecules_after,
        "Excluded_At_Stage": excluded_at_stage,
        "Retained_From_Previous_Percentage": (
            percentage_retained_from_previous
        ),
        "Retained_From_Initial_Percentage": (
            percentage_of_initial
        )
    })


# ============================================================
# 18. Διαδοχική εφαρμογή των προκαθορισμένων φίλτρων
# ============================================================

current_pool_df = generated_df.copy()


# ------------------------------------------------------------
# Στάδιο 1: Exact canonical novelty
# ------------------------------------------------------------

previous_pool_df = current_pool_df.copy()

current_pool_df = current_pool_df[
    current_pool_df["Is_Canonically_Novel"]
].copy()

record_filter_stage(
    stage_number=1,
    stage_name=(
        "Canonical novelty versus full training set"
    ),
    dataframe_before=previous_pool_df,
    dataframe_after=current_pool_df
)


# ------------------------------------------------------------
# Στάδιο 2: Zero strict Lipinski violations
# ------------------------------------------------------------

previous_pool_df = current_pool_df.copy()

current_pool_df = current_pool_df[
    current_pool_df[
        "Candidate_Pass_Zero_Lipinski"
    ]
].copy()

record_filter_stage(
    stage_number=2,
    stage_name=(
        "Zero strict Lipinski violations"
    ),
    dataframe_before=previous_pool_df,
    dataframe_after=current_pool_df
)


# ------------------------------------------------------------
# Στάδιο 3: Rotatable Bonds <= 10
# ------------------------------------------------------------

previous_pool_df = current_pool_df.copy()

current_pool_df = current_pool_df[
    current_pool_df[
        "Candidate_Pass_Rotatable_Bonds"
    ]
].copy()

record_filter_stage(
    stage_number=3,
    stage_name=(
        "Rotatable Bonds <= 10"
    ),
    dataframe_before=previous_pool_df,
    dataframe_after=current_pool_df
)


# ------------------------------------------------------------
# Στάδιο 4: TPSA <= 140 Å²
# ------------------------------------------------------------

previous_pool_df = current_pool_df.copy()

current_pool_df = current_pool_df[
    current_pool_df[
        "Candidate_Pass_TPSA"
    ]
].copy()

record_filter_stage(
    stage_number=4,
    stage_name=(
        "TPSA <= 140 square Angstrom"
    ),
    dataframe_before=previous_pool_df,
    dataframe_after=current_pool_df
)


# ------------------------------------------------------------
# Στάδιο 5: SAS <= 6
# ------------------------------------------------------------

previous_pool_df = current_pool_df.copy()

current_pool_df = current_pool_df[
    current_pool_df[
        "Candidate_Pass_SAS"
    ]
].copy()

record_filter_stage(
    stage_number=5,
    stage_name=(
        "SAS <= 6"
    ),
    dataframe_before=previous_pool_df,
    dataframe_after=current_pool_df
)


# ------------------------------------------------------------
# Στάδιο 6: Absolute logP error <= 0.5
# ------------------------------------------------------------

previous_pool_df = current_pool_df.copy()

current_pool_df = current_pool_df[
    current_pool_df[
        "Candidate_Pass_logP_Error"
    ]
].copy()

record_filter_stage(
    stage_number=6,
    stage_name=(
        "Absolute logP error <= 0.5"
    ),
    dataframe_before=previous_pool_df,
    dataframe_after=current_pool_df
)


# ============================================================
# 19. Τελικός έλεγχος ότι όλα τα μόρια περνούν τα criteria
# ============================================================

final_required_flags = [
    "Is_Canonically_Novel",
    "Candidate_Pass_Zero_Lipinski",
    "Candidate_Pass_Rotatable_Bonds",
    "Candidate_Pass_TPSA",
    "Candidate_Pass_SAS",
    "Candidate_Pass_logP_Error"
]

failed_final_checks = (
    ~current_pool_df[
        final_required_flags
    ].all(axis=1)
).sum()

if failed_final_checks > 0:
    raise RuntimeError(
        "Βρέθηκαν μόρια στο candidate pool που δεν "
        "ικανοποιούν όλα τα προκαθορισμένα κριτήρια."
    )


# ============================================================
# 20. Προσθήκη αναγνωριστικού υποψηφίου
# ============================================================

current_pool_df = (
    current_pool_df
    .reset_index(drop=True)
)

current_pool_df.insert(
    0,
    "Candidate_Pool_ID",
    [
        f"PGC_{index:06d}"
        for index in range(
            1,
            len(current_pool_df) + 1
        )
    ]
)


# ============================================================
# 21. Αποθήκευση candidate pool
# ============================================================

current_pool_df.to_csv(
    candidate_pool_output,
    index=False
)


# ============================================================
# 22. Δημιουργία και αποθήκευση flow summary
# ============================================================

filter_flow_df = pd.DataFrame(
    filter_flow_rows
)

percentage_columns = [
    "Retained_From_Previous_Percentage",
    "Retained_From_Initial_Percentage"
]

filter_flow_df[
    percentage_columns
] = filter_flow_df[
    percentage_columns
].round(4)

filter_flow_df.to_csv(
    filter_flow_output,
    index=False
)


# ============================================================
# 23. Περιγραφικά στατιστικά του τελικού pool
# ============================================================

candidate_pool_summary_df = (
    current_pool_df[
        [
            "QED",
            "SAS",
            "Absolute_logP_Error",
            "LogP_RDKit",
            "Molecular_Weight",
            "HBD",
            "HBA",
            "Rotatable_Bonds",
            "TPSA"
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
    .transpose()
    .round(6)
)


# ============================================================
# 24. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 82)
print("ΠΡΟΚΑΘΟΡΙΣΜΕΝΑ ΚΡΙΤΗΡΙΑ")
print("=" * 82)

display(
    criteria_df
)


print("\n" + "=" * 82)
print("ΡΟΗ ΠΡΟΦΙΛΤΡΑΡΙΣΜΑΤΟΣ")
print("=" * 82)

display(
    filter_flow_df
)


print("\n" + "=" * 82)
print("ΠΕΡΙΓΡΑΦΙΚΑ ΣΤΑΤΙΣΤΙΚΑ ΤΟΥ CANDIDATE POOL")
print("=" * 82)

display(
    candidate_pool_summary_df
)


# ============================================================
# 25. Τελική αναφορά
# ============================================================

print("\n" + "-" * 82)
print("ΤΟ PROPERTY-QUALIFIED CANDIDATE POOL ΔΗΜΙΟΥΡΓΗΘΗΚΕ")
print("-" * 82)

print(
    f"Αρχικά generated molecules: "
    f"{len(generated_df):,}"
)

print(
    f"Μόρια στο τελικό property-qualified pool: "
    f"{len(current_pool_df):,}"
)

print(
    f"Ποσοστό της αρχικής generated βιβλιοθήκης: "
    f"{(len(current_pool_df) / len(generated_df)) * 100:.4f}%"
)

print(
    f"\nCandidate pool:\n"
    f"{candidate_pool_output}"
)

print(
    f"\nΡοή φιλτραρίσματος:\n"
    f"{filter_flow_output}"
)

print(
    f"\nΑρχείο προκαθορισμένων κριτηρίων:\n"
    f"{criteria_output}"
)

print(
    "\nΣημείωση: Τα μόρια του pool δεν αποτελούν ακόμη "
    "τελικούς υποψηφίους. Θα ακολουθήσει κατάταξη και "
    "δομική διαφοροποίηση."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 5B:
# Pareto selection και MaxMin δομική διαφοροποίηση
# των 20 Selected Generated Candidates
#
# PARETO ΣΤΟΧΟΙ:
# 1. Μεγιστοποίηση QED
# 2. Ελαχιστοποίηση SAS
# 3. Ελαχιστοποίηση Absolute logP Error
#
# ΔΟΜΙΚΗ ΔΙΑΦΟΡΟΠΟΙΗΣΗ:
# - Morgan fingerprints
# - radius = 2
# - 1024 bits
# - Tanimoto similarity
# - RDKit MaxMinPicker
#
# Σημαντικό:
# Η Pareto + MaxMin διαδικασία είναι ρητή target-independent
# προσαρμογή της πτυχιακής και όχι αυτούσια μέθοδος
# τελικής επιλογής από το βασικό paper.
# ============================================================

import os
import time

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.SimDivFilters import rdSimDivPickers

from tqdm.notebook import tqdm
from IPython.display import display


# ============================================================
# 1. Ρυθμίσεις RDKit
# ============================================================

RDLogger.DisableLog("rdApp.*")


# ============================================================
# 2. Αρχείο εισόδου
# ============================================================

candidate_pool_file = (
    "PROPERTY_QUALIFIED_GENERATED_CANDIDATE_POOL.csv"
)


# ============================================================
# 3. Αρχεία εξόδου
# ============================================================

pareto_output_file = (
    "PARETO_CANDIDATE_SET_FOR_SELECTION.csv"
)

selected_candidates_output_file = (
    "SELECTED_20_GENERATED_CANDIDATES.csv"
)

pairwise_matrix_output_file = (
    "SELECTED_20_PAIRWISE_TANIMOTO_MATRIX.csv"
)

diversity_summary_output_file = (
    "SELECTED_20_DIVERSITY_SUMMARY.csv"
)

selection_metadata_output_file = (
    "SELECTED_20_SELECTION_METADATA.csv"
)


# ============================================================
# 4. Προκαθορισμένες παράμετροι
# ============================================================

FINAL_SELECTION_SIZE = 20

RANDOM_STATE = 42

MORGAN_RADIUS = 2
FINGERPRINT_BITS = 1024

# Αριθμητική ανοχή για τις συγκρίσεις Pareto.
PARETO_EPSILON = 1e-12


# ============================================================
# 5. Έλεγχος ύπαρξης αρχείου
# ============================================================

if not os.path.exists(candidate_pool_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το candidate pool:\n"
        f"{candidate_pool_file}"
    )


# ============================================================
# 6. Φόρτωση candidate pool
# ============================================================

candidate_pool_df = pd.read_csv(
    candidate_pool_file
)

print("=" * 85)
print("ΦΟΡΤΩΣΗ PROPERTY-QUALIFIED CANDIDATE POOL")
print("=" * 85)

print(
    f"Μόρια στο candidate pool: "
    f"{len(candidate_pool_df):,}"
)


# ============================================================
# 7. Έλεγχος απαιτούμενων στηλών
# ============================================================

required_columns = [
    "Candidate_Pool_ID",
    "QED",
    "SAS",
    "Absolute_logP_Error",
    "Target_logP",
    "LogP_RDKit",
    "Molecular_Weight",
    "HBD",
    "HBA",
    "Rotatable_Bonds",
    "TPSA"
]

missing_columns = [
    column
    for column in required_columns
    if column not in candidate_pool_df.columns
]

if missing_columns:
    raise ValueError(
        "Λείπουν οι παρακάτω απαιτούμενες στήλες:\n"
        f"{missing_columns}\n\n"
        f"Διαθέσιμες στήλες:\n"
        f"{candidate_pool_df.columns.tolist()}"
    )


# ============================================================
# 8. Εντοπισμός της κατάλληλης canonical SMILES στήλης
# ============================================================

candidate_smiles_columns = [
    "Canonical_SMILES_Check",
    "Canonical_SMILES",
    "Evaluation_SMILES",
    "SMILES"
]

smiles_column = None

for candidate_column in candidate_smiles_columns:
    if candidate_column in candidate_pool_df.columns:
        smiles_column = candidate_column
        break

if smiles_column is None:
    raise ValueError(
        "Δεν βρέθηκε κατάλληλη SMILES στήλη.\n"
        f"Διαθέσιμες στήλες:\n"
        f"{candidate_pool_df.columns.tolist()}"
    )

print(
    f"Στήλη που θα χρησιμοποιηθεί για fingerprints: "
    f"{smiles_column}"
)


# ============================================================
# 9. Μετατροπή των Pareto objectives σε αριθμητική μορφή
# ============================================================

pareto_objective_columns = [
    "QED",
    "SAS",
    "Absolute_logP_Error"
]

for column in pareto_objective_columns:
    candidate_pool_df[column] = pd.to_numeric(
        candidate_pool_df[column],
        errors="coerce"
    )

candidate_pool_df[
    pareto_objective_columns
] = candidate_pool_df[
    pareto_objective_columns
].replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# 10. Αυστηρός έλεγχος κενών τιμών
# ============================================================

missing_objective_values = (
    candidate_pool_df[
        pareto_objective_columns
    ]
    .isna()
    .sum()
)

if missing_objective_values.sum() > 0:
    raise ValueError(
        "Βρέθηκαν κενές ή μη αριθμητικές τιμές "
        "στα Pareto objectives:\n"
        f"{missing_objective_values[missing_objective_values > 0]}"
    )

candidate_pool_df[smiles_column] = (
    candidate_pool_df[smiles_column]
    .astype(str)
)


# ============================================================
# 11. Βοηθητική δομή Fenwick Tree για ακριβή Pareto front
# ============================================================

class FenwickMinimum:
    """
    Fenwick tree που αποθηκεύει ελάχιστες τιμές.

    Χρησιμοποιείται για αποδοτικό και ακριβή εντοπισμό
    του Pareto front σε τρεις στόχους χωρίς να γίνουν
    όλες οι δυνατές συγκρίσεις ανά ζεύγος μορίων.
    """

    def __init__(self, size):
        self.size = size

        self.tree = np.full(
            size + 1,
            np.inf,
            dtype=float
        )

    def update(self, position, value):
        """
        Ενημερώνει τη δομή με μία νέα τιμή.
        """

        while position <= self.size:

            if value < self.tree[position]:
                self.tree[position] = value

            position += (
                position & -position
            )

    def query_prefix_minimum(self, position):
        """
        Επιστρέφει την ελάχιστη τιμή από την αρχή
        έως και τη συγκεκριμένη θέση.
        """

        minimum_value = np.inf

        while position > 0:

            if self.tree[position] < minimum_value:
                minimum_value = self.tree[position]

            position -= (
                position & -position
            )

        return minimum_value


# ============================================================
# 12. Συνάρτηση εξαγωγής ενός ακριβούς Pareto front
# ============================================================

def extract_pareto_front(dataframe):
    """
    Εξάγει το πρώτο μη κυριαρχούμενο Pareto front.

    Στόχοι:
    - QED: μεγιστοποίηση
    - SAS: ελαχιστοποίηση
    - Absolute_logP_Error: ελαχιστοποίηση

    Ένα μόριο θεωρείται dominated όταν υπάρχει άλλο
    μόριο που είναι τουλάχιστον εξίσου καλό και στους
    τρεις στόχους και αυστηρά καλύτερο σε τουλάχιστον έναν.
    """

    if len(dataframe) == 0:
        return dataframe.copy()


    # --------------------------------------------------------
    # Ταξινόμηση με βάση την επιθυμητή κατεύθυνση
    # των τριών στόχων.
    # --------------------------------------------------------

    sorted_df = (
        dataframe[
            [
                "QED",
                "SAS",
                "Absolute_logP_Error"
            ]
        ]
        .copy()
    )

    sorted_df["_Original_Index"] = (
        dataframe.index
    )

    sorted_df = (
        sorted_df
        .sort_values(
            by=[
                "QED",
                "SAS",
                "Absolute_logP_Error"
            ],
            ascending=[
                False,  # Υψηλότερο QED
                True,   # Χαμηλότερο SAS
                True    # Χαμηλότερο logP error
            ],
            kind="mergesort"
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Συμπίεση των SAS τιμών για χρήση στο Fenwick tree.
    # --------------------------------------------------------

    unique_sas_values = np.sort(
        sorted_df["SAS"].unique()
    )

    sas_positions = (
        np.searchsorted(
            unique_sas_values,
            sorted_df["SAS"].to_numpy()
        )
        + 1
    )

    minimum_error_tree = FenwickMinimum(
        size=len(unique_sas_values)
    )


    # --------------------------------------------------------
    # Αποθήκευση των αρχικών indices των Pareto μορίων.
    # --------------------------------------------------------

    pareto_original_indices = []

    objective_values = sorted_df[
        [
            "QED",
            "SAS",
            "Absolute_logP_Error"
        ]
    ].to_numpy(dtype=float)


    # --------------------------------------------------------
    # Επεξεργασία ομάδων με απολύτως ίδιους στόχους.
    #
    # Τα μόρια με ίδιες τιμές και στους τρεις στόχους
    # αντιμετωπίζονται μαζί, ώστε να μη θεωρηθεί λανθασμένα
    # ότι το ένα κυριαρχεί το άλλο.
    # --------------------------------------------------------

    row_index = 0
    total_rows = len(sorted_df)

    while row_index < total_rows:

        group_end_index = row_index + 1

        while (
            group_end_index < total_rows
            and
            np.array_equal(
                objective_values[group_end_index],
                objective_values[row_index]
            )
        ):
            group_end_index += 1


        current_sas_position = int(
            sas_positions[row_index]
        )

        current_error = float(
            objective_values[
                row_index,
                2
            ]
        )

        best_previous_error = (
            minimum_error_tree
            .query_prefix_minimum(
                current_sas_position
            )
        )


        # Αν δεν υπάρχει προηγούμενο μόριο με:
        # - υψηλότερο ή ίσο QED,
        # - χαμηλότερο ή ίσο SAS,
        # - χαμηλότερο ή ίσο error,
        # τότε η ομάδα ανήκει στο Pareto front.
        is_non_dominated = (
            best_previous_error
            > current_error + PARETO_EPSILON
        )

        if is_non_dominated:

            group_original_indices = (
                sorted_df
                .iloc[
                    row_index:group_end_index
                ]["_Original_Index"]
                .tolist()
            )

            pareto_original_indices.extend(
                group_original_indices
            )


        # Η ενημέρωση γίνεται μετά την αξιολόγηση ολόκληρης
        # της ομάδας ίδιων objective values.
        minimum_error_tree.update(
            current_sas_position,
            current_error
        )

        row_index = group_end_index


    return dataframe.loc[
        pareto_original_indices
    ].copy()


# ============================================================
# 13. Δημιουργία διαδοχικών Pareto layers
#
# Κανονικά περιμένουμε το πρώτο front να περιλαμβάνει
# περισσότερα από 20 μόρια.
#
# Αν περιλαμβάνει λιγότερα, προσθέτουμε το δεύτερο front,
# μετά το τρίτο κ.ο.κ., μόνο μέχρι να υπάρχει επαρκής
# αριθμός μορίων για την τελική επιλογή.
# ============================================================

pareto_start_time = time.time()

remaining_candidates_df = (
    candidate_pool_df.copy()
)

pareto_layers = []

total_pareto_candidates = 0
pareto_layer_number = 1

while total_pareto_candidates < FINAL_SELECTION_SIZE:

    current_front_df = extract_pareto_front(
        remaining_candidates_df
    )

    if len(current_front_df) == 0:
        raise RuntimeError(
            "Δεν ήταν δυνατή η δημιουργία επαρκούς "
            "Pareto candidate set."
        )

    current_front_df[
        "Pareto_Layer"
    ] = pareto_layer_number

    pareto_layers.append(
        current_front_df
    )

    total_pareto_candidates += len(
        current_front_df
    )

    remaining_candidates_df = (
        remaining_candidates_df
        .drop(
            index=current_front_df.index
        )
        .copy()
    )

    print(
        f"Pareto layer {pareto_layer_number}: "
        f"{len(current_front_df):,} μόρια"
    )

    pareto_layer_number += 1


pareto_candidate_df = pd.concat(
    pareto_layers,
    axis=0
).copy()


# ============================================================
# 14. Ταξινόμηση του Pareto candidate set
# ============================================================

pareto_candidate_df = (
    pareto_candidate_df
    .sort_values(
        by=[
            "Pareto_Layer",
            "QED",
            "SAS",
            "Absolute_logP_Error"
        ],
        ascending=[
            True,
            False,
            True,
            True
        ],
        kind="mergesort"
    )
    .reset_index(drop=True)
)

pareto_candidate_df.insert(
    0,
    "Pareto_Set_ID",
    [
        f"PAR_{index:06d}"
        for index in range(
            1,
            len(pareto_candidate_df) + 1
        )
    ]
)

pareto_elapsed_time = (
    time.time() - pareto_start_time
)

print(
    f"\nΣυνολικά μόρια στο Pareto candidate set: "
    f"{len(pareto_candidate_df):,}"
)

print(
    f"Χρόνος Pareto υπολογισμού: "
    f"{pareto_elapsed_time:.2f} δευτερόλεπτα"
)


# ============================================================
# 15. Αποθήκευση του Pareto candidate set
# ============================================================

pareto_candidate_df.to_csv(
    pareto_output_file,
    index=False
)


# ============================================================
# 16. Δημιουργία Morgan fingerprint generator
# ============================================================

morgan_generator = (
    rdFingerprintGenerator
    .GetMorganGenerator(
        radius=MORGAN_RADIUS,
        fpSize=FINGERPRINT_BITS,
        includeChirality=False
    )
)


# ============================================================
# 17. Υπολογισμός fingerprints του Pareto candidate set
# ============================================================

pareto_fingerprints = []

print(
    "\nΥπολογισμός Morgan fingerprints "
    "για το Pareto candidate set..."
)

for smiles in tqdm(
    pareto_candidate_df[smiles_column],
    desc="Pareto fingerprints"
):

    molecule = Chem.MolFromSmiles(
        str(smiles)
    )

    if molecule is None:
        raise ValueError(
            "Βρέθηκε μη έγκυρο SMILES μέσα στο "
            "property-qualified candidate pool."
        )

    fingerprint = (
        morgan_generator
        .GetFingerprint(
            molecule
        )
    )

    pareto_fingerprints.append(
        fingerprint
    )


if len(pareto_fingerprints) != len(
    pareto_candidate_df
):
    raise RuntimeError(
        "Ο αριθμός fingerprints δεν συμφωνεί "
        "με τον αριθμό Pareto candidates."
    )


# ============================================================
# 18. Καθορισμός αρχικών picks για τον MaxMinPicker
# ============================================================

first_layer_positions = (
    pareto_candidate_df.index[
        pareto_candidate_df[
            "Pareto_Layer"
        ] == 1
    ]
    .tolist()
)


if len(first_layer_positions) >= FINAL_SELECTION_SIZE:

    # Όταν το πρώτο Pareto front έχει τουλάχιστον 20 μόρια,
    # ξεκινάμε με ένα deterministic seed molecule:
    #
    # 1. υψηλότερο QED
    # 2. σε ισοβαθμία, χαμηλότερο SAS
    # 3. σε νέα ισοβαθμία, χαμηλότερο logP error
    #
    # Δεν δημιουργείται weighted composite score.

    deterministic_seed_position = int(
        pareto_candidate_df
        .sort_values(
            by=[
                "QED",
                "SAS",
                "Absolute_logP_Error"
            ],
            ascending=[
                False,
                True,
                True
            ],
            kind="mergesort"
        )
        .index[0]
    )

    first_picks = [
        deterministic_seed_position
    ]

else:

    # Αν το πρώτο front έχει λιγότερα από 20 μόρια,
    # διατηρούνται όλα υποχρεωτικά και ο MaxMinPicker
    # συμπληρώνει τις υπόλοιπες θέσεις από τα επόμενα layers.

    first_layer_sorted_positions = (
        pareto_candidate_df[
            pareto_candidate_df[
                "Pareto_Layer"
            ] == 1
        ]
        .sort_values(
            by=[
                "QED",
                "SAS",
                "Absolute_logP_Error"
            ],
            ascending=[
                False,
                True,
                True
            ],
            kind="mergesort"
        )
        .index
        .tolist()
    )

    first_picks = [
        int(position)
        for position in first_layer_sorted_positions
    ]


print(
    f"Αρχικά υποχρεωτικά picks για MaxMin: "
    f"{len(first_picks)}"
)


# ============================================================
# 19. MaxMin diversity selection
# ============================================================

if len(pareto_candidate_df) < FINAL_SELECTION_SIZE:
    raise ValueError(
        "Το Pareto candidate set έχει λιγότερα από "
        f"{FINAL_SELECTION_SIZE} μόρια."
    )


if len(pareto_candidate_df) == FINAL_SELECTION_SIZE:

    selected_positions = list(
        range(FINAL_SELECTION_SIZE)
    )

else:

    maxmin_picker = (
        rdSimDivPickers
        .MaxMinPicker()
    )

    selected_positions = list(
        maxmin_picker.LazyBitVectorPick(
            pareto_fingerprints,
            len(pareto_fingerprints),
            FINAL_SELECTION_SIZE,
            tuple(first_picks),
            RANDOM_STATE
        )
    )


if len(selected_positions) != FINAL_SELECTION_SIZE:
    raise RuntimeError(
        "Ο MaxMinPicker δεν επέστρεψε τον αναμενόμενο "
        f"αριθμό των {FINAL_SELECTION_SIZE} μορίων."
    )

if len(set(selected_positions)) != FINAL_SELECTION_SIZE:
    raise RuntimeError(
        "Ο MaxMinPicker επέστρεψε διπλότυπες θέσεις."
    )


# ============================================================
# 20. Δημιουργία τελικού πίνακα επιλεγμένων μορίων
# ============================================================

selected_candidates_df = (
    pareto_candidate_df
    .iloc[selected_positions]
    .copy()
    .reset_index(drop=True)
)

selected_candidates_df.insert(
    0,
    "Selection_Order",
    np.arange(
        1,
        FINAL_SELECTION_SIZE + 1
    )
)

selected_candidates_df.insert(
    1,
    "Selected_Candidate_ID",
    [
        f"SGC_{index:02d}"
        for index in range(
            1,
            FINAL_SELECTION_SIZE + 1
        )
    ]
)


# ============================================================
# 21. Fingerprints των τελικά επιλεγμένων μορίων
# ============================================================

selected_fingerprints = [
    pareto_fingerprints[position]
    for position in selected_positions
]


# ============================================================
# 22. Υπολογισμός pairwise Tanimoto matrix
# ============================================================

number_of_selected = len(
    selected_candidates_df
)

pairwise_similarity_matrix = np.eye(
    number_of_selected,
    dtype=float
)

for row_index in range(number_of_selected):

    similarities = (
        DataStructs.BulkTanimotoSimilarity(
            selected_fingerprints[row_index],
            selected_fingerprints
        )
    )

    pairwise_similarity_matrix[
        row_index,
        :
    ] = np.asarray(
        similarities,
        dtype=float
    )


selected_candidate_ids = (
    selected_candidates_df[
        "Selected_Candidate_ID"
    ].tolist()
)

pairwise_matrix_df = pd.DataFrame(
    pairwise_similarity_matrix,
    index=selected_candidate_ids,
    columns=selected_candidate_ids
)

pairwise_matrix_df.index.name = (
    "Selected_Candidate_ID"
)

pairwise_matrix_df.to_csv(
    pairwise_matrix_output_file
)


# ============================================================
# 23. Nearest selected neighbor για κάθε τελικό candidate
# ============================================================

nearest_selected_ids = []
nearest_selected_similarities = []

for row_index in range(number_of_selected):

    row_similarities = (
        pairwise_similarity_matrix[
            row_index
        ].copy()
    )

    # Αποκλείουμε τη self-similarity.
    row_similarities[row_index] = -np.inf

    nearest_index = int(
        np.argmax(
            row_similarities
        )
    )

    nearest_selected_ids.append(
        selected_candidate_ids[
            nearest_index
        ]
    )

    nearest_selected_similarities.append(
        float(
            row_similarities[
                nearest_index
            ]
        )
    )


selected_candidates_df[
    "Nearest_Selected_Candidate_ID"
] = nearest_selected_ids

selected_candidates_df[
    "Nearest_Selected_Tanimoto"
] = nearest_selected_similarities


# ============================================================
# 24. Συλλογή όλων των μοναδικών pairwise similarities
# ============================================================

upper_triangle_indices = np.triu_indices(
    number_of_selected,
    k=1
)

unique_pairwise_similarities = (
    pairwise_similarity_matrix[
        upper_triangle_indices
    ]
)


# ============================================================
# 25. Συνοπτικά στατιστικά δομικής διαφοροποίησης
# ============================================================

diversity_summary_df = pd.DataFrame([
    {
        "Metric": "Selected molecules",
        "Value": number_of_selected
    },
    {
        "Metric": "Unique molecular pairs",
        "Value": len(
            unique_pairwise_similarities
        )
    },
    {
        "Metric": "Mean pairwise Tanimoto",
        "Value": float(
            np.mean(
                unique_pairwise_similarities
            )
        )
    },
    {
        "Metric": "Median pairwise Tanimoto",
        "Value": float(
            np.median(
                unique_pairwise_similarities
            )
        )
    },
    {
        "Metric": "Standard deviation pairwise Tanimoto",
        "Value": float(
            np.std(
                unique_pairwise_similarities,
                ddof=0
            )
        )
    },
    {
        "Metric": "Minimum pairwise Tanimoto",
        "Value": float(
            np.min(
                unique_pairwise_similarities
            )
        )
    },
    {
        "Metric": "Maximum pairwise Tanimoto",
        "Value": float(
            np.max(
                unique_pairwise_similarities
            )
        )
    },
    {
        "Metric": "Mean nearest-selected Tanimoto",
        "Value": float(
            selected_candidates_df[
                "Nearest_Selected_Tanimoto"
            ].mean()
        )
    },
    {
        "Metric": "Maximum nearest-selected Tanimoto",
        "Value": float(
            selected_candidates_df[
                "Nearest_Selected_Tanimoto"
            ].max()
        )
    }
])

diversity_summary_df[
    "Value"
] = pd.to_numeric(
    diversity_summary_df["Value"],
    errors="coerce"
).round(6)

diversity_summary_df.to_csv(
    diversity_summary_output_file,
    index=False
)


# ============================================================
# 26. Περιγραφικά στατιστικά των επιλεγμένων candidates
# ============================================================

selected_property_summary_df = (
    selected_candidates_df[
        [
            "QED",
            "SAS",
            "Absolute_logP_Error",
            "Target_logP",
            "LogP_RDKit",
            "Molecular_Weight",
            "HBD",
            "HBA",
            "Rotatable_Bonds",
            "TPSA",
            "Nearest_Selected_Tanimoto"
        ]
    ]
    .agg(
        [
            "mean",
            "median",
            "std",
            "min",
            "max"
        ]
    )
    .transpose()
    .round(6)
)


# ============================================================
# 27. Metadata της διαδικασίας επιλογής
# ============================================================

selection_metadata_df = pd.DataFrame([
    {
        "Parameter": "Input_Candidate_Pool_Size",
        "Value": len(candidate_pool_df)
    },
    {
        "Parameter": "Pareto_Objective_1",
        "Value": "Maximize QED"
    },
    {
        "Parameter": "Pareto_Objective_2",
        "Value": "Minimize SAS"
    },
    {
        "Parameter": "Pareto_Objective_3",
        "Value": "Minimize Absolute logP Error"
    },
    {
        "Parameter": "Weighted_Composite_Score",
        "Value": "Not used"
    },
    {
        "Parameter": "Pareto_Layers_Used",
        "Value": len(pareto_layers)
    },
    {
        "Parameter": "Pareto_Candidate_Set_Size",
        "Value": len(pareto_candidate_df)
    },
    {
        "Parameter": "Final_Selection_Size",
        "Value": FINAL_SELECTION_SIZE
    },
    {
        "Parameter": "Fingerprint_Type",
        "Value": "Morgan bit fingerprint"
    },
    {
        "Parameter": "Morgan_Radius",
        "Value": MORGAN_RADIUS
    },
    {
        "Parameter": "Fingerprint_Bits",
        "Value": FINGERPRINT_BITS
    },
    {
        "Parameter": "Fingerprint_Chirality",
        "Value": False
    },
    {
        "Parameter": "Structural_Selection_Method",
        "Value": "RDKit MaxMinPicker"
    },
    {
        "Parameter": "Random_State",
        "Value": RANDOM_STATE
    },
    {
        "Parameter": "Methodological_Status",
        "Value": (
            "Explicit target-independent thesis adaptation; "
            "not the target-specific docking selection of the paper"
        )
    }
])

selection_metadata_df.to_csv(
    selection_metadata_output_file,
    index=False
)


# ============================================================
# 28. Αποθήκευση τελικών επιλεγμένων candidates
# ============================================================

selected_candidates_df.to_csv(
    selected_candidates_output_file,
    index=False
)


# ============================================================
# 29. Εμφάνιση βασικών αποτελεσμάτων
# ============================================================

columns_to_display = [
    "Selection_Order",
    "Selected_Candidate_ID",
    "Pareto_Layer",
    smiles_column,
    "QED",
    "SAS",
    "Target_logP",
    "LogP_RDKit",
    "Absolute_logP_Error",
    "Molecular_Weight",
    "TPSA",
    "Nearest_Selected_Candidate_ID",
    "Nearest_Selected_Tanimoto"
]


print("\n" + "=" * 85)
print("SELECTED GENERATED CANDIDATES")
print("=" * 85)

display(
    selected_candidates_df[
        columns_to_display
    ]
)


print("\n" + "=" * 85)
print("ΣΤΑΤΙΣΤΙΚΑ ΔΟΜΙΚΗΣ ΔΙΑΦΟΡΟΠΟΙΗΣΗΣ")
print("=" * 85)

display(
    diversity_summary_df
)


print("\n" + "=" * 85)
print("ΠΕΡΙΓΡΑΦΙΚΑ ΣΤΑΤΙΣΤΙΚΑ ΕΠΙΛΕΓΜΕΝΩΝ ΜΟΡΙΩΝ")
print("=" * 85)

display(
    selected_property_summary_df
)


print("\n" + "=" * 85)
print("METADATA ΔΙΑΔΙΚΑΣΙΑΣ ΕΠΙΛΟΓΗΣ")
print("=" * 85)

display(
    selection_metadata_df
)


# ============================================================
# 30. Τελική αναφορά αρχείων
# ============================================================

print("\n" + "-" * 85)
print("Η PARETO + MAXMIN ΕΠΙΛΟΓΗ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 85)

print(
    f"Pareto candidate set:\n"
    f"{pareto_output_file}"
)

print(
    f"\n20 Selected Generated Candidates:\n"
    f"{selected_candidates_output_file}"
)

print(
    f"\nPairwise Tanimoto matrix:\n"
    f"{pairwise_matrix_output_file}"
)

print(
    f"\nDiversity summary:\n"
    f"{diversity_summary_output_file}"
)

print(
    f"\nSelection metadata:\n"
    f"{selection_metadata_output_file}"
)

print(
    "\nΣημείωση: Δεν εφαρμόστηκε docking, "
    "protein-specific ranking ή weighted composite score."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 6:
# Paper-style οπτικοποίηση των Selected Generated Candidates
#
# Δομή βασισμένη στο Supplementary Figure S2 του paper:
# - 3 μόρια ανά σειρά
# - έως 5 σειρές ανά σελίδα
# - 15 μόρια στην πρώτη σελίδα
# - απλή αριθμημένη λεζάντα
# - χωρίς φυσικοχημικές ιδιότητες μέσα στην εικόνα
#
# Ρητή προσαρμογή:
# Παρουσιάζονται τα 20 Selected Generated Candidates
# και όχι όλα τα generated molecules.
# ============================================================

import os
from io import BytesIO

import numpy as np
import pandas as pd

from PIL import (
    Image as PILImage,
    ImageDraw,
    ImageFont
)

from matplotlib import font_manager

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, rdDepictor

from IPython.display import (
    display,
    Image as IPythonImage
)


# ============================================================
# 1. Ρυθμίσεις RDKit
# ============================================================

RDLogger.DisableLog("rdApp.*")

# Προτιμάμε συνεπή και καθαρή παραγωγή 2D συντεταγμένων.
rdDepictor.SetPreferCoordGen(True)


# ============================================================
# 2. Αρχείο εισόδου
# ============================================================

selected_candidates_file = (
    "SELECTED_20_GENERATED_CANDIDATES.csv"
)


# ============================================================
# 3. Αρχεία εξόδου
# ============================================================

paper_style_page_1_output = (
    "SELECTED_GENERATED_MOLECULES_PAPER_STYLE_PAGE_1.png"
)

paper_style_page_2_output = (
    "SELECTED_GENERATED_MOLECULES_PAPER_STYLE_PAGE_2.png"
)

mapping_output = (
    "SELECTED_GENERATED_MOLECULES_PAPER_STYLE_MAPPING.csv"
)


# ============================================================
# 4. Έλεγχος ύπαρξης αρχείου
# ============================================================

if not os.path.exists(selected_candidates_file):
    raise FileNotFoundError(
        f"Δεν βρέθηκε το αρχείο:\n"
        f"{selected_candidates_file}"
    )


# ============================================================
# 5. Φόρτωση των 20 selected candidates
# ============================================================

selected_df = pd.read_csv(
    selected_candidates_file
)

print("=" * 82)
print("ΦΟΡΤΩΣΗ SELECTED GENERATED CANDIDATES")
print("=" * 82)

print(
    f"Αριθμός μορίων: "
    f"{len(selected_df):,}"
)


# ============================================================
# 6. Έλεγχος ότι υπάρχουν ακριβώς 20 μόρια
# ============================================================

if len(selected_df) != 20:
    raise ValueError(
        "Το αρχείο πρέπει να περιέχει ακριβώς 20 μόρια.\n"
        f"Βρέθηκαν: {len(selected_df)}"
    )


# ============================================================
# 7. Εντοπισμός της canonical SMILES στήλης
# ============================================================

smiles_column_candidates = [
    "Canonical_SMILES_Check",
    "Canonical_SMILES",
    "Evaluation_SMILES",
    "SMILES"
]

smiles_column = None

for candidate_column in smiles_column_candidates:

    if candidate_column in selected_df.columns:
        smiles_column = candidate_column
        break


if smiles_column is None:
    raise ValueError(
        "Δεν βρέθηκε κατάλληλη στήλη SMILES.\n"
        f"Διαθέσιμες στήλες:\n"
        f"{selected_df.columns.tolist()}"
    )


print(
    f"Στήλη SMILES: "
    f"{smiles_column}"
)


# ============================================================
# 8. Έλεγχος απαιτούμενων στηλών
# ============================================================

required_columns = [
    "Selection_Order",
    "Selected_Candidate_ID",
    "QED",
    "SAS",
    "Target_logP",
    "LogP_RDKit",
    "Absolute_logP_Error",
    "Molecular_Weight",
    "TPSA",
    smiles_column
]

missing_columns = [
    column
    for column in required_columns
    if column not in selected_df.columns
]

if missing_columns:
    raise ValueError(
        "Λείπουν οι ακόλουθες απαιτούμενες στήλες:\n"
        f"{missing_columns}"
    )


# ============================================================
# 9. Μετατροπή αριθμητικών στηλών
# ============================================================

numeric_columns = [
    "Selection_Order",
    "QED",
    "SAS",
    "Target_logP",
    "LogP_RDKit",
    "Absolute_logP_Error",
    "Molecular_Weight",
    "TPSA"
]

for column in numeric_columns:

    selected_df[column] = pd.to_numeric(
        selected_df[column],
        errors="coerce"
    )


selected_df[numeric_columns] = (
    selected_df[numeric_columns]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)


missing_numeric_values = (
    selected_df[numeric_columns]
    .isna()
    .sum()
)

if missing_numeric_values.sum() > 0:
    raise ValueError(
        "Βρέθηκαν κενές ή μη αριθμητικές τιμές:\n"
        f"{missing_numeric_values[missing_numeric_values > 0]}"
    )


# ============================================================
# 10. Σταθερή σειρά εμφάνισης
#
# Το Selection_Order δεν αποτελεί σειρά ποιότητας.
# Χρησιμοποιείται μόνο για τη διάταξη των εικόνων.
# ============================================================

selected_df = (
    selected_df
    .sort_values(
        by="Selection_Order"
    )
    .reset_index(drop=True)
)


# ============================================================
# 11. Δημιουργία RDKit molecules
# ============================================================

molecules = []
simple_legends = []
mapping_rows = []

for index, row in selected_df.iterrows():

    presentation_number = index + 1

    candidate_id = str(
        row["Selected_Candidate_ID"]
    )

    smiles = str(
        row[smiles_column]
    ).strip()

    molecule = Chem.MolFromSmiles(
        smiles
    )

    if molecule is None:
        raise ValueError(
            f"Μη έγκυρο SMILES για το μόριο "
            f"{candidate_id}:\n"
            f"{smiles}"
        )


    # Δημιουργία 2D συντεταγμένων.
    rdDepictor.Compute2DCoords(
        molecule
    )


    # --------------------------------------------------------
    # Απλή λεζάντα σύμφωνα με τη λογική του paper.
    # --------------------------------------------------------

    simple_legend = (
        f"Selected generated molecule "
        f"{presentation_number}"
    )


    molecules.append(
        molecule
    )

    simple_legends.append(
        simple_legend
    )


    # --------------------------------------------------------
    # Οι ιδιότητες αποθηκεύονται σε ξεχωριστό πίνακα
    # και δεν εμφανίζονται μέσα στο molecular grid.
    # --------------------------------------------------------

    mapping_rows.append({
        "Presentation_Number": presentation_number,
        "Figure_Label": simple_legend,
        "Selected_Candidate_ID": candidate_id,
        "Canonical_SMILES": smiles,
        "QED": round(
            float(row["QED"]),
            6
        ),
        "SAS": round(
            float(row["SAS"]),
            6
        ),
        "Target_logP": round(
            float(row["Target_logP"]),
            6
        ),
        "Actual_logP_RDKit": round(
            float(row["LogP_RDKit"]),
            6
        ),
        "Absolute_logP_Error": round(
            float(row["Absolute_logP_Error"]),
            6
        ),
        "Molecular_Weight": round(
            float(row["Molecular_Weight"]),
            6
        ),
        "TPSA": round(
            float(row["TPSA"]),
            6
        )
    })


print(
    f"\nΈγκυρα RDKit molecules: "
    f"{len(molecules)}"
)


# ============================================================
# 12. Αποθήκευση πίνακα αντιστοίχισης
# ============================================================

mapping_df = pd.DataFrame(
    mapping_rows
)

mapping_df.to_csv(
    mapping_output,
    index=False
)


# ============================================================
# 13. Εξαγωγή PNG bytes από αποτέλεσμα RDKit
# ============================================================

def extract_png_bytes(drawing_result):
    """
    Εξάγει τα PNG bytes από διαφορετικούς δυνατούς
    τύπους αποτελέσματος της RDKit.
    """

    if isinstance(
        drawing_result,
        (bytes, bytearray)
    ):
        return bytes(
            drawing_result
        )


    if hasattr(
        drawing_result,
        "data"
    ):

        image_data = drawing_result.data

        if isinstance(
            image_data,
            (bytes, bytearray)
        ):
            return bytes(
                image_data
            )


    if hasattr(
        drawing_result,
        "_repr_png_"
    ):

        image_data = drawing_result._repr_png_()

        if isinstance(
            image_data,
            (bytes, bytearray)
        ):
            return bytes(
                image_data
            )


    return None


# ============================================================
# 14. Φόρτωση κατάλληλης γραμματοσειράς για τον τίτλο
# ============================================================

try:

    font_path = font_manager.findfont(
        "DejaVu Sans"
    )

    title_font = ImageFont.truetype(
        font_path,
        size=40
    )

except Exception:

    title_font = ImageFont.load_default()


# ============================================================
# 15. Βοηθητική συνάρτηση δημιουργίας paper-style σελίδας
# ============================================================

def create_paper_style_page(
    page_molecules,
    page_legends,
    output_path,
    show_title=False
):
    """
    Δημιουργεί μία σελίδα απεικόνισης σύμφωνα με
    τη γενική διάταξη του Supplementary Figure S2:

    - 3 μόρια ανά σειρά
    - έως 5 σειρές
    - απλή αρίθμηση
    - λευκό φόντο
    """

    # --------------------------------------------------------
    # Σταθερές διαστάσεις κάθε molecular panel.
    # --------------------------------------------------------

    molecules_per_row = 3

    sub_image_width = 450
    sub_image_height = 310

    maximum_rows_per_page = 5

    full_grid_width = (
        molecules_per_row
        * sub_image_width
    )

    full_grid_height = (
        maximum_rows_per_page
        * sub_image_height
    )


    # --------------------------------------------------------
    # Δημιουργία του RDKit grid.
    # --------------------------------------------------------

    drawing_result = Draw.MolsToGridImage(
        mols=page_molecules,
        molsPerRow=molecules_per_row,
        subImgSize=(
            sub_image_width,
            sub_image_height
        ),
        legends=page_legends,
        useSVG=False,
        returnPNG=True
    )


    png_bytes = extract_png_bytes(
        drawing_result
    )

    if png_bytes is None:
        raise TypeError(
            "Δεν ήταν δυνατή η εξαγωγή PNG δεδομένων "
            "από το αποτέλεσμα της RDKit."
        )


    molecular_grid = PILImage.open(
        BytesIO(png_bytes)
    ).convert(
        "RGB"
    )


    # --------------------------------------------------------
    # Δημιουργία σταθερής portrait-like σελίδας.
    #
    # Η δεύτερη σελίδα διατηρεί τις ίδιες διαστάσεις,
    # ακόμη και αν περιλαμβάνει μόνο 5 μόρια.
    # --------------------------------------------------------

    top_margin = 45
    title_area_height = 85 if show_title else 35
    bottom_margin = 45

    page_width = full_grid_width

    page_height = (
        top_margin
        + title_area_height
        + full_grid_height
        + bottom_margin
    )


    page_canvas = PILImage.new(
        mode="RGB",
        size=(
            page_width,
            page_height
        ),
        color="white"
    )


    # --------------------------------------------------------
    # Προσθήκη τίτλου μόνο στην πρώτη σελίδα,
    # όπως στη δομή του Supplementary Figure S2.
    # --------------------------------------------------------

    if show_title:

        title_text = (
            "Graph representation of the "
            "selected generated molecules"
        )

        title_draw = ImageDraw.Draw(
            page_canvas
        )

        title_box = title_draw.textbbox(
            (0, 0),
            title_text,
            font=title_font
        )

        title_width = (
            title_box[2]
            - title_box[0]
        )

        title_x = (
            page_width
            - title_width
        ) // 2

        title_y = top_margin

        title_draw.text(
            (
                title_x,
                title_y
            ),
            title_text,
            fill="black",
            font=title_font
        )


    # --------------------------------------------------------
    # Τοποθέτηση molecular grid στη σελίδα.
    # --------------------------------------------------------

    grid_x = 0

    grid_y = (
        top_margin
        + title_area_height
    )

    page_canvas.paste(
        molecular_grid,
        (
            grid_x,
            grid_y
        )
    )


    # --------------------------------------------------------
    # Αποθήκευση τελικού PNG.
    # --------------------------------------------------------

    page_canvas.save(
        output_path,
        format="PNG",
        dpi=(
            300,
            300
        )
    )


    if not os.path.exists(
        output_path
    ):
        raise RuntimeError(
            f"Δεν δημιουργήθηκε το αρχείο:\n"
            f"{output_path}"
        )


    file_size_kb = (
        os.path.getsize(output_path)
        / 1024
    )

    print(
        f"Αποθηκεύτηκε: {output_path} "
        f"({file_size_kb:.1f} KB)"
    )


# ============================================================
# 16. Σελίδα 1: μόρια 1–15
#
# Ακολουθεί την πλήρη διάταξη 3 × 5 του paper.
# ============================================================

print("\n" + "=" * 82)
print("ΔΗΜΙΟΥΡΓΙΑ PAPER-STYLE PAGE 1")
print("=" * 82)

create_paper_style_page(
    page_molecules=molecules[:15],
    page_legends=simple_legends[:15],
    output_path=paper_style_page_1_output,
    show_title=True
)


# ============================================================
# 17. Σελίδα 2: μόρια 16–20
#
# Διατηρείται η ίδια κλίμακα και η ίδια διάταξη
# με την πρώτη σελίδα.
# ============================================================

print("\n" + "=" * 82)
print("ΔΗΜΙΟΥΡΓΙΑ PAPER-STYLE PAGE 2")
print("=" * 82)

create_paper_style_page(
    page_molecules=molecules[15:20],
    page_legends=simple_legends[15:20],
    output_path=paper_style_page_2_output,
    show_title=False
)


# ============================================================
# 18. Εμφάνιση πρώτης σελίδας
# ============================================================

print("\n" + "=" * 82)
print("PAPER-STYLE VISUALIZATION — PAGE 1")
print("=" * 82)

display(
    IPythonImage(
        filename=paper_style_page_1_output
    )
)


# ============================================================
# 19. Εμφάνιση δεύτερης σελίδας
# ============================================================

print("\n" + "=" * 82)
print("PAPER-STYLE VISUALIZATION — PAGE 2")
print("=" * 82)

display(
    IPythonImage(
        filename=paper_style_page_2_output
    )
)


# ============================================================
# 20. Εμφάνιση ξεχωριστού πίνακα ιδιοτήτων
# ============================================================

print("\n" + "=" * 82)
print("ΑΝΤΙΣΤΟΙΧΙΣΗ ΜΟΡΙΩΝ ΚΑΙ ΦΥΣΙΚΟΧΗΜΙΚΩΝ ΙΔΙΟΤΗΤΩΝ")
print("=" * 82)

display(
    mapping_df[
        [
            "Presentation_Number",
            "Selected_Candidate_ID",
            "QED",
            "SAS",
            "Target_logP",
            "Actual_logP_RDKit",
            "Absolute_logP_Error",
            "Molecular_Weight",
            "TPSA"
        ]
    ]
)


# ============================================================
# 21. Τελική αναφορά
# ============================================================

print("\n" + "-" * 82)
print("Η PAPER-STYLE ΟΠΤΙΚΟΠΟΙΗΣΗ ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 82)

print(
    f"Σελίδα 1 — μόρια 1–15:\n"
    f"{paper_style_page_1_output}"
)

print(
    f"\nΣελίδα 2 — μόρια 16–20:\n"
    f"{paper_style_page_2_output}"
)

print(
    f"\nΠίνακας αντιστοίχισης και ιδιοτήτων:\n"
    f"{mapping_output}"
)

print(
    "\nΜεθοδολογική σημείωση: "
    "Η διάταξη της απεικόνισης ακολουθεί τη λογική "
    "του Supplementary Figure S2 του paper. "
    "Η παρουσίαση μόνο των 20 selected candidates "
    "αποτελεί προσαρμογή της παρούσας εργασίας."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7B:
# Structural audit της VEGFR2 δομής 3BE2
#
# Στόχοι:
# 1. Λήψη του επίσημου PDB αρχείου από την RCSB
# 2. Καταγραφή protein chains και observed residues
# 3. Καταγραφή SEQADV records και πιθανών μεταλλάξεων
# 4. Καταγραφή missing residues από REMARK 465
# 5. Καταγραφή νερών και λοιπών hetero-residues
# 6. Εξαγωγή του co-crystallized ligand RAJ
# 7. Υπολογισμός του γεωμετρικού κέντρου του ligand
#
# Σημαντικό:
# Δεν δημιουργούμε ακόμη docking-ready receptor.
# Δεν αφαιρούμε επιλεκτικά νερά, δεν προσθέτουμε υδρογόνα
# και δεν καθορίζουμε ακόμη τελικό docking box.
# ============================================================

import os
import re
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

from IPython.display import display


# ============================================================
# 1. Βασικές ρυθμίσεις
# ============================================================

PDB_ID = "3BE2"

# Co-crystallized ligand της συγκεκριμένης δομής.
TARGET_LIGAND_RESNAME = "RAJ"

# Αναμενόμενη πρωτεϊνική αλυσίδα.
TARGET_PROTEIN_CHAIN = "A"

pdb_download_url = (
    f"https://files.rcsb.org/download/{PDB_ID}.pdb"
)

pdb_file = Path(
    f"{PDB_ID}.pdb"
)


# ============================================================
# 2. Αρχεία εξόδου
# ============================================================

metadata_output = Path(
    f"{PDB_ID}_STRUCTURE_METADATA.csv"
)

chain_summary_output = Path(
    f"{PDB_ID}_CHAIN_SUMMARY.csv"
)

seqadv_output = Path(
    f"{PDB_ID}_SEQADV_RECORDS.csv"
)

missing_residues_output = Path(
    f"{PDB_ID}_MISSING_RESIDUES.csv"
)

hetero_residues_output = Path(
    f"{PDB_ID}_HETERO_RESIDUES.csv"
)

water_summary_output = Path(
    f"{PDB_ID}_WATER_SUMMARY.csv"
)

ligand_pdb_output = Path(
    f"{PDB_ID}_{TARGET_LIGAND_RESNAME}_CRYSTAL_LIGAND.pdb"
)

ligand_coordinates_output = Path(
    f"{PDB_ID}_{TARGET_LIGAND_RESNAME}_CRYSTAL_COORDINATES.csv"
)

ligand_geometry_output = Path(
    f"{PDB_ID}_{TARGET_LIGAND_RESNAME}_GEOMETRY.csv"
)

protein_only_output = Path(
    f"{PDB_ID}_CHAIN_{TARGET_PROTEIN_CHAIN}_"
    "PROTEIN_ONLY_UNPREPARED.pdb"
)


# ============================================================
# 3. Λήψη του επίσημου PDB αρχείου
# ============================================================

def download_pdb_file(url, output_path):
    """
    Κατεβάζει το επίσημο PDB αρχείο από την RCSB.

    Το αρχείο δεν κατεβαίνει ξανά αν υπάρχει ήδη και
    έχει εύλογο μέγεθος.
    """

    if output_path.exists() and output_path.stat().st_size > 1000:
        print(
            f"Το αρχείο υπάρχει ήδη και θα χρησιμοποιηθεί:\n"
            f"{output_path}"
        )
        return

    print(
        f"Λήψη της δομής {PDB_ID} από την RCSB..."
    )

    try:
        request = Request(
            url,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 thesis-structural-audit"
                )
            }
        )

        with urlopen(
            request,
            timeout=60
        ) as response:

            pdb_content = response.read()

        if len(pdb_content) < 1000:
            raise RuntimeError(
                "Το αρχείο που λήφθηκε είναι απροσδόκητα μικρό."
            )

        output_path.write_bytes(
            pdb_content
        )

    except Exception as error:
        raise RuntimeError(
            "Δεν ήταν δυνατή η αυτόματη λήψη του PDB αρχείου.\n"
            "Κατέβασε χειροκίνητα το αρχείο 3BE2.pdb από την "
            "RCSB και τοποθέτησέ το στον φάκελο του notebook.\n"
            f"Αρχικό σφάλμα: {error}"
        )

    print(
        f"Αποθηκεύτηκε: {output_path} "
        f"({output_path.stat().st_size / 1024:.1f} KB)"
    )


download_pdb_file(
    pdb_download_url,
    pdb_file
)


# ============================================================
# 4. Ανάγνωση και βασικός έλεγχος του PDB
# ============================================================

pdb_lines = pdb_file.read_text(
    encoding="utf-8",
    errors="replace"
).splitlines()


if not any(
    line.startswith("ATOM")
    for line in pdb_lines
):
    raise ValueError(
        "Το αρχείο δεν περιέχει εγγραφές ATOM "
        "και δεν φαίνεται να είναι έγκυρο PDB αρχείο."
    )


print("\n" + "=" * 84)
print("ΒΑΣΙΚΟΣ ΕΛΕΓΧΟΣ PDB")
print("=" * 84)

print(
    f"Συνολικές γραμμές: "
    f"{len(pdb_lines):,}"
)


# ============================================================
# 5. Εξαγωγή βασικών metadata από το PDB header
# ============================================================

title_parts = [
    line[10:].strip()
    for line in pdb_lines
    if line.startswith("TITLE")
]

experimental_method_parts = [
    line[10:].strip()
    for line in pdb_lines
    if line.startswith("EXPDTA")
]

title_text = " ".join(
    title_parts
).strip()

experimental_method = " ".join(
    experimental_method_parts
).strip()


resolution_angstrom = np.nan

for line in pdb_lines:

    if line.startswith("REMARK   2 RESOLUTION."):

        resolution_match = re.search(
            r"RESOLUTION\.\s+([0-9.]+)\s+ANGSTROMS",
            line
        )

        if resolution_match:
            resolution_angstrom = float(
                resolution_match.group(1)
            )
            break


# ============================================================
# 6. Συνάρτηση ασφαλούς ανάγνωσης αριθμών
# ============================================================

def safe_int(value):
    """
    Μετατρέπει μία συμβολοσειρά σε integer ή επιστρέφει None.
    """

    try:
        return int(
            str(value).strip()
        )
    except Exception:
        return None


def safe_float(value):
    """
    Μετατρέπει μία συμβολοσειρά σε float ή επιστρέφει NaN.
    """

    try:
        return float(
            str(value).strip()
        )
    except Exception:
        return np.nan


# ============================================================
# 7. Ανάλυση ATOM και HETATM records
# ============================================================

atom_rows = []

for line in pdb_lines:

    record_type = line[0:6].strip()

    if record_type not in {
        "ATOM",
        "HETATM"
    }:
        continue

    # Παραβλέπουμε γραμμές που είναι υπερβολικά μικρές
    # για να περιέχουν κανονικές PDB συντεταγμένες.
    if len(line) < 54:
        continue

    atom_rows.append({
        "Record_Type": record_type,
        "Atom_Serial": safe_int(
            line[6:11]
        ),
        "Atom_Name": line[12:16].strip(),
        "AltLoc": line[16:17].strip(),
        "Residue_Name": line[17:20].strip(),
        "Chain_ID": (
            line[21:22].strip()
            if line[21:22].strip()
            else "_"
        ),
        "Residue_Number": safe_int(
            line[22:26]
        ),
        "Insertion_Code": line[26:27].strip(),
        "X": safe_float(
            line[30:38]
        ),
        "Y": safe_float(
            line[38:46]
        ),
        "Z": safe_float(
            line[46:54]
        ),
        "Occupancy": safe_float(
            line[54:60]
        ),
        "B_Factor": safe_float(
            line[60:66]
        ),
        "Element": (
            line[76:78].strip()
            if len(line) >= 78
            else ""
        ),
        "Original_Line": line
    })


atom_df = pd.DataFrame(
    atom_rows
)


if atom_df.empty:
    raise ValueError(
        "Δεν ήταν δυνατή η ανάγνωση ATOM/HETATM records."
    )


# Για μετρήσεις και κέντρο ligand χρησιμοποιούμε:
# - atoms χωρίς alternate location,
# - ή την κύρια alternate location A.
primary_atom_df = atom_df[
    atom_df["AltLoc"].isin(
        ["", "A"]
    )
].copy()


print(
    f"ATOM/HETATM records: "
    f"{len(atom_df):,}"
)

print(
    f"Primary-conformation records: "
    f"{len(primary_atom_df):,}"
)


# ============================================================
# 8. Σύνοψη των protein chains
# ============================================================

protein_atom_df = primary_atom_df[
    primary_atom_df["Record_Type"] == "ATOM"
].copy()


if protein_atom_df.empty:
    raise ValueError(
        "Δεν βρέθηκαν protein ATOM records."
    )


protein_residue_df = (
    protein_atom_df[
        [
            "Chain_ID",
            "Residue_Name",
            "Residue_Number",
            "Insertion_Code"
        ]
    ]
    .drop_duplicates()
    .copy()
)


chain_summary_rows = []

for chain_id, chain_atoms_df in protein_atom_df.groupby(
    "Chain_ID",
    sort=True
):

    chain_residues_df = protein_residue_df[
        protein_residue_df["Chain_ID"] == chain_id
    ]

    residue_numbers = (
        chain_residues_df["Residue_Number"]
        .dropna()
        .astype(int)
    )

    chain_summary_rows.append({
        "Chain_ID": chain_id,
        "Observed_Atom_Count": int(
            len(chain_atoms_df)
        ),
        "Observed_Residue_Count": int(
            len(chain_residues_df)
        ),
        "Minimum_Observed_Residue_Number": (
            int(residue_numbers.min())
            if len(residue_numbers) > 0
            else np.nan
        ),
        "Maximum_Observed_Residue_Number": (
            int(residue_numbers.max())
            if len(residue_numbers) > 0
            else np.nan
        ),
        "Alternate_Locations_Present": bool(
            (
                atom_df[
                    atom_df["Chain_ID"] == chain_id
                ]["AltLoc"] != ""
            ).any()
        )
    })


chain_summary_df = pd.DataFrame(
    chain_summary_rows
)

chain_summary_df.to_csv(
    chain_summary_output,
    index=False
)


# ============================================================
# 9. Ανάλυση SEQADV records
#
# Τα SEQADV records μπορούν να περιλαμβάνουν:
# - engineered mutations,
# - sequence conflicts,
# - expression tags,
# - άλλες αποκλίσεις από τη reference sequence.
#
# Δεν χαρακτηρίζουμε κάθε SEQADV αυτομάτως ως mutation.
# ============================================================

seqadv_rows = []

for line in pdb_lines:

    if not line.startswith("SEQADV"):
        continue

    conflict_description = (
        line[49:70].strip()
        if len(line) >= 50
        else ""
    )

    seqadv_rows.append({
        "PDB_ID": line[7:11].strip(),
        "PDB_Residue_Name": line[12:15].strip(),
        "Chain_ID": (
            line[16:17].strip()
            if line[16:17].strip()
            else "_"
        ),
        "PDB_Residue_Number": safe_int(
            line[18:22]
        ),
        "Insertion_Code": line[22:23].strip(),
        "Reference_Database": line[24:28].strip(),
        "Reference_Accession": line[29:38].strip(),
        "Reference_Residue_Name": line[39:42].strip(),
        "Reference_Residue_Number": safe_int(
            line[43:48]
        ),
        "Conflict_Description": conflict_description,
        "Is_Explicit_Engineered_Mutation": (
            "ENGINEERED MUTATION"
            in conflict_description.upper()
        ),
        "Original_Record": line
    })


seqadv_df = pd.DataFrame(
    seqadv_rows
)


if seqadv_df.empty:

    seqadv_df = pd.DataFrame(
        columns=[
            "PDB_ID",
            "PDB_Residue_Name",
            "Chain_ID",
            "PDB_Residue_Number",
            "Insertion_Code",
            "Reference_Database",
            "Reference_Accession",
            "Reference_Residue_Name",
            "Reference_Residue_Number",
            "Conflict_Description",
            "Is_Explicit_Engineered_Mutation",
            "Original_Record"
        ]
    )


seqadv_df.to_csv(
    seqadv_output,
    index=False
)


# ============================================================
# 10. Εξαγωγή missing residues από REMARK 465
# ============================================================

missing_residue_rows = []

for line in pdb_lines:

    if not line.startswith("REMARK 465"):
        continue

    payload = line[10:].strip()
    parts = payload.split()

    # Αναμενόμενα δεδομένα στο τέλος:
    # RESNAME CHAIN RESSEQ
    #
    # Μερικές εγγραφές μπορεί να έχουν και model number.
    if len(parts) < 3:
        continue

    residue_position_match = re.fullmatch(
        r"(-?\d+)([A-Za-z]?)",
        parts[-1]
    )

    if residue_position_match is None:
        continue

    residue_name = parts[-3]
    chain_id = parts[-2]

    # Αποκλείουμε τη γραμμή κεφαλίδας της REMARK 465.
    if len(residue_name) != 3:
        continue

    model_number = None

    if len(parts) >= 4 and parts[-4].isdigit():
        model_number = int(
            parts[-4]
        )

    missing_residue_rows.append({
        "Model_Number": model_number,
        "Residue_Name": residue_name,
        "Chain_ID": chain_id,
        "Residue_Number": int(
            residue_position_match.group(1)
        ),
        "Insertion_Code": (
            residue_position_match.group(2)
        ),
        "Original_Record": line
    })


missing_residues_df = pd.DataFrame(
    missing_residue_rows
)


if missing_residues_df.empty:

    missing_residues_df = pd.DataFrame(
        columns=[
            "Model_Number",
            "Residue_Name",
            "Chain_ID",
            "Residue_Number",
            "Insertion_Code",
            "Original_Record"
        ]
    )


missing_residues_df.to_csv(
    missing_residues_output,
    index=False
)


# ============================================================
# 11. Καταγραφή hetero-residues και νερών
# ============================================================

water_residue_names = {
    "HOH",
    "WAT",
    "DOD"
}


hetero_atom_df = primary_atom_df[
    primary_atom_df["Record_Type"] == "HETATM"
].copy()


hetero_atom_df["Is_Water"] = (
    hetero_atom_df["Residue_Name"]
    .isin(water_residue_names)
)


non_water_hetero_df = hetero_atom_df[
    ~hetero_atom_df["Is_Water"]
].copy()

water_atom_df = hetero_atom_df[
    hetero_atom_df["Is_Water"]
].copy()


hetero_summary_rows = []

grouping_columns = [
    "Residue_Name",
    "Chain_ID",
    "Residue_Number",
    "Insertion_Code"
]


for group_key, group_df in non_water_hetero_df.groupby(
    grouping_columns,
    dropna=False,
    sort=True
):

    (
        residue_name,
        chain_id,
        residue_number,
        insertion_code
    ) = group_key

    elements_upper = (
        group_df["Element"]
        .fillna("")
        .astype(str)
        .str.upper()
    )

    heavy_atom_mask = (
        elements_upper != "H"
    )

    hetero_summary_rows.append({
        "Residue_Name": residue_name,
        "Chain_ID": chain_id,
        "Residue_Number": residue_number,
        "Insertion_Code": insertion_code,
        "Atom_Count": int(
            len(group_df)
        ),
        "Heavy_Atom_Count": int(
            heavy_atom_mask.sum()
        ),
        "Centroid_X": float(
            group_df["X"].mean()
        ),
        "Centroid_Y": float(
            group_df["Y"].mean()
        ),
        "Centroid_Z": float(
            group_df["Z"].mean()
        )
    })


hetero_summary_df = pd.DataFrame(
    hetero_summary_rows
)


if hetero_summary_df.empty:

    hetero_summary_df = pd.DataFrame(
        columns=[
            "Residue_Name",
            "Chain_ID",
            "Residue_Number",
            "Insertion_Code",
            "Atom_Count",
            "Heavy_Atom_Count",
            "Centroid_X",
            "Centroid_Y",
            "Centroid_Z"
        ]
    )


hetero_summary_df.to_csv(
    hetero_residues_output,
    index=False
)


water_residue_df = (
    water_atom_df[
        [
            "Chain_ID",
            "Residue_Number",
            "Insertion_Code",
            "Residue_Name"
        ]
    ]
    .drop_duplicates()
    .copy()
)


water_summary_df = pd.DataFrame([
    {
        "PDB_ID": PDB_ID,
        "Water_Atom_Count": int(
            len(water_atom_df)
        ),
        "Unique_Water_Residue_Count": int(
            len(water_residue_df)
        ),
        "Water_Residue_Names": ", ".join(
            sorted(
                water_atom_df[
                    "Residue_Name"
                ].dropna().unique()
            )
        )
    }
])


water_summary_df.to_csv(
    water_summary_output,
    index=False
)


# ============================================================
# 12. Εντοπισμός του crystallographic ligand RAJ
# ============================================================

ligand_atom_df = hetero_atom_df[
    hetero_atom_df["Residue_Name"]
    == TARGET_LIGAND_RESNAME
].copy()


if ligand_atom_df.empty:
    raise ValueError(
        f"Δεν βρέθηκε ο ligand "
        f"{TARGET_LIGAND_RESNAME} στο PDB αρχείο."
    )


ligand_instances_df = (
    ligand_atom_df[
        [
            "Residue_Name",
            "Chain_ID",
            "Residue_Number",
            "Insertion_Code"
        ]
    ]
    .drop_duplicates()
)


print(
    f"\nLigand instances με resname "
    f"{TARGET_LIGAND_RESNAME}: "
    f"{len(ligand_instances_df)}"
)

display(
    ligand_instances_df
)


# ============================================================
# 13. Επιλογή του ligand instance στην αλυσίδα A
#
# Αν υπάρχουν περισσότερα από ένα instances, επιλέγεται
# αυτό στην προκαθορισμένη protein chain.
# ============================================================

target_chain_ligand_df = ligand_atom_df[
    ligand_atom_df["Chain_ID"]
    == TARGET_PROTEIN_CHAIN
].copy()


if target_chain_ligand_df.empty:

    raise ValueError(
        f"Βρέθηκε ο ligand {TARGET_LIGAND_RESNAME}, "
        f"αλλά όχι στην αναμενόμενη αλυσίδα "
        f"{TARGET_PROTEIN_CHAIN}."
    )


target_ligand_keys = (
    target_chain_ligand_df[
        [
            "Residue_Number",
            "Insertion_Code"
        ]
    ]
    .drop_duplicates()
)


if len(target_ligand_keys) != 1:

    raise ValueError(
        "Βρέθηκαν πολλαπλά RAJ residues στην αλυσίδα A. "
        "Χρειάζεται χειροκίνητη επιλογή πριν συνεχίσουμε."
    )


target_ligand_residue_number = int(
    target_ligand_keys.iloc[0][
        "Residue_Number"
    ]
)

target_ligand_insertion_code = str(
    target_ligand_keys.iloc[0][
        "Insertion_Code"
    ]
)


target_ligand_df = target_chain_ligand_df[
    (
        target_chain_ligand_df["Residue_Number"]
        == target_ligand_residue_number
    )
    &
    (
        target_chain_ligand_df["Insertion_Code"]
        == target_ligand_insertion_code
    )
].copy()


# ============================================================
# 14. Αποθήκευση των crystallographic ligand coordinates
# ============================================================

ligand_coordinate_columns = [
    "Atom_Serial",
    "Atom_Name",
    "Element",
    "AltLoc",
    "Residue_Name",
    "Chain_ID",
    "Residue_Number",
    "Insertion_Code",
    "X",
    "Y",
    "Z",
    "Occupancy",
    "B_Factor"
]


target_ligand_df[
    ligand_coordinate_columns
].to_csv(
    ligand_coordinates_output,
    index=False
)


# ============================================================
# 15. Υπολογισμός γεωμετρικού κέντρου του ligand
#
# Χρησιμοποιούνται τα heavy atoms.
# Το κέντρο αυτό αποτελεί μόνο αρχική ένδειξη του pocket
# και όχι τελικό docking-box definition.
# ============================================================

ligand_elements_upper = (
    target_ligand_df["Element"]
    .fillna("")
    .astype(str)
    .str.upper()
)


ligand_heavy_atom_df = target_ligand_df[
    ligand_elements_upper != "H"
].copy()


if ligand_heavy_atom_df.empty:
    raise ValueError(
        "Δεν βρέθηκαν heavy atoms στον crystallographic ligand."
    )


ligand_center = (
    ligand_heavy_atom_df[
        ["X", "Y", "Z"]
    ]
    .mean()
)


ligand_minimum = (
    ligand_heavy_atom_df[
        ["X", "Y", "Z"]
    ]
    .min()
)


ligand_maximum = (
    ligand_heavy_atom_df[
        ["X", "Y", "Z"]
    ]
    .max()
)


ligand_span = (
    ligand_maximum
    - ligand_minimum
)


ligand_geometry_df = pd.DataFrame([
    {
        "PDB_ID": PDB_ID,
        "Ligand_Resname": TARGET_LIGAND_RESNAME,
        "Chain_ID": TARGET_PROTEIN_CHAIN,
        "Residue_Number": target_ligand_residue_number,
        "Insertion_Code": target_ligand_insertion_code,
        "Heavy_Atom_Count": int(
            len(ligand_heavy_atom_df)
        ),
        "Center_X": float(
            ligand_center["X"]
        ),
        "Center_Y": float(
            ligand_center["Y"]
        ),
        "Center_Z": float(
            ligand_center["Z"]
        ),
        "Minimum_X": float(
            ligand_minimum["X"]
        ),
        "Maximum_X": float(
            ligand_maximum["X"]
        ),
        "Minimum_Y": float(
            ligand_minimum["Y"]
        ),
        "Maximum_Y": float(
            ligand_maximum["Y"]
        ),
        "Minimum_Z": float(
            ligand_minimum["Z"]
        ),
        "Maximum_Z": float(
            ligand_maximum["Z"]
        ),
        "Ligand_Span_X": float(
            ligand_span["X"]
        ),
        "Ligand_Span_Y": float(
            ligand_span["Y"]
        ),
        "Ligand_Span_Z": float(
            ligand_span["Z"]
        ),
        "Interpretation": (
            "Preliminary crystallographic ligand geometry; "
            "not a finalized docking box"
        )
    }
])


ligand_geometry_df.to_csv(
    ligand_geometry_output,
    index=False
)


# ============================================================
# 16. Εξαγωγή του crystallographic ligand σε PDB
#
# Διατηρούνται:
# - οι HETATM γραμμές του συγκεκριμένου RAJ,
# - σχετικά CONECT records, όπου είναι διαθέσιμα.
# ============================================================

target_ligand_serials = set(
    target_ligand_df[
        "Atom_Serial"
    ]
    .dropna()
    .astype(int)
)


ligand_pdb_lines = [
    f"REMARK Extracted crystallographic ligand "
    f"{TARGET_LIGAND_RESNAME} from {PDB_ID}",
    f"REMARK Chain {TARGET_PROTEIN_CHAIN}, "
    f"residue {target_ligand_residue_number}"
]


# Χρησιμοποιούμε τις original HETATM γραμμές ώστε να
# διατηρηθούν οι αρχικές crystallographic coordinates.
for _, atom_row in target_ligand_df.iterrows():

    ligand_pdb_lines.append(
        atom_row["Original_Line"]
    )


# Προσθήκη των σχετικών CONECT records.
for line in pdb_lines:

    if not line.startswith("CONECT"):
        continue

    connected_serials = []

    for start_position in range(
        6,
        len(line),
        5
    ):

        serial_value = safe_int(
            line[
                start_position:
                start_position + 5
            ]
        )

        if serial_value is not None:
            connected_serials.append(
                serial_value
            )

    if (
        connected_serials
        and connected_serials[0]
        in target_ligand_serials
    ):
        ligand_pdb_lines.append(
            line
        )


ligand_pdb_lines.append(
    "END"
)


ligand_pdb_output.write_text(
    "\n".join(ligand_pdb_lines) + "\n",
    encoding="utf-8"
)


# ============================================================
# 17. Εξαγωγή της protein chain A χωρίς heteroatoms
#
# ΠΡΟΣΟΧΗ:
# Το αρχείο είναι UNPREPARED και δεν χρησιμοποιείται ακόμη
# απευθείας για docking.
# ============================================================

protein_only_lines = [
    f"REMARK Protein-only extraction from {PDB_ID}",
    "REMARK UNPREPARED: no protonation, repair, or optimization"
]


for line in pdb_lines:

    if not line.startswith("ATOM"):
        continue

    line_chain_id = (
        line[21:22].strip()
        if len(line) >= 22
        else ""
    )

    if line_chain_id == TARGET_PROTEIN_CHAIN:
        protein_only_lines.append(
            line
        )


protein_only_lines.extend([
    "TER",
    "END"
])


protein_only_output.write_text(
    "\n".join(protein_only_lines) + "\n",
    encoding="utf-8"
)


# ============================================================
# 18. Δημιουργία συνοπτικών metadata
# ============================================================

explicit_mutation_count = int(
    seqadv_df[
        "Is_Explicit_Engineered_Mutation"
    ].sum()
) if len(seqadv_df) > 0 else 0


metadata_df = pd.DataFrame([
    {
        "Field": "PDB_ID",
        "Value": PDB_ID
    },
    {
        "Field": "Title",
        "Value": title_text
    },
    {
        "Field": "Experimental_Method",
        "Value": experimental_method
    },
    {
        "Field": "Resolution_Angstrom",
        "Value": resolution_angstrom
    },
    {
        "Field": "Protein_Chains_Observed",
        "Value": ", ".join(
            sorted(
                protein_atom_df[
                    "Chain_ID"
                ].unique()
            )
        )
    },
    {
        "Field": "Total_Observed_Protein_Residues",
        "Value": int(
            len(protein_residue_df)
        )
    },
    {
        "Field": "SEQADV_Record_Count",
        "Value": int(
            len(seqadv_df)
        )
    },
    {
        "Field": "Explicit_Engineered_Mutation_Count",
        "Value": explicit_mutation_count
    },
    {
        "Field": "Missing_Residue_Record_Count",
        "Value": int(
            len(missing_residues_df)
        )
    },
    {
        "Field": "Unique_Water_Count",
        "Value": int(
            len(water_residue_df)
        )
    },
    {
        "Field": "Non_Water_Hetero_Residue_Count",
        "Value": int(
            len(hetero_summary_df)
        )
    },
    {
        "Field": "Reference_Ligand",
        "Value": TARGET_LIGAND_RESNAME
    },
    {
        "Field": "Reference_Ligand_Chain",
        "Value": TARGET_PROTEIN_CHAIN
    },
    {
        "Field": "Reference_Ligand_Residue_Number",
        "Value": target_ligand_residue_number
    },
    {
        "Field": "Reference_Ligand_Heavy_Atom_Count",
        "Value": int(
            len(ligand_heavy_atom_df)
        )
    },
    {
        "Field": "Preliminary_Ligand_Center_X",
        "Value": float(
            ligand_center["X"]
        )
    },
    {
        "Field": "Preliminary_Ligand_Center_Y",
        "Value": float(
            ligand_center["Y"]
        )
    },
    {
        "Field": "Preliminary_Ligand_Center_Z",
        "Value": float(
            ligand_center["Z"]
        )
    }
])


metadata_df.to_csv(
    metadata_output,
    index=False
)


# ============================================================
# 19. Εμφάνιση αποτελεσμάτων
# ============================================================

print("\n" + "=" * 84)
print("STRUCTURE METADATA")
print("=" * 84)

display(
    metadata_df
)


print("\n" + "=" * 84)
print("PROTEIN CHAIN SUMMARY")
print("=" * 84)

display(
    chain_summary_df
)


print("\n" + "=" * 84)
print("SEQADV RECORDS")
print("=" * 84)

if seqadv_df.empty:
    print(
        "Δεν βρέθηκαν SEQADV records."
    )
else:
    display(
        seqadv_df[
            [
                "PDB_Residue_Name",
                "Chain_ID",
                "PDB_Residue_Number",
                "Reference_Residue_Name",
                "Reference_Residue_Number",
                "Conflict_Description",
                "Is_Explicit_Engineered_Mutation"
            ]
        ]
    )


print("\n" + "=" * 84)
print("MISSING RESIDUES — REMARK 465")
print("=" * 84)

if missing_residues_df.empty:
    print(
        "Δεν βρέθηκαν missing-residue records στο REMARK 465."
    )
else:
    display(
        missing_residues_df[
            [
                "Residue_Name",
                "Chain_ID",
                "Residue_Number",
                "Insertion_Code"
            ]
        ]
    )


print("\n" + "=" * 84)
print("NON-WATER HETERO-RESIDUES")
print("=" * 84)

display(
    hetero_summary_df
)


print("\n" + "=" * 84)
print("CRYSTALLOGRAPHIC LIGAND GEOMETRY")
print("=" * 84)

display(
    ligand_geometry_df
)


# ============================================================
# 20. Τελική αναφορά αρχείων
# ============================================================

print("\n" + "-" * 84)
print("ΤΟ STRUCTURAL AUDIT ΤΗΣ 3BE2 ΟΛΟΚΛΗΡΩΘΗΚΕ")
print("-" * 84)

created_files = [
    pdb_file,
    metadata_output,
    chain_summary_output,
    seqadv_output,
    missing_residues_output,
    hetero_residues_output,
    water_summary_output,
    ligand_pdb_output,
    ligand_coordinates_output,
    ligand_geometry_output,
    protein_only_output
]


for created_file in created_files:

    if created_file.exists():

        print(
            f"{created_file} "
            f"({created_file.stat().st_size / 1024:.1f} KB)"
        )


print(
    "\nΣημαντικό: Το protein-only αρχείο είναι ακατέργαστο "
    "και δεν πρέπει ακόμη να χρησιμοποιηθεί για docking."
)

print(
    "Το κέντρο του RAJ αποτελεί αρχική γεωμετρική αναφορά "
    "και όχι οριστικό docking box."
)

In [ ]:
# ============================================================
# ΒΗΜΑ 7C.0:
# Έλεγχος υπολογιστικού περιβάλλοντος για molecular docking
#
# Σκοπός:
# Να επιβεβαιωθεί ποια εργαλεία είναι πραγματικά
# εγκατεστημένα πριν επιλεγεί ο ακριβής τρόπος:
# - receptor preparation,
# - ligand preparation,
# - redocking,
# - RMSD calculation.
#
# Δεν μεταβάλλεται κανένα αρχείο πρωτεΐνης ή ligand.
# ============================================================

import os
import sys
import shutil
import subprocess
import importlib.util

import pandas as pd

from IPython.display import display


# ============================================================
# 1. Αρχεία που πρέπει ήδη να υπάρχουν
# ============================================================

required_structure_files = [
    "3BE2.pdb",
    "3BE2_RAJ_CRYSTAL_LIGAND.pdb",
    "3BE2_CHAIN_A_PROTEIN_ONLY_UNPREPARED.pdb"
]


# ============================================================
# 2. Έλεγχος αρχείων
# ============================================================

file_audit_rows = []

for file_path in required_structure_files:

    exists = os.path.exists(
        file_path
    )

    size_bytes = (
        os.path.getsize(file_path)
        if exists
        else 0
    )

    file_audit_rows.append({
        "Category": "Required_File",
        "Name": file_path,
        "Available": exists,
        "Location_or_Version": (
            os.path.abspath(file_path)
            if exists
            else "Not found"
        ),
        "Size_Bytes": size_bytes
    })


missing_files = [
    row["Name"]
    for row in file_audit_rows
    if not row["Available"]
]

if missing_files:
    raise FileNotFoundError(
        "Λείπουν τα ακόλουθα απαραίτητα αρχεία:\n"
        f"{missing_files}"
    )


# ============================================================
# 3. Συνάρτηση ασφαλούς λήψης έκδοσης command-line εργαλείου
# ============================================================

def get_command_version(
    command_path,
    version_arguments
):
    """
    Εκτελεί μία σύντομη εντολή έκδοσης.

    Επιστρέφει το πρώτο διαθέσιμο τμήμα της εξόδου
    χωρίς να διακόπτει το notebook αν η εντολή αποτύχει.
    """

    try:

        completed_process = subprocess.run(
            [command_path] + version_arguments,
            capture_output=True,
            text=True,
            timeout=20
        )

        output_text = (
            completed_process.stdout.strip()
            or completed_process.stderr.strip()
        )

        if not output_text:
            return (
                f"Command found; return code "
                f"{completed_process.returncode}"
            )

        output_lines = [
            line.strip()
            for line in output_text.splitlines()
            if line.strip()
        ]

        return (
            output_lines[0]
            if output_lines
            else "Version not reported"
        )

    except Exception as error:

        return (
            f"Version check failed: "
            f"{type(error).__name__}: {error}"
        )


# ============================================================
# 4. Έλεγχος command-line εργαλείων
# ============================================================

command_definitions = [
    {
        "Name": "smina",
        "Candidates": [
            "smina"
        ],
        "Version_Arguments": [
            "--version"
        ]
    },
    {
        "Name": "OpenBabel",
        "Candidates": [
            "obabel",
            "babel"
        ],
        "Version_Arguments": [
            "-V"
        ]
    },
    {
        "Name": "PyMOL",
        "Candidates": [
            "pymol"
        ],
        "Version_Arguments": [
            "-cq"
        ]
    },
    {
        "Name": "Meeko receptor preparation",
        "Candidates": [
            "mk_prepare_receptor.py",
            "mk_prepare_receptor"
        ],
        "Version_Arguments": [
            "--help"
        ]
    },
    {
        "Name": "Meeko ligand preparation",
        "Candidates": [
            "mk_prepare_ligand.py",
            "mk_prepare_ligand"
        ],
        "Version_Arguments": [
            "--help"
        ]
    },
    {
        "Name": "AutoDockTools prepare_receptor4",
        "Candidates": [
            "prepare_receptor4.py"
        ],
        "Version_Arguments": [
            "-h"
        ]
    },
    {
        "Name": "AutoDockTools prepare_ligand4",
        "Candidates": [
            "prepare_ligand4.py"
        ],
        "Version_Arguments": [
            "-h"
        ]
    }
]


command_audit_rows = []


for command_info in command_definitions:

    located_path = None

    for candidate_command in command_info["Candidates"]:

        candidate_path = shutil.which(
            candidate_command
        )

        if candidate_path is not None:
            located_path = candidate_path
            break


    if located_path is None:

        command_audit_rows.append({
            "Category": "Command_Line_Tool",
            "Name": command_info["Name"],
            "Available": False,
            "Location_or_Version": "Not found in PATH",
            "Size_Bytes": None
        })

        continue


    command_version = get_command_version(
        command_path=located_path,
        version_arguments=command_info[
            "Version_Arguments"
        ]
    )


    command_audit_rows.append({
        "Category": "Command_Line_Tool",
        "Name": command_info["Name"],
        "Available": True,
        "Location_or_Version": (
            f"{located_path} | {command_version}"
        ),
        "Size_Bytes": None
    })


# ============================================================
# 5. Έλεγχος Python packages
# ============================================================

python_package_definitions = [
    {
        "Display_Name": "RDKit",
        "Import_Name": "rdkit"
    },
    {
        "Display_Name": "Meeko",
        "Import_Name": "meeko"
    },
    {
        "Display_Name": "OpenBabel Python bindings",
        "Import_Name": "openbabel"
    },
    {
        "Display_Name": "PyMOL Python module",
        "Import_Name": "pymol"
    },
    {
        "Display_Name": "BioPython",
        "Import_Name": "Bio"
    },
    {
        "Display_Name": "MDAnalysis",
        "Import_Name": "MDAnalysis"
    },
    {
        "Display_Name": "spyrmsd",
        "Import_Name": "spyrmsd"
    }
]


python_audit_rows = []


for package_info in python_package_definitions:

    package_spec = importlib.util.find_spec(
        package_info["Import_Name"]
    )

    package_available = (
        package_spec is not None
    )

    package_version = "Not installed"


    if package_available:

        try:

            imported_module = __import__(
                package_info["Import_Name"]
            )

            package_version = getattr(
                imported_module,
                "__version__",
                "Installed; version not exposed"
            )

        except Exception as error:

            package_version = (
                "Found but import failed: "
                f"{type(error).__name__}: {error}"
            )


    python_audit_rows.append({
        "Category": "Python_Package",
        "Name": package_info["Display_Name"],
        "Available": package_available,
        "Location_or_Version": str(
            package_version
        ),
        "Size_Bytes": None
    })


# ============================================================
# 6. Περιγραφή Python environment
# ============================================================

environment_rows = [
    {
        "Category": "Python_Environment",
        "Name": "Python executable",
        "Available": True,
        "Location_or_Version": sys.executable,
        "Size_Bytes": None
    },
    {
        "Category": "Python_Environment",
        "Name": "Python version",
        "Available": True,
        "Location_or_Version": sys.version.replace(
            "\n",
            " "
        ),
        "Size_Bytes": None
    },
    {
        "Category": "Python_Environment",
        "Name": "Working directory",
        "Available": True,
        "Location_or_Version": os.getcwd(),
        "Size_Bytes": None
    }
]


# ============================================================
# 7. Ενοποίηση και αποθήκευση του audit
# ============================================================

environment_audit_df = pd.DataFrame(
    file_audit_rows
    + command_audit_rows
    + python_audit_rows
    + environment_rows
)


environment_audit_output = (
    "DOCKING_ENVIRONMENT_AUDIT.csv"
)

environment_audit_df.to_csv(
    environment_audit_output,
    index=False
)


# ============================================================
# 8. Εμφάνιση αποτελεσμάτων
# ============================================================

pd.set_option(
    "display.max_colwidth",
    None
)

print("=" * 90)
print("DOCKING ENVIRONMENT AUDIT")
print("=" * 90)

display(
    environment_audit_df
)


# ============================================================
# 9. Συνοπτική αναφορά
# ============================================================

print("\n" + "-" * 90)
print("ΣΥΝΟΨΗ")
print("-" * 90)

available_commands = [
    row["Name"]
    for row in command_audit_rows
    if row["Available"]
]

missing_commands = [
    row["Name"]
    for row in command_audit_rows
    if not row["Available"]
]

available_packages = [
    row["Name"]
    for row in python_audit_rows
    if row["Available"]
]


print(
    "Διαθέσιμα command-line εργαλεία:\n"
    f"{available_commands}"
)

print(
    "\nCommand-line εργαλεία που δεν βρέθηκαν:\n"
    f"{missing_commands}"
)

print(
    "\nΔιαθέσιμα Python packages:\n"
    f"{available_packages}"
)

print(
    f"\nΤο πλήρες audit αποθηκεύτηκε ως:\n"
    f"{environment_audit_output}"
)

print(
    "\nΔεν πραγματοποιήθηκε ακόμη receptor ή ligand preparation."
)

In [ ]:
# Find existing dataframe variables
print([x for x in globals().keys() if "df" in x.lower()])